# Customer Retention Prediction — Auto2000 Kenjeran

Portfolio version of my undergraduate thesis project on customer retention prediction for automotive aftersales services at **Auto2000 Kenjeran**.

The study compares three retention-labeling scenarios and three tree-based machine learning models: **Random Forest, XGBoost, and CatBoost**. The public notebook uses the **210-day scenario** as the main modeling pipeline because it produced the strongest overall test performance among the three scenarios.

> **Portfolio note:** Proprietary datasets, internal file paths, checkpoint artifacts, and other company-specific source files are intentionally excluded from this public version. The analytical workflow is preserved, while sensitive configuration values are replaced with placeholders.

## Final Scenario Comparison

| Labeling Scenario | Best Model | Aggregation | Top-K | Accuracy | F1-Score | ROC-AUC | Log Loss |
|---|---|---|---:|---:|---:|---:|---:|
| 180 Days | XGBoost | L2 Norm | 11 | 0.6642 | 0.6523 | 0.7160 | 0.6160 |
| End-of-Month | CatBoost | Median | 10 | 0.6631 | 0.7169 | 0.7033 | 0.6162 |
| **210 Days** | **XGBoost** | **Median** | **11** | **0.6863** | **0.7551** | **0.7272** | **0.5898** |

## Main Workflow

**Data Integration → Data Cleaning & Filtering → Exploratory Data Analysis → Retention Labeling → Feature Engineering → Train/Validation/Test Split → Model-Specific Preprocessing → SHAP-Based Feature Selection → Optuna Hyperparameter Tuning → Model Comparison → Final Evaluation → Error Analysis**

### Models
- Random Forest
- XGBoost
- CatBoost

### Main Engineered Features
- Visit Count
- Recency Days
- Loyalty Score
- Average Revenue Before
- Service Month

---


# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

# Load Data

In [ ]:
data_path = "path/to/proprietary_service_data.xlsx"
xls = pd.ExcelFile(data_path)

print("Jumlah sheet:", len(xls.sheet_names))
print("Daftar sheet:")
print(xls.sheet_names)

# Data Integration

In [ ]:
dfs = []
sheet_summary = []

for sheet in xls.sheet_names:
    df_sheet = pd.read_excel(data_path,sheet_name=sheet)
    df_sheet = df_sheet.loc[
        :,
        ~df_sheet.columns.astype(str).str.contains(
            "^Unnamed",
            case=False,
            regex=True)]
    df_sheet["SourceSheet"] = sheet

    if "PKBDate" in df_sheet.columns:
        pkb_date_temp = pd.to_datetime(
            df_sheet["PKBDate"],
            errors="coerce",
            dayfirst=True)
        min_date = pkb_date_temp.min()
        max_date = pkb_date_temp.max()
    else:
        min_date = pd.NaT
        max_date = pd.NaT
    sheet_summary.append({
        "Sheet": sheet,
        "Jumlah_Baris": len(df_sheet),
        "Min_PKBDate": min_date,
        "Max_PKBDate": max_date})
    dfs.append(df_sheet)

df_integrated = pd.concat(dfs,ignore_index=True)
df_sheet_summary = pd.DataFrame(sheet_summary)

print("RINGKASAN DATA PER SHEET")
display(df_sheet_summary)
print("\nTotal baris seluruh data:", df_sheet_summary["Jumlah_Baris"].sum())
print("Shape data hasil integrasi:", df_integrated.shape)

In [ ]:
selected_columns = [
    "PKBNo",
    "PKBDate",
    "EquipmentNo",
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor",
    "Kilometer",
    "Estimated Revenue"
]
missing_cols = [
    col
    for col in selected_columns
    if col not in df_integrated.columns]

if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

df_integrated = df_integrated[selected_columns].copy()
print("Shape data setelah pemilihan variabel:", df_integrated.shape)
print("\nInformasi data hasil integrasi:")
df_integrated.info()
print("\nContoh data hasil integrasi:")
display(df_integrated.head(10))

# Data Cleaning

In [ ]:
df = df_integrated.copy()
print("Shape awal data:", df.shape)

In [ ]:
text_cols = [
    "PKBNo",
    "EquipmentNo",
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor"
]

df[text_cols] = df[text_cols].replace(
    r"^\s*$",
    pd.NA,
    regex=True)

## STANDARISASI

In [ ]:
for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.upper())

print("\nData setelah standardisasi teks:")
display(df.head())

In [ ]:
typekendaraan_mapping = {
    "HI LUX": "HILUX",
    "HI ACE": "HIACE",
    "HI-ACE": "HIACE",
    "LAND CRUI": "LANDCRUISER",
    "LANDCRUISE": "LANDCRUISER",
    "LANDCRUIS": "LANDCRUISER",
    "LND-CR-FS": "LANDCRUISER",
    "LND-CRUISER": "LANDCRUISER",
    "LND-CR-ST": "LANDCRUISER",
    "LAND CRUISER": "LANDCRUISER"
}

df["TypeKendaraan"] = (
    df["TypeKendaraan"]
    .replace(typekendaraan_mapping))

print("\nJumlah kategori TypeKendaraan setelah standardisasi:",df["TypeKendaraan"].nunique(dropna=True))

## **KONVERSI TIPE DATA**

In [ ]:
print("\nTipe data sebelum konversi:")
print(df.dtypes)
# Konversi tanggal
df["PKBDate"] = pd.to_datetime(df["PKBDate"], errors="coerce", dayfirst=True)
# Konversi numerik
numeric_cols = ["Kilometer", "Estimated Revenue"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("\nTipe data setelah konversi:")
print(df.dtypes)

## MISSING VALUE

In [ ]:
missing_summary = df.isna().sum().reset_index()
missing_summary.columns = ["Kolom", "Jumlah Missing Value"]
display(missing_summary)

In [ ]:
rows_before = len(df)
missing_before = df.isna().sum()
df = df.dropna(subset=["Service Category"]).copy()
rows_after = len(df)
print("Jumlah baris sebelum handling missing:", rows_before)
print("Jumlah baris sesudah handling missing:", rows_after)
print("Jumlah baris terhapus:", rows_before - rows_after)

## DUPLIKASI

In [ ]:
duplicate_cols = [
    col for col in df.columns
    if col != "SourceSheet"]
dup_all = df.duplicated(
    subset=duplicate_cols).sum()
dup_pkb = df.duplicated(
    subset=["PKBNo"]).sum()
print("\nAudit duplikasi:")
print("Jumlah duplikasi berdasarkan seluruh atribut transaksi:",dup_all)
print("Jumlah duplikasi berdasarkan PKBNo:",dup_pkb)

# Data Filtering

In [ ]:
print("Shape sebelum filtering:", df.shape)
print("\nDistribusi Service Category sebelum filtering:")
display(df["Service Category"].value_counts(dropna=False).reset_index(name="Jumlah").rename(columns={"index": "Service Category"}))
print("\nDistribusi PKBType sebelum filtering:")
display(df["PKBType"].value_counts(dropna=False).reset_index(name="Jumlah").rename(columns={"index": "PKBType"}))

## **SERVICE CATEGORY**

In [ ]:
allowed_service_cat = [
    "SBEK", 
    "SBE", 
    "SBI", 
    "SBIK", 
    "GR", 
    "OILCHA", 
    "EMK"
]
rows_before = len(df)
df_sc = df[df["Service Category"].isin(allowed_service_cat)].copy()
rows_after = len(df_sc)
print("Filtering Service Category")
print("Sebelum:", rows_before)
print("Sesudah:", rows_after)
print("Terhapus:", rows_before - rows_after)

## **PKB TYPE**

In [ ]:
allowed_pkb_type = [
    "BOOKING", 
    "WALK IN", 
    "FIELD SERVICE"
]
rows_before = len(df_sc)
df_filtered = df_sc[df_sc["PKBType"].isin(allowed_pkb_type)].copy()
rows_after = len(df_filtered)
print("Filtering PKBType")
print("Sebelum:", rows_before)
print("Sesudah:", rows_after)
print("Terhapus:", rows_before - rows_after)

In [ ]:
print("Shape setelah filtering:", df_filtered.shape)
print("\nDistribusi Service Category setelah filtering:")
display(df_filtered["Service Category"].value_counts().reset_index(name="Jumlah").rename(columns={"index": "Service Category"}))
print("\nDistribusi PKBType setelah filtering:")
display(df_filtered["PKBType"].value_counts().reset_index(name="Jumlah").rename(columns={"index": "PKBType"}))

# Exploratory Data Analysis

In [ ]:
df_cut = df_filtered.copy()

In [ ]:
eda_df = df_cut.copy()
print("Shape data EDA:", eda_df.shape)
print("\nTipe data:")
print(eda_df.dtypes)
display(eda_df.head())

# **UNIVARIAT**

## NUMERIK

In [ ]:
# UNIVARIAT NUMERIK
num_cols = ["Kilometer", "Estimated Revenue"]
num_summary = eda_df[num_cols].describe().T
num_summary["median"] = eda_df[num_cols].median()
num_summary["Q1"] = eda_df[num_cols].quantile(0.25)
num_summary["Q3"] = eda_df[num_cols].quantile(0.75)
num_summary["IQR"] = num_summary["Q3"] - num_summary["Q1"]
display(num_summary)

In [ ]:
# Revenue > 500 jt
display(
    eda_df.loc[
        eda_df["Estimated Revenue"] > 500_000_000,
        ["PKBNo", "PKBDate", "EquipmentNo", "Service Category", "PKBType", "Kilometer", "Estimated Revenue"]
    ].sort_values("Estimated Revenue", ascending=False))

In [ ]:
display(
    eda_df.loc[
        eda_df["Estimated Revenue"] < 20_000,
        ["PKBNo", "PKBDate", "EquipmentNo", "Service Category", "PKBType", "Kilometer", "Estimated Revenue"]
    ].sort_values("Estimated Revenue", ascending=False))

In [ ]:
display(
    eda_df.loc[
        (eda_df["Estimated Revenue"] < 50_000)
        & (eda_df["PKBType"] == "WALK IN"),
        [
            "PKBNo",
            "PKBDate",
            "EquipmentNo",
            "Service Category",
            "PKBType",
            "Kilometer",
            "Estimated Revenue"
        ]
    ].sort_values(
        "Estimated Revenue",
        ascending=True
    )
)

In [ ]:
# VISUALISASI NUMERIK
for col in num_cols:
    plt.figure(figsize=(8, 4))
    plt.hist(eda_df[col].dropna(), bins=30)
    plt.title(f"Histogram {col}")
    plt.xlabel(col)
    plt.ylabel("Frekuensi")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(6, 4))
    plt.boxplot(eda_df[col].dropna(), vert=True)
    plt.title(f"Boxplot {col}")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## KATEGORIK

In [ ]:
# UNIVARIAT KATEGORIK
cat_cols = [
    "Service Category",
    "PKBType",
    "TypeKendaraan",
    "Service Advisor"
]
for col in cat_cols:
    freq_table = (
        eda_df[col]
        .value_counts(dropna=False)
        .reset_index())
    freq_table.columns = [col, "Jumlah"]
    freq_table["Persentase"] = (freq_table["Jumlah"] / len(eda_df) * 100).round(2)
    print(f"\nDistribusi {col}:")
    display(freq_table.head(50))

In [ ]:
# VISUALISASI KATEGORIK
top_n_map = {
    "Service Category": None,
    "PKBType": None,
    "TypeKendaraan": 10,
    "Service Advisor": 10}
for col in cat_cols:
    vc = eda_df[col].value_counts(dropna=False)
    if top_n_map[col] is not None:
        vc = vc.head(top_n_map[col])
    plt.figure(figsize=(8, 4))
    plt.bar(vc.index.astype(str), vc.values)
    plt.title(f"Distribusi {col}")
    plt.xlabel(col)
    plt.ylabel("Jumlah")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================

df_cut = df_filtered.copy()
eda_df = df_cut.copy()

print("Shape data EDA:", eda_df.shape)

print("\nTipe data:")
print(eda_df.dtypes)

print("\nContoh data:")
display(eda_df.head())


# ============================================================
# ANALISIS UNIVARIAT NUMERIK
# ============================================================

num_cols = [
    "Kilometer",
    "Estimated Revenue"
]

num_summary = pd.DataFrame({
    "Jumlah Data": eda_df[num_cols].count(),
    "Rata-rata": eda_df[num_cols].mean(),
    "Standar Deviasi": eda_df[num_cols].std(),
    "Minimum": eda_df[num_cols].min(),
    "Q1": eda_df[num_cols].quantile(0.25),
    "Median": eda_df[num_cols].median(),
    "Q3": eda_df[num_cols].quantile(0.75),
    "Maksimum": eda_df[num_cols].max()
})

num_summary["IQR"] = (
    num_summary["Q3"] - num_summary["Q1"]
)

print("\nStatistik deskriptif variabel numerik:")
display(num_summary)


# ============================================================
# VISUALISASI VARIABEL NUMERIK
# ============================================================

for col in num_cols:

    plt.figure(figsize=(8, 4))
    plt.hist(
        eda_df[col].dropna(),
        bins=30
    )
    plt.title(f"Histogram {col}")
    plt.xlabel(col)
    plt.ylabel("Frekuensi")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.boxplot(
        eda_df[col].dropna(),
        vert=True
    )
    plt.title(f"Boxplot {col}")
    plt.ylabel(col)
    plt.xticks([])
    plt.tight_layout()
    plt.show()


# ============================================================
# ANALISIS UNIVARIAT KATEGORIK
# ============================================================

cat_cols = [
    "Service Category",
    "PKBType",
    "TypeKendaraan",
    "Service Advisor"
]

categorical_summaries = {}

for col in cat_cols:

    freq_table = (
        eda_df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    freq_table.columns = [
        col,
        "Jumlah"
    ]

    freq_table["Persentase"] = (
        freq_table["Jumlah"]
        / len(eda_df)
        * 100
    ).round(2)

    categorical_summaries[col] = freq_table

    print(f"\nDistribusi {col}:")
    display(freq_table)


# ============================================================
# VISUALISASI VARIABEL KATEGORIK
# ============================================================

top_n_map = {
    "Service Category": None,
    "PKBType": None,
    "TypeKendaraan": 10,
    "Service Advisor": 10
}

for col in cat_cols:

    vc = eda_df[col].value_counts(
        dropna=False
    )

    top_n = top_n_map[col]

    if top_n is not None:
        vc = vc.head(top_n)

    plt.figure(figsize=(8, 4))
    plt.bar(
        vc.index.astype(str),
        vc.values
    )
    plt.title(f"Distribusi {col}")
    plt.xlabel(col)
    plt.ylabel("Jumlah")
    plt.xticks(
        rotation=45,
        ha="right"
    )
    plt.tight_layout()
    plt.show()

# Retention Labeling

In [ ]:
df_label = df_filtered.copy()
df_label = df_label.sort_values(["EquipmentNo", "PKBDate", "PKBNo"]).reset_index(drop=True)
df_label["NextPKBDate"] = (
    df_label
    .groupby("EquipmentNo")["PKBDate"]
    .shift(-1))
df_label["DaysToNext"] = (df_label["NextPKBDate"] - df_label["PKBDate"]).dt.days
max_date = df_label["PKBDate"].max()
print("Max PKBDate:", max_date)
print("Jumlah data awal sebelum cutoff:", len(df_label))

## LABEL 180 HARI

In [ ]:
df_label["DueDate_180D"] = (df_label["PKBDate"] + pd.Timedelta(days=180))
df_label["RETAINED_180D"] = (
    (df_label["DaysToNext"] >= 0) &
    (df_label["NextPKBDate"] <= df_label["DueDate_180D"])).astype(int)
df_cut_180 = df_label[df_label["DueDate_180D"] <= max_date].copy()
print("LABEL 180 HARI")
print("Jumlah data sebelum cutoff:", len(df_label))
print("Jumlah data setelah cutoff :", len(df_cut_180))
print("Jumlah data terhapus       :", len(df_label) - len(df_cut_180))

## LABEL AKHIR BULAN

In [ ]:
df_label["DueDate_AkhirBulan"] = (df_label["DueDate_180D"] + pd.offsets.MonthEnd(0))
df_label["RETAINED_AKHIR_BULAN"] = (
    (df_label["DaysToNext"] >= 0) &
    (df_label["NextPKBDate"] <= df_label["DueDate_AkhirBulan"])).astype(int)
df_cut_akhir_bulan = df_label[df_label["DueDate_AkhirBulan"] <= max_date].copy()
print("\nLABEL TOLERANSI AKHIR BULAN")
print("Jumlah data sebelum cutoff:", len(df_label))
print("Jumlah data setelah cutoff :", len(df_cut_akhir_bulan))
print("Jumlah data terhapus       :", len(df_label) - len(df_cut_akhir_bulan))

## LABEL 210 HARI

In [ ]:
df_label["DueDate_210D"] = (df_label["DueDate_180D"] + pd.Timedelta(days=30))
df_label["RETAINED_210D"] = (
    (df_label["DaysToNext"] >= 0) &
    (df_label["NextPKBDate"] <= df_label["DueDate_210D"])).astype(int)
df_cut_210 = df_label[df_label["DueDate_210D"] <= max_date].copy()
print("\nLABEL 210 HARI")
print("Jumlah data sebelum cutoff:", len(df_label))
print("Jumlah data setelah cutoff :", len(df_cut_210))
print("Jumlah data terhapus       :", len(df_label) - len(df_cut_210))

# Labeling Scenario Comparison

In [ ]:
def make_target_summary(df, target_col):
    summary = (
        df[target_col]
        .value_counts()
        .sort_index()
        .rename_axis(target_col)
        .reset_index(name="Jumlah"))
    summary["Persentase"] = (summary["Jumlah"] / len(df) * 100).round(2)
    return summary
target_summary_180 = make_target_summary(df_cut_180,"RETAINED_180D")
target_summary_akhir_bulan = make_target_summary(df_cut_akhir_bulan,"RETAINED_AKHIR_BULAN")
target_summary_210 = make_target_summary(df_cut_210,"RETAINED_210D")
print("\nDistribusi target label 180 hari:")
display(target_summary_180)
print("\nDistribusi target label toleransi akhir bulan:")
display(target_summary_akhir_bulan)
print("\nDistribusi target label 210 hari:")
display(target_summary_210)

In [ ]:
comparison_label_summary = pd.DataFrame({
    "Skema_Label": [
        "180 Hari",
        "Toleransi Akhir Bulan",
        "210 Hari"
    ],
    "Jumlah_Data": [
        len(df_cut_180),
        len(df_cut_akhir_bulan),
        len(df_cut_210)
    ],
    "Not_Retained_0": [
        (df_cut_180["RETAINED_180D"] == 0).sum(),
        (df_cut_akhir_bulan["RETAINED_AKHIR_BULAN"] == 0).sum(),
        (df_cut_210["RETAINED_210D"] == 0).sum()
    ],
    "Retained_1": [
        (df_cut_180["RETAINED_180D"] == 1).sum(),
        (df_cut_akhir_bulan["RETAINED_AKHIR_BULAN"] == 1).sum(),
        (df_cut_210["RETAINED_210D"] == 1).sum()
    ],
    "Proporsi_0": [
        (df_cut_180["RETAINED_180D"] == 0).mean(),
        (df_cut_akhir_bulan["RETAINED_AKHIR_BULAN"] == 0).mean(),
        (df_cut_210["RETAINED_210D"] == 0).mean()
    ],
    "Proporsi_1": [
        (df_cut_180["RETAINED_180D"] == 1).mean(),
        (df_cut_akhir_bulan["RETAINED_AKHIR_BULAN"] == 1).mean(),
        (df_cut_210["RETAINED_210D"] == 1).mean()
    ]})
comparison_label_summary["Proporsi_0"] = (comparison_label_summary["Proporsi_0"] * 100).round(2)
comparison_label_summary["Proporsi_1"] = (comparison_label_summary["Proporsi_1"] * 100).round(2)
display(comparison_label_summary)

# Feature Engineering

In [ ]:
df_model = df_cut_210.copy()
target_col = "RETAINED_210D"
label_name = "LABEL 210 HARI"
print(label_name)
print("Shape sebelum FE:", df_model.shape)

In [ ]:
required_cols = [
    "EquipmentNo",
    "PKBDate",
    "PKBNo",
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor",
    "Kilometer",
    "Estimated Revenue",
    target_col
]
missing_cols = [
    col for col in required_cols
    if col not in df_model.columns]

if missing_cols:
    raise KeyError(f"Kolom berikut tidak ditemukan: {missing_cols}")

df_model["PKBDate"] = pd.to_datetime(
    df_model["PKBDate"],
    errors="coerce")
df_model["Estimated Revenue"] = pd.to_numeric(
    df_model["Estimated Revenue"],
    errors="coerce")
df_model["Kilometer"] = pd.to_numeric(
    df_model["Kilometer"],
    errors="coerce")
df_model = (
    df_model
    .sort_values(
        [
            "EquipmentNo",
            "PKBDate",
            "PKBNo"
        ])
    .reset_index(drop=True))
group_equipment = df_model.groupby("EquipmentNo",sort=False)

## PEMBENTUKAN FITUR

In [ ]:
# FEATURE 1: VISIT COUNT
# Jumlah transaksi sejak transaksi pertama sampai transaksi saat ini pada kendaraan yang sama.
df_model["Visit Count"] = (
    group_equipment
    .cumcount()
    .add(1)
    .astype("int32"))

# FEATURE 2: RECENCY DAYS
# Selisih hari antara transaksi saat ini dan transaksi sebelumnya pada kendaraan yang sama.
# Nilai -1 menunjukkan transaksi pertama karena belum terdapat transaksi sebelumnya.
df_model["prev_pkbdate"] = (group_equipment["PKBDate"].shift(1))

df_model["Recency Days"] = (df_model["PKBDate"]- df_model["prev_pkbdate"]).dt.days

# Hitung jumlah missing struktural sebelum diganti
n_recency_structural = (df_model["Recency Days"].isna().sum())

# Representasikan transaksi pertama dengan nilai -1
df_model["Recency Days"] = (
    df_model["Recency Days"]
    .fillna(-1)
    .astype("int32"))

print("Jumlah transaksi pertama dengan Recency Days = -1:",n_recency_structural)

# FEATURE 3: LOYALTY SCORE
# Skor intensitas kunjungan berdasarkan jumlah transaksi sampai saat ini dan lama riwayat servis kendaraan.
df_model["first_pkbdate"] = (group_equipment["PKBDate"].transform("min"))
df_model["tenure_days"] = (df_model["PKBDate"]- df_model["first_pkbdate"]).dt.days
df_model["Loyalty Score"] = (
    df_model["Visit Count"]
    / (
        (df_model["tenure_days"] / 365)
        + 1))

# FEATURE 4: AVG REVENUE BEFORE
# Rata-rata Estimated Revenue dari seluruh transaksi sebelum transaksi saat ini pada kendaraan yang sama.
# Nilai 0 pada transaksi pertama menunjukkan belum terdapat transaksi sebelumnya.
df_model["Avg Revenue Before"] = (
    df_model
    .groupby(
        "EquipmentNo",
        sort=False
    )["Estimated Revenue"]
    .transform(
        lambda x: (
            x.shift(1)
            .expanding(min_periods=1)
            .mean())))

df_model.loc[df_model["Visit Count"] == 1,"Avg Revenue Before"] = 0.0

# FEATURE 5: SERVICE MONTH
# Bulan transaksi dalam kategori M01 sampai M12.
df_model["Service Month"] = (
    df_model["PKBDate"]
    .dt.month
    .map(lambda month: f"M{month:02d}")
    .astype("string"))

In [ ]:
helper_cols = [
    "prev_pkbdate",
    "first_pkbdate",
    "tenure_days"]

df_model = df_model.drop(
    columns=helper_cols,
    errors="ignore")

In [ ]:
base_features = [
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor",
    "Kilometer",
    "Estimated Revenue"]

engineered_features = [
    "Visit Count",
    "Recency Days",
    "Loyalty Score",
    "Avg Revenue Before",
    "Service Month"]

features_before_fs = (base_features + engineered_features)

categorical_features = [
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor",
    "Service Month"]

numerical_features = [
    "Kilometer",
    "Estimated Revenue",
    "Visit Count",
    "Recency Days",
    "Loyalty Score",
    "Avg Revenue Before"]

print("\nJumlah fitur asli:", len(base_features))
print("Jumlah fitur hasil feature engineering:", len(engineered_features))
print("Jumlah fitur sebelum feature selection:", len(features_before_fs))
print("Jumlah fitur kategorik:", len(categorical_features))
print("Jumlah fitur numerik:", len(numerical_features))

In [ ]:
feature_summary = pd.DataFrame({
    "No": range(
        1,
        len(features_before_fs) + 1
    ),
    "Fitur": features_before_fs,
    "Sumber": [
        (
            "Fitur Asli"
            if feature in base_features
            else "Feature Engineering"
        )
        for feature in features_before_fs
    ],
    "Tipe": [
        (
            "Kategorik"
            if feature in categorical_features
            else "Numerik"
        )
        for feature in features_before_fs
    ]
})

print("\nShape setelah feature engineering:", df_model.shape)
display(feature_summary)

In [ ]:
n_first_visit = (
    df_model["Visit Count"]
    .eq(1)
    .sum())
n_recency_sentinel = (
    df_model["Recency Days"]
    .eq(-1)
    .sum())
n_recency_missing = (
    df_model["Recency Days"]
    .isna()
    .sum())

# Nilai negatif selain -1 tidak diperbolehkan
n_invalid_negative = (
    (
        df_model["Recency Days"] < 0
    )
    & (
        df_model["Recency Days"] != -1
    )
).sum()

print("Jumlah transaksi pertama:", n_first_visit)
print("Jumlah Recency Days = -1:", n_recency_sentinel)
print("Jumlah missing Recency Days:", n_recency_missing)
print("Transaksi pertama sama dengan Recency Days = -1:", n_first_visit == n_recency_sentinel)

if n_first_visit != n_recency_sentinel:
    raise ValueError(
        "Jumlah transaksi pertama tidak sama dengan jumlah "
        "Recency Days yang bernilai -1.")

if n_recency_missing > 0:
    raise ValueError("Masih terdapat missing value pada Recency Days.")

if n_invalid_negative > 0:
    raise ValueError("Ditemukan nilai negatif Recency Days selain -1.")

## MISSING VALUE AFTER FE

In [ ]:
missing_fe = (
    df_model[features_before_fs]
    .isna()
    .sum()
    .reset_index())

missing_fe.columns = [
    "Fitur",
    "Jumlah Missing"]

missing_fe["Persentase (%)"] = (
    missing_fe["Jumlah Missing"]
    / len(df_model)
    * 100).round(2)

missing_fe = (
    missing_fe
    .sort_values(
        "Jumlah Missing",
        ascending=False
    )
    .reset_index(drop=True))

print("\nMissing value setelah feature engineering:")
display(missing_fe)

In [ ]:
structural_missing_summary = pd.DataFrame({
    "Fitur": ["Recency Days"],
    "Penyebab": ["Transaksi pertama tidak memiliki transaksi sebelumnya"],
    "Representasi": [-1],
    "Jumlah": [n_recency_sentinel],
    "Persentase (%)": [
        round(
            n_recency_sentinel
            / len(df_model)
            * 100,
            2)]})

print("\nRingkasan nilai struktural:")
display(structural_missing_summary)

In [ ]:
preview_cols = [
    "EquipmentNo",
    "PKBDate",
    "PKBNo",
    "Service Category",
    "Estimated Revenue",
    "Visit Count",
    "Recency Days",
    "Loyalty Score",
    "Avg Revenue Before",
    "Service Month",
    target_col]

preview_df = (df_model[preview_cols].head(10).copy())
preview_df["Estimated Revenue"] = (preview_df["Estimated Revenue"].round(2))
preview_df["Avg Revenue Before"] = (preview_df["Avg Revenue Before"].round(2))
preview_df["Loyalty Score"] = (preview_df["Loyalty Score"].round(4))
print("\nContoh hasil feature engineering:")
display(preview_df)

In [ ]:
equipment_counts = (df_model["EquipmentNo"].value_counts())
eligible_equipment = equipment_counts[equipment_counts >= 3]
if len(eligible_equipment) > 0:
    example_equipment = (eligible_equipment.index[0])
    example_history = (
        df_model.loc[
            df_model["EquipmentNo"]
            == example_equipment,
            preview_cols]
        .sort_values(["PKBDate", "PKBNo"])
        .reset_index(drop=True))
    example_history["Estimated Revenue"] = (example_history["Estimated Revenue"].round(2))
    example_history["Avg Revenue Before"] = (example_history["Avg Revenue Before"].round(2))
    example_history["Loyalty Score"] = (example_history["Loyalty Score"].round(4))
    print("\nContoh riwayat feature engineering untuk EquipmentNo:", example_equipment)
    display(example_history.head(10))
else:
    print("\nTidak terdapat EquipmentNo dengan minimal " "tiga transaksi.")

In [ ]:
first_visit_check = pd.DataFrame({
    "Jumlah": [
        (df_model["Recency Days"] == -1).sum(),
        (df_model["Recency Days"] >= 0).sum()],
    "Kategori": [
        "Transaksi pertama",
        "Memiliki transaksi sebelumnya"]})

first_visit_check["Persentase"] = (
    first_visit_check["Jumlah"]
    / len(df_model) * 100)

display(first_visit_check)

In [ ]:
invalid_first_visit = df_model.loc[
    (df_model["Recency Days"] == -1)
    & (df_model["Visit Count"] != 1)]

print("Jumlah Recency Days = -1 tetapi Visit Count bukan 1:", len(invalid_first_visit))

invalid_visit_count = df_model.loc[
    (df_model["Visit Count"] == 1)
    & (df_model["Recency Days"] != -1)]

print("Jumlah Visit Count = 1 tetapi Recency Days bukan -1:", len(invalid_visit_count))

In [ ]:
same_day_summary = pd.DataFrame({
    "Kondisi": [
        "Transaksi pertama",
        "Kunjungan ulang hari yang sama",
        "Kunjungan ulang beda hari"],
    "Jumlah": [
        (df_model["Recency Days"] == -1).sum(),
        (df_model["Recency Days"] == 0).sum(),
        (df_model["Recency Days"] > 0).sum()]})

same_day_summary["Persentase"] = (
    same_day_summary["Jumlah"]
    / len(df_model) * 100)

display(same_day_summary)

In [ ]:
same_day_rows = (
    df_model.loc[
        df_model["Recency Days"] == 0,
        [
            "EquipmentNo",
            "PKBDate",
            "PKBNo",
            "Visit Count",
            "Recency Days"]]
    .sort_values(
        [
            "EquipmentNo",
            "PKBDate",
            "PKBNo"]))

display(same_day_rows.head(20))

In [ ]:
same_day_detail = (
    df_model.loc[
        df_model["Recency Days"] == 0,
        [
            "EquipmentNo",
            "PKBDate",
            "PKBNo",
            "PKBType",
            "Service Category",
            "Kilometer",
            "Estimated Revenue",
            "Visit Count",
            "Recency Days"
        ]
    ]
    .sort_values(
        [
            "EquipmentNo",
            "PKBDate",
            "PKBNo"
        ]
    )
)

display(same_day_detail.head(10))

In [ ]:
same_day_duplicate_check = (
    same_day_detail
    .duplicated(
        subset=[
            "EquipmentNo",
            "PKBDate",
            "PKBNo"
        ],
        keep=False
    )
)

print(
    "Jumlah duplikasi EquipmentNo-PKBDate-PKBNo:",
    same_day_duplicate_check.sum()
)

In [ ]:
# jumlah_equipment_unik = df_model["EquipmentNo"].nunique()
# jumlah_missing_equipment = df_model["EquipmentNo"].isna().sum()
# print("Jumlah EquipmentNo unik   :", jumlah_equipment_unik)
# print("Jumlah EquipmentNo kosong :", jumlah_missing_equipment)
# print("Jumlah seluruh baris       :", len(df_model))

# Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split
X = df_model[features_before_fs].copy()
y = df_model[target_col].copy()
print(f"{label_name}")
print("Shape X     :", X.shape)
print("Shape y     :", y.shape)
print("Target      :", target_col)
print("Jumlah fitur:", len(features_before_fs))

In [ ]:
all_idx = df_model.index.to_numpy()
train_idx, temp_idx = train_test_split(
    all_idx,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y.loc[temp_idx])
print("Train      :", len(train_idx))
print("Validation :", len(val_idx))
print("Test       :", len(test_idx))

In [ ]:
X_train = (
    X.loc[train_idx]
    .reset_index(drop=True))
X_val = (
    X.loc[val_idx]
    .reset_index(drop=True))
X_test = (
    X.loc[test_idx]
    .reset_index(drop=True))
y_train = (
    y.loc[train_idx]
    .reset_index(drop=True))
y_val = (
    y.loc[val_idx]
    .reset_index(drop=True))
y_test = (
    y.loc[test_idx]
    .reset_index(drop=True))
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print()
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

In [ ]:
if len(set(train_idx) & set(val_idx)) > 0:
    raise ValueError(
        "Terdapat data yang sama pada train dan validation."
    )
if len(set(train_idx) & set(test_idx)) > 0:
    raise ValueError(
        "Terdapat data yang sama pada train dan test."
    )
if len(set(val_idx) & set(test_idx)) > 0:
    raise ValueError(
        "Terdapat data yang sama pada validation dan test."
    )
if (
    len(train_idx)
    + len(val_idx)
    + len(test_idx)
    != len(df_model)
):
    raise ValueError(
        "Jumlah data setelah splitting tidak sama "
        "dengan jumlah data awal."
    )
print("Tidak ada baris yang tumpang tindih.")
print("Seluruh data berhasil dibagi.")

In [ ]:
split_summary = pd.DataFrame({
    "Dataset": [
        "Train",
        "Validation",
        "Test"
    ],
    "Jumlah Data": [
        len(y_train),
        len(y_val),
        len(y_test)
    ],
    "Jumlah Tidak Retained": [
        (y_train == 0).sum(),
        (y_val == 0).sum(),
        (y_test == 0).sum()
    ],
    "Jumlah Retained": [
        (y_train == 1).sum(),
        (y_val == 1).sum(),
        (y_test == 1).sum()
    ],
    "Proporsi Tidak Retained (%)": [
        (y_train == 0).mean() * 100,
        (y_val == 0).mean() * 100,
        (y_test == 0).mean() * 100
    ],
    "Proporsi Retained (%)": [
        (y_train == 1).mean() * 100,
        (y_val == 1).mean() * 100,
        (y_test == 1).mean() * 100
    ],
    "Proporsi Data (%)": [
        len(y_train) / len(y) * 100,
        len(y_val) / len(y) * 100,
        len(y_test) / len(y) * 100
    ]
})

percentage_cols = [
    "Proporsi Tidak Retained (%)",
    "Proporsi Retained (%)",
    "Proporsi Data (%)"
]

split_summary[percentage_cols] = (
    split_summary[percentage_cols]
    .round(2))

display(split_summary)

# Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import TargetEncoder

TARGET_ENCODING_FOLDS = 5
MISSING_CATEGORY = "__MISSING__"
UNKNOWN_CATEGORY = "__UNKNOWN__"

num_cols = [
    "Kilometer",
    "Estimated Revenue",
    "Visit Count",
    "Recency Days",
    "Loyalty Score",
    "Avg Revenue Before"]

cat_cols = [
    "PKBType",
    "Service Category",
    "TypeKendaraan",
    "Service Advisor",
    "Service Month"]

preprocessing_cols = num_cols + cat_cols

if set(preprocessing_cols) != set(features_before_fs):
    missing_features = sorted(set(features_before_fs) - set(preprocessing_cols))
    extra_features = sorted(set(preprocessing_cols) - set(features_before_fs))

    raise ValueError(
        f"Fitur belum sesuai.\n"
        f"Belum diklasifikasikan: {missing_features}\n"
        f"Fitur tambahan: {extra_features}")

if set(num_cols).intersection(cat_cols):
    raise ValueError("Terdapat fitur yang masuk kelompok numerik dan kategorik.")

if len(preprocessing_cols) != len(set(preprocessing_cols)):
    raise ValueError("Terdapat fitur duplikat pada daftar preprocessing.")

print("Jumlah fitur numerik   :", len(num_cols))
print("Jumlah fitur kategorik :", len(cat_cols))
print("Total fitur            :", len(preprocessing_cols))

In [ ]:
def prepare_raw_features(X_data):
    X_result = X_data[features_before_fs].copy()
    X_result[num_cols] = (
        X_result[num_cols]
        .apply(pd.to_numeric, errors="coerce")
        .astype(float))
    for col in cat_cols:
        X_result[col] = (
            X_result[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))
    return X_result

X_train_raw = prepare_raw_features(X_train)
X_val_raw = prepare_raw_features(X_val)
X_test_raw = prepare_raw_features(X_test)

print("X_train raw:", X_train_raw.shape)
print("X_val raw  :", X_val_raw.shape)
print("X_test raw :", X_test_raw.shape)

In [ ]:
raw_datasets = {
    "Train": X_train_raw,
    "Validation": X_val_raw,
    "Test": X_test_raw}

raw_audit = []
for dataset_name, X_data in raw_datasets.items():
    numeric_missing = int(X_data[num_cols].isna().sum().sum())
    numeric_infinity = int(np.isinf(X_data[num_cols].to_numpy()).sum())
    categorical_missing = int(X_data[cat_cols].isna().sum().sum())
    raw_audit.append({
        "Dataset": dataset_name,
        "Jumlah Data": len(X_data),
        "Jumlah Fitur": X_data.shape[1],
        "Missing Numerik": numeric_missing,
        "Infinity Numerik": numeric_infinity,
        "Missing Kategorik": categorical_missing})

raw_audit = pd.DataFrame(raw_audit)
display(raw_audit)

if (raw_audit[
        [
            "Missing Numerik",
            "Infinity Numerik",
            "Missing Kategorik"
        ]
    ].to_numpy().sum()
    > 0):
    raise ValueError("Data masih mengandung missing value atau infinity.")
print("Status data dasar: valid.")

## RANDOM FOREST

In [ ]:
target_encoder_rf = TargetEncoder(
    target_type="binary",
    smooth="auto",
    cv=TARGET_ENCODING_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE)

train_cat_rf = pd.DataFrame(
    target_encoder_rf.fit_transform(
        X_train_raw[cat_cols],
        y_train.to_numpy(dtype=int)),
    columns=cat_cols,
    index=X_train_raw.index)

val_cat_rf = pd.DataFrame(
    target_encoder_rf.transform(
        X_val_raw[cat_cols]),
    columns=cat_cols,
    index=X_val_raw.index)

test_cat_rf = pd.DataFrame(
    target_encoder_rf.transform(
        X_test_raw[cat_cols]),
    columns=cat_cols,
    index=X_test_raw.index)

X_train_rf = pd.concat(
    [X_train_raw[num_cols], train_cat_rf],
    axis=1
)[features_before_fs].astype(float)

X_val_rf = pd.concat(
    [X_val_raw[num_cols], val_cat_rf],
    axis=1
)[features_before_fs].astype(float)

X_test_rf = pd.concat(
    [X_test_raw[num_cols], test_cat_rf],
    axis=1
)[features_before_fs].astype(float)

print("Random Forest:")
print("Train     :", X_train_rf.shape)
print("Validation:", X_val_rf.shape)
print("Test      :", X_test_rf.shape)
print("Global target mean train:", round(float(target_encoder_rf.target_mean_), 6))

In [ ]:
rf_unseen_results = []

for dataset_name, X_data in {
    "Validation": X_val_raw,
    "Test": X_test_raw}.items():

    for col in cat_cols:
        train_categories = set(X_train_raw[col].unique())
        current_categories = set(X_data[col].unique())
        unseen_categories = sorted(current_categories - train_categories)
        rf_unseen_results.append({
            "Dataset": dataset_name,
            "Fitur": col,
            "Jumlah Kategori Baru": len(unseen_categories),
            "Contoh Kategori Baru": (
                ", ".join(map(str, unseen_categories[:5]))
                if unseen_categories
                else "-")})

rf_unseen_summary = pd.DataFrame(rf_unseen_results)
display(rf_unseen_summary)

## XGBOOST

In [ ]:
X_train_xgb = X_train_raw.copy()
X_val_xgb = X_val_raw.copy()
X_test_xgb = X_test_raw.copy()

xgb_category_levels = {}
xgb_unseen_results = []

for col in cat_cols:
    train_categories = list(pd.unique(X_train_raw[col]))
    categories_with_unknown = (train_categories + [UNKNOWN_CATEGORY])
    xgb_category_levels[col] = categories_with_unknown
    category_dtype = pd.CategoricalDtype(categories=categories_with_unknown)
    
    X_train_xgb[col] = (X_train_raw[col].astype(category_dtype))
    
    val_unseen_mask = ~X_val_raw[col].isin(train_categories)
    X_val_xgb[col] = (
        X_val_raw[col]
        .where(~val_unseen_mask, UNKNOWN_CATEGORY)
        .astype(category_dtype))
    
    test_unseen_mask = ~X_test_raw[col].isin(train_categories)
    X_test_xgb[col] = (
        X_test_raw[col]
        .where(~test_unseen_mask, UNKNOWN_CATEGORY)
        .astype(category_dtype))
    
    xgb_unseen_results.extend([
        {
            "Dataset": "Validation",
            "Fitur": col,
            "Jumlah Baris Kategori Baru": int(val_unseen_mask.sum())},
        {
            "Dataset": "Test",
            "Fitur": col,
            "Jumlah Baris Kategori Baru": int(test_unseen_mask.sum())}])

X_train_xgb[num_cols] = X_train_xgb[num_cols].astype(float)
X_val_xgb[num_cols] = X_val_xgb[num_cols].astype(float)
X_test_xgb[num_cols] = X_test_xgb[num_cols].astype(float)

X_train_xgb = X_train_xgb[features_before_fs]
X_val_xgb = X_val_xgb[features_before_fs]
X_test_xgb = X_test_xgb[features_before_fs]

print("XGBoost:")
print("Train     :", X_train_xgb.shape)
print("Validation:", X_val_xgb.shape)
print("Test      :", X_test_xgb.shape)

print("\nTipe fitur kategorik XGBoost:")
print(X_train_xgb[cat_cols].dtypes)

In [ ]:
xgb_unseen_summary = pd.DataFrame(xgb_unseen_results)
display(xgb_unseen_summary)

## CATBOOST

In [ ]:
X_train_cb = X_train_raw.copy()
X_val_cb = X_val_raw.copy()
X_test_cb = X_test_raw.copy()

for X_data in [
    X_train_cb,
    X_val_cb,
    X_test_cb]:

    X_data[num_cols] = X_data[num_cols].astype(float)

    for col in cat_cols:
        X_data[col] = (
            X_data[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))

X_train_cb = X_train_cb[features_before_fs]
X_val_cb = X_val_cb[features_before_fs]
X_test_cb = X_test_cb[features_before_fs]

print("CatBoost:")
print("Train     :", X_train_cb.shape)
print("Validation:", X_val_cb.shape)
print("Test      :", X_test_cb.shape)

print("\nTipe fitur kategorik CatBoost:")
print(X_train_cb[cat_cols].dtypes)

In [ ]:
def audit_numeric_data(
    dataset_name,
    model_name,
    X_data):
    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Jumlah Data": len(X_data),
        "Jumlah Fitur": X_data.shape[1],
        "Missing Value": int(X_data.isna().sum().sum()),
        "Infinity Numerik": int(np.isinf(X_data[num_cols].to_numpy(dtype=float)).sum())}

preprocessing_audit = []

model_datasets = {
    "Random Forest": {
        "Train": X_train_rf,
        "Validation": X_val_rf,
        "Test": X_test_rf},
    "XGBoost": {
        "Train": X_train_xgb,
        "Validation": X_val_xgb,
        "Test": X_test_xgb},
    "CatBoost": {
        "Train": X_train_cb,
        "Validation": X_val_cb,
        "Test": X_test_cb}}

for model_name, datasets in model_datasets.items():
    for dataset_name, X_data in datasets.items():
        preprocessing_audit.append(audit_numeric_data(dataset_name, model_name, X_data))

preprocessing_summary = pd.DataFrame(preprocessing_audit)
display(preprocessing_summary)

In [ ]:
expected_columns = list(features_before_fs)
for model_name, datasets in model_datasets.items():
    for dataset_name, X_data in datasets.items():
        if list(X_data.columns) != expected_columns:
            raise ValueError(
                f"Urutan fitur {model_name} — "
                f"{dataset_name} tidak sesuai.")
        if X_data.isna().any().any():
            raise ValueError(
                f"Terdapat missing value pada "
                f"{model_name} — {dataset_name}.")
        if np.isinf(X_data[num_cols].to_numpy(dtype=float)).any():
            raise ValueError(
                f"Terdapat infinity pada "
                f"{model_name} — {dataset_name}.")

for X_model in [
    X_train_rf,
    X_train_xgb,
    X_train_cb]:
    if not X_model.index.equals(y_train.index):
        raise ValueError("Indeks data train tidak sejajar dengan y_train.")
for X_model in [
    X_val_rf,
    X_val_xgb,
    X_val_cb]:
    if not X_model.index.equals(y_val.index):
        raise ValueError("Indeks data validation tidak sejajar dengan y_val.")
for X_model in [
    X_test_rf,
    X_test_xgb,
    X_test_cb]:
    if not X_model.index.equals(y_test.index):
        raise ValueError("Indeks data test tidak sejajar dengan y_test.")

print("Status preprocessing: berhasil.")
print()
print("Random Forest : Target Encoding")
print("XGBoost       : native categorical")
print("CatBoost      : native categorical")

# Model Training, Feature Selection, and Tuning

In [ ]:
!pip install -q optuna xgboost catboost shap

import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
import xgboost as xgb

from IPython.display import display
from optuna.samplers import TPESampler
from optuna.trial import TrialState

from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool


# ============================================================
# PENGATURAN UMUM
# ============================================================

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42

N_SPLITS = 5
N_TRIALS_INITIAL = 30
N_TRIALS_TOPK = 30

EARLY_STOPPING_ROUNDS = 50
TARGET_ENCODING_FOLDS = 5
THRESHOLD = 0.50

MIN_K = 9

SHAP_SAMPLE_SIZE = None

MISSING_CATEGORY = "__MISSING__"
UNKNOWN_CATEGORY = "__UNKNOWN__"

LABEL_DISPLAY_NAME = label_name

# Objective utama optimasi
OPTIMIZATION_METRIC = "Log Loss"
OPTIMIZATION_DIRECTION = "minimize"


MODEL_ORDER = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

AGGREGATION_METHODS = [
    "Arithmetic Mean",
    "Median",
    "Geometric Mean",
    "L2 Norm"
]

# Metrik untuk membandingkan model pada tahap evaluasi
SELECTION_METRICS = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC"
]

# Seluruh metrik yang dihitung
ALL_METRICS = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC",
    "PR-AUC",
    "Log Loss",
    "Brier Score"
]

PARAMETER_ORDER = {
    "Random Forest": [
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
        "bootstrap"
    ],
    "XGBoost": [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "gamma",
        "min_child_weight",
        "reg_alpha",
        "reg_lambda"
    ],
    "CatBoost": [
        "iterations",
        "depth",
        "learning_rate",
        "l2_leaf_reg",
        "bagging_temperature",
        "random_strength"
    ]
}


# ============================================================
# PEMERIKSAAN OBJEK
# ============================================================

required_objects = [
    "X_train_raw",
    "X_val_raw",
    "X_test_raw",

    "X_train_xgb",
    "X_val_xgb",
    "X_test_xgb",

    "X_train_cb",
    "X_val_cb",
    "X_test_cb",

    "y_train",
    "y_val",
    "y_test",

    "num_cols",
    "cat_cols",
    "features_before_fs",
    "label_name"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Objek berikut belum tersedia: {missing_objects}"
    )

MAX_K = len(features_before_fs)

if MIN_K < 1:
    raise ValueError("MIN_K minimal bernilai 1.")

if MAX_K != len(features_before_fs):
    raise ValueError(
        "MAX_K harus sama dengan jumlah seluruh fitur."
    )

if MIN_K > MAX_K:
    raise ValueError(
        "MIN_K tidak boleh lebih besar daripada MAX_K."
    )


# ============================================================
# INFORMASI KONFIGURASI
# ============================================================

print("Skenario pelabelan :", LABEL_DISPLAY_NAME)
print("Jumlah fitur       :", len(features_before_fs))
print("Top-K              :", MIN_K, "sampai", MAX_K)
print("Trial tuning awal  :", N_TRIALS_INITIAL)
print("Trial tiap Top-K   :", N_TRIALS_TOPK)
print("Cross-validation   :", N_SPLITS, "fold")
print("Objective optimasi : Mean CV Log Loss terendah")
print("Arah optimasi      :", OPTIMIZATION_DIRECTION)


# ============================================================
# FUNGSI BANTU
# ============================================================

def display_without_index(
    dataframe,
    formats=None
):
    """Menampilkan DataFrame tanpa indeks."""
    styler = dataframe.style.hide(axis="index")

    if formats:
        styler = styler.format(formats)

    display(styler)


def params_to_text(params):
    """Mengubah dictionary hyperparameter menjadi teks JSON."""
    return json.dumps(
        params,
        ensure_ascii=False,
        sort_keys=True
    )


# ============================================================
# PEMBENTUKAN FOLD CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_splits = list(
    cv.split(
        np.zeros(len(y_train)),
        y_train
    )
)

for fold_number, (train_pos, valid_pos) in enumerate(
    cv_splits,
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"train={len(train_pos)}, "
        f"validation={len(valid_pos)}"
    )

In [ ]:
def prepare_rf_fold(
    X_fold_train_raw,
    y_fold_train,
    X_fold_valid_raw,
    selected_features):
    selected_num = [
        col
        for col in selected_features
        if col in num_cols]
    selected_cat = [
        col
        for col in selected_features
        if col in cat_cols]
    
    train_parts = []
    valid_parts = []

    if selected_num:
        train_num = (
            X_fold_train_raw[selected_num]
            .astype(float)
            .reset_index(drop=True))
        valid_num = (
            X_fold_valid_raw[selected_num]
            .astype(float)
            .reset_index(drop=True))
        train_parts.append(train_num)
        valid_parts.append(valid_num)

    if selected_cat:
        train_cat = (
            X_fold_train_raw[selected_cat]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str)
            .reset_index(drop=True))
        valid_cat = (
            X_fold_valid_raw[selected_cat]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str)
            .reset_index(drop=True))
        encoder = TargetEncoder(
            target_type="binary",
            smooth="auto",
            cv=TARGET_ENCODING_FOLDS,
            shuffle=True,
            random_state=RANDOM_STATE)
        train_encoded = pd.DataFrame(
            encoder.fit_transform(
                train_cat,
                y_fold_train.to_numpy(dtype=int)), columns=selected_cat)
        valid_encoded = pd.DataFrame(
            encoder.transform(valid_cat),
            columns=selected_cat)
        train_parts.append(train_encoded)
        valid_parts.append(valid_encoded)

    X_fold_train = (
        pd.concat(train_parts, axis=1)
        [selected_features].astype(float))
    X_fold_valid = (
        pd.concat(valid_parts, axis=1)
        [selected_features].astype(float))

    return (X_fold_train, X_fold_valid, [])

def prepare_xgb_fold(
    X_fold_train_raw,
    X_fold_valid_raw,
    selected_features):
    X_fold_train = (
        X_fold_train_raw[selected_features]
        .copy()
        .reset_index(drop=True))
    X_fold_valid = (
        X_fold_valid_raw[selected_features]
        .copy()
        .reset_index(drop=True))
    selected_num = [
        col
        for col in selected_features
        if col in num_cols]
    selected_cat = [
        col
        for col in selected_features
        if col in cat_cols]
    if selected_num:
        X_fold_train[selected_num] = (X_fold_train[selected_num].astype(float))
        X_fold_valid[selected_num] = (X_fold_valid[selected_num].astype(float))
    for col in selected_cat:
        train_values = (
            X_fold_train[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))
        valid_values = (
            X_fold_valid[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))
        train_categories = list(pd.unique(train_values))
        categories = train_categories.copy()
        if UNKNOWN_CATEGORY not in categories:
            categories.append(UNKNOWN_CATEGORY)
        category_dtype = pd.CategoricalDtype(categories=categories)
        valid_values = valid_values.where(valid_values.isin(train_categories), UNKNOWN_CATEGORY)
        X_fold_train[col] = (train_values.astype(category_dtype))
        X_fold_valid[col] = (valid_values.astype(category_dtype))

    return (
        X_fold_train,
        X_fold_valid,
        selected_cat)

def prepare_catboost_fold(
    X_fold_train_raw,
    X_fold_valid_raw,
    selected_features):
    X_fold_train = (
        X_fold_train_raw[selected_features]
        .copy()
        .reset_index(drop=True))
    X_fold_valid = (
        X_fold_valid_raw[selected_features]
        .copy()
        .reset_index(drop=True))
    selected_num = [
        col
        for col in selected_features
        if col in num_cols]
    selected_cat = [
        col
        for col in selected_features
        if col in cat_cols]

    if selected_num:
        X_fold_train[selected_num] = (X_fold_train[selected_num].astype(float))
        X_fold_valid[selected_num] = (X_fold_valid[selected_num].astype(float))
    for col in selected_cat:
        X_fold_train[col] = (
            X_fold_train[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))
        X_fold_valid[col] = (
            X_fold_valid[col]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .astype(str))

    return (X_fold_train, X_fold_valid, selected_cat)

def prepare_fold_data(
    model_name,
    X_fold_train_raw,
    y_fold_train,
    X_fold_valid_raw,
    selected_features):
    if model_name == "Random Forest":
        return prepare_rf_fold(
            X_fold_train_raw=X_fold_train_raw,
            y_fold_train=y_fold_train,
            X_fold_valid_raw=X_fold_valid_raw,
            selected_features=selected_features)
    if model_name == "XGBoost":
        return prepare_xgb_fold(
            X_fold_train_raw=X_fold_train_raw,
            X_fold_valid_raw=X_fold_valid_raw,
            selected_features=selected_features)
    if model_name == "CatBoost":
        return prepare_catboost_fold(
            X_fold_train_raw=X_fold_train_raw,
            X_fold_valid_raw=X_fold_valid_raw,
            selected_features=selected_features)

    raise ValueError(f"Model tidak dikenali: {model_name}")

In [ ]:
def suggest_parameters(trial, model_name):
    if model_name == "Random Forest":
        return {
            "n_estimators": trial.suggest_categorical(
                "n_estimators",
                [100, 200, 300, 500]
            ),
            "max_depth": trial.suggest_categorical(
                "max_depth",
                [5, 10, 15, 20, None]
            ),
            "min_samples_split": trial.suggest_categorical(
                "min_samples_split",
                [2, 5, 10, 20]
            ),
            "min_samples_leaf": trial.suggest_categorical(
                "min_samples_leaf",
                [1, 2, 4, 8]
            ),
            "max_features": trial.suggest_categorical(
                "max_features",
                ["sqrt", "log2", None]
            ),
            "bootstrap": trial.suggest_categorical(
                "bootstrap",
                [True, False]
            )
        }

    if model_name == "XGBoost":
        return {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                1000,
                step=1
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.30,
                log=True
            ),
            "subsample": trial.suggest_float(
                "subsample",
                0.50,
                1.00
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.50,
                1.00
            ),
            "gamma": trial.suggest_float(
                "gamma",
                0.0,
                5.0
            ),
            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                0.0,
                5.0
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                0.001,
                100.0,
                log=True
            )
        }

    if model_name == "CatBoost":
        return {
            "iterations": trial.suggest_int(
                "iterations",
                200,
                1000,
                step=1
            ),
            "depth": trial.suggest_int(
                "depth",
                4,
                10
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.30,
                log=True
            ),
            "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg",
                1.0,
                100.0,
                log=True
            ),
            "bagging_temperature": trial.suggest_float(
                "bagging_temperature",
                0.0,
                10.0
            ),
            "random_strength": trial.suggest_float(
                "random_strength",
                0.0,
                10.0
            )
        }

    raise ValueError(
        f"Model tidak dikenali: {model_name}"
    )


def build_model(
    model_name,
    params,
    use_early_stopping=True
):
    if model_name == "Random Forest":
        return RandomForestClassifier(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    if model_name == "XGBoost":
        extra_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "enable_categorical": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbosity": 0
        }

        if use_early_stopping:
            extra_params[
                "early_stopping_rounds"
            ] = EARLY_STOPPING_ROUNDS

        return XGBClassifier(
            **params,
            **extra_params
        )

    if model_name == "CatBoost":
        return CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=RANDOM_STATE,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )

    raise ValueError(
        f"Model tidak dikenali: {model_name}"
    )


def fit_model(
    model_name,
    model,
    X_train_data,
    y_train_data,
    X_valid_data=None,
    y_valid_data=None,
    cat_features=None,
    include_train_in_eval=False,
    use_early_stopping=True
):
    if model_name == "Random Forest":
        model.fit(
            X_train_data,
            y_train_data
        )

        return model

    if (
        X_valid_data is None
        or y_valid_data is None
    ):
        raise ValueError(
            f"{model_name} membutuhkan data validation."
        )

    if model_name == "XGBoost":
        if include_train_in_eval:
            eval_set = [
                (
                    X_train_data,
                    y_train_data
                ),
                (
                    X_valid_data,
                    y_valid_data
                )
            ]
        else:
            eval_set = [
                (
                    X_valid_data,
                    y_valid_data
                )
            ]

        model.fit(
            X_train_data,
            y_train_data,
            eval_set=eval_set,
            verbose=False
        )

        return model

    if model_name == "CatBoost":
        model.fit(
            X_train_data,
            y_train_data,
            eval_set=(
                X_valid_data,
                y_valid_data
            ),
            cat_features=cat_features,
            use_best_model=use_early_stopping,
            early_stopping_rounds=(
                EARLY_STOPPING_ROUNDS
                if use_early_stopping
                else None
            ),
            verbose=False
        )

        return model

    raise ValueError(
        f"Model tidak dikenali: {model_name}"
    )


def calculate_metrics(
    y_true,
    y_proba,
    threshold=THRESHOLD
):
    y_true_array = np.asarray(
        y_true,
        dtype=int
    )

    y_proba = np.asarray(
        y_proba,
        dtype=float
    )

    y_proba_safe = np.clip(
        y_proba,
        1e-15,
        1 - 1e-15
    )

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    metrics = {
        "Accuracy": accuracy_score(
            y_true_array,
            y_pred
        ),
        "Precision": precision_score(
            y_true_array,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true_array,
            y_pred,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_true_array,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true_array,
            y_proba
        ),
        "PR-AUC": average_precision_score(
            y_true_array,
            y_proba
        ),
        "Log Loss": log_loss(
            y_true_array,
            y_proba_safe,
            labels=[0, 1]
        ),
        "Brier Score": brier_score_loss(
            y_true_array,
            y_proba_safe
        )
    }

    return metrics, y_pred


def create_logloss_objective(
    model_name,
    selected_features
):
    """
    Membentuk objective Optuna berbasis Mean CV Log Loss.

    Mean CV Log Loss digunakan sebagai nilai objective yang
    diminimalkan. Mean CV ROC-AUC tetap dihitung dan disimpan
    sebagai informasi pendamping.
    """

    def objective(trial):
        params = suggest_parameters(
            trial=trial,
            model_name=model_name
        )

        fold_logloss = []
        fold_auc = []

        for train_pos, valid_pos in cv_splits:
            X_fold_train_raw = (
                X_train_raw
                .iloc[train_pos]
                .reset_index(drop=True)
            )

            X_fold_valid_raw = (
                X_train_raw
                .iloc[valid_pos]
                .reset_index(drop=True)
            )

            y_fold_train = (
                y_train
                .iloc[train_pos]
                .reset_index(drop=True)
            )

            y_fold_valid = (
                y_train
                .iloc[valid_pos]
                .reset_index(drop=True)
            )

            (
                X_fold_train,
                X_fold_valid,
                selected_cat
            ) = prepare_fold_data(
                model_name=model_name,
                X_fold_train_raw=X_fold_train_raw,
                y_fold_train=y_fold_train,
                X_fold_valid_raw=X_fold_valid_raw,
                selected_features=selected_features
            )

            model = build_model(
                model_name=model_name,
                params=params,
                use_early_stopping=True
            )

            model = fit_model(
                model_name=model_name,
                model=model,
                X_train_data=X_fold_train,
                y_train_data=y_fold_train,
                X_valid_data=X_fold_valid,
                y_valid_data=y_fold_valid,
                cat_features=selected_cat,
                include_train_in_eval=False,
                use_early_stopping=True
            )

            y_valid_proba = (
                model.predict_proba(
                    X_fold_valid
                )[:, 1]
            )

            y_valid_proba_safe = np.clip(
                np.asarray(
                    y_valid_proba,
                    dtype=float
                ),
                1e-15,
                1 - 1e-15
            )

            current_logloss = log_loss(
                y_fold_valid,
                y_valid_proba_safe,
                labels=[0, 1]
            )

            current_auc = roc_auc_score(
                y_fold_valid,
                y_valid_proba
            )

            fold_logloss.append(
                float(current_logloss)
            )

            fold_auc.append(
                float(current_auc)
            )

            del (
                model,
                X_fold_train,
                X_fold_valid,
                X_fold_train_raw,
                X_fold_valid_raw,
                y_fold_train,
                y_fold_valid,
                y_valid_proba,
                y_valid_proba_safe
            )

            gc.collect()

        mean_logloss = float(
            np.mean(fold_logloss)
        )

        mean_auc = float(
            np.mean(fold_auc)
        )

        # Menyimpan hasil setiap fold
        trial.set_user_attr(
            "fold_logloss",
            [
                float(value)
                for value in fold_logloss
            ]
        )

        trial.set_user_attr(
            "fold_auc",
            [
                float(value)
                for value in fold_auc
            ]
        )

        # Menyimpan rata-rata kedua metrik
        trial.set_user_attr(
            "mean_cv_log_loss",
            mean_logloss
        )

        trial.set_user_attr(
            "mean_cv_roc_auc",
            mean_auc
        )

        # Objective Optuna: Log Loss terendah
        return mean_logloss

    return objective


def study_trials_table(
    study,
    model_name,
    context_columns=None
):
    """
    Membentuk tabel seluruh trial yang selesai.

    Tabel diurutkan berdasarkan Mean CV Log Loss terendah.
    Mean CV ROC-AUC tetap ditampilkan sebagai metrik pendamping.
    """

    context_columns = (
        context_columns
        or {}
    )

    rows = []

    for trial in study.trials:
        if (
            trial.state != TrialState.COMPLETE
            or trial.value is None
        ):
            continue

        row = {
            **context_columns,
            "Trial": trial.number,
            "Mean CV Log Loss": float(
                trial.value
            ),
            "Mean CV ROC-AUC": trial.user_attrs.get(
                "mean_cv_roc_auc",
                np.nan
            )
        }

        for parameter in PARAMETER_ORDER[
            model_name
        ]:
            row[parameter] = trial.params.get(
                parameter
            )

        rows.append(row)

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = (
        result
        .sort_values(
            by=[
                "Mean CV Log Loss",
                "Mean CV ROC-AUC",
                "Trial"
            ],
            ascending=[
                True,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [ ]:
# ============================================================
# BLOK 3 — TUNING AWAL SELURUH FITUR
# UNTUK MODEL PEMBENTUK RANKING SHAP
# OBJECTIVE: MEAN CV LOG LOSS TERENDAH
# DENGAN CHECKPOINT, RESUME, DAN OUTPUT TOP 5
# ============================================================

import os
import json


# ------------------------------------------------------------
# Fungsi untuk membentuk nama file yang aman
# ------------------------------------------------------------

def safe_file_name(text):
    return (
        str(text)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("—", "_")
        .replace(":", "_")
    )


# ------------------------------------------------------------
# Folder penyimpanan hasil tuning awal
#
# Nama folder dibuat berbeda dari eksperimen lama berbasis
# ROC-AUC agar study lama tidak bercampur dengan study baru.
# ------------------------------------------------------------

INITIAL_TUNING_DIR = (
    "./artifacts/"
    f"initial_tuning_logloss_"
    f"{safe_file_name(LABEL_DISPLAY_NAME)}"
)

os.makedirs(
    INITIAL_TUNING_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Database Optuna untuk resume hingga tingkat trial
# ------------------------------------------------------------

INITIAL_OPTUNA_DB_PATH = os.path.join(
    INITIAL_TUNING_DIR,
    "initial_tuning_logloss_optuna.db"
)

INITIAL_OPTUNA_STORAGE = (
    f"sqlite:///{INITIAL_OPTUNA_DB_PATH}"
)


print(
    "Folder hasil tuning awal:",
    INITIAL_TUNING_DIR
)

print(
    "Database Optuna:",
    INITIAL_OPTUNA_DB_PATH
)

print(
    "Objective optimasi:",
    "Mean CV Log Loss terendah"
)

print(
    "Metrik pendamping:",
    "Mean CV ROC-AUC"
)


# ------------------------------------------------------------
# Objek penyimpanan hasil pada notebook
# ------------------------------------------------------------

initial_tuning_studies = {}

initial_tuning_times = {}

initial_best_params = {}

initial_best_results = {}

initial_trials_full_parts = []

initial_top5_parts = []

initial_tuning_summary_rows = []


# ------------------------------------------------------------
# Melakukan tuning awal setiap model
# ------------------------------------------------------------

for model_name in MODEL_ORDER:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "TUNING AWAL SELURUH FITUR "
        f"— {model_name}"
    )

    print(
        "=" * 60
    )


    # Nama study dibuat berbeda dari eksperimen ROC-AUC lama
    study_name = (
        f"initial_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}_"
        f"{safe_file_name(model_name)}"
    )


    # --------------------------------------------------------
    # Membuat atau memuat study
    #
    # direction="minimize" karena nilai Log Loss yang lebih
    # rendah menunjukkan hasil yang lebih baik.
    # --------------------------------------------------------

    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=TPESampler(
            seed=RANDOM_STATE
        ),
        storage=INITIAL_OPTUNA_STORAGE,
        load_if_exists=True
    )


    # --------------------------------------------------------
    # Menghitung jumlah trial yang sudah selesai
    # --------------------------------------------------------

    completed_trials = [
        trial
        for trial in study.trials
        if (
            trial.state == TrialState.COMPLETE
            and trial.value is not None
        )
    ]


    n_completed = len(
        completed_trials
    )


    remaining_trials = max(
        0,
        N_TRIALS_INITIAL - n_completed
    )


    print(
        "Trial selesai sebelumnya :",
        n_completed
    )

    print(
        "Sisa trial               :",
        remaining_trials
    )

    print(
        "Target total trial        :",
        N_TRIALS_INITIAL
    )


    # --------------------------------------------------------
    # Mengambil waktu tuning yang telah tersimpan
    # --------------------------------------------------------

    previous_time = float(
        study.user_attrs.get(
            "cumulative_tuning_seconds",
            0.0
        )
    )


    # --------------------------------------------------------
    # Melanjutkan tuning jika masih ada trial tersisa
    # --------------------------------------------------------

    if remaining_trials > 0:

        start_time = time.perf_counter()


        study.optimize(
            create_logloss_objective(
                model_name=model_name,
                selected_features=list(
                    features_before_fs
                )
            ),
            n_trials=remaining_trials,
            n_jobs=1,
            gc_after_trial=True
        )


        current_elapsed_time = (
            time.perf_counter()
            - start_time
        )


        cumulative_time = (
            previous_time
            + current_elapsed_time
        )


        study.set_user_attr(
            "cumulative_tuning_seconds",
            cumulative_time
        )


    else:

        current_elapsed_time = 0.0

        cumulative_time = previous_time


        print(
            "Seluruh trial model ini "
            "sudah selesai. Tuning dilewati."
        )


    # --------------------------------------------------------
    # Memastikan study memiliki trial yang selesai
    # --------------------------------------------------------

    completed_trials_after_tuning = [
        trial
        for trial in study.trials
        if (
            trial.state == TrialState.COMPLETE
            and trial.value is not None
        )
    ]


    if not completed_trials_after_tuning:
        raise RuntimeError(
            f"Tidak terdapat trial yang berhasil untuk {model_name}."
        )


    # --------------------------------------------------------
    # Mengambil trial terbaik berdasarkan Log Loss terendah
    # --------------------------------------------------------

    best_trial = study.best_trial

    best_mean_logloss = float(
        best_trial.value
    )

    best_mean_auc = float(
        best_trial.user_attrs.get(
            "mean_cv_roc_auc",
            np.nan
        )
    )


    # --------------------------------------------------------
    # Menyimpan objek utama
    # --------------------------------------------------------

    initial_tuning_studies[
        model_name
    ] = study


    initial_tuning_times[
        model_name
    ] = cumulative_time


    initial_best_params[
        model_name
    ] = study.best_params.copy()


    initial_best_results[
        model_name
    ] = {
        "Best Mean CV Log Loss":
            best_mean_logloss,

        "Mean CV ROC-AUC":
            best_mean_auc,

        "Waktu Tuning Kumulatif (s)":
            cumulative_time,

        "Hyperparameter Terbaik":
            study.best_params.copy()
    }


    # --------------------------------------------------------
    # Membentuk tabel seluruh trial
    #
    # Tabel sudah diurutkan oleh study_trials_table()
    # berdasarkan:
    # 1. Mean CV Log Loss terendah;
    # 2. Mean CV ROC-AUC tertinggi;
    # 3. nomor trial terendah.
    # --------------------------------------------------------

    full_trials = study_trials_table(
        study=study,
        model_name=model_name,
        context_columns={
            "Skenario Pelabelan":
                LABEL_DISPLAY_NAME,

            "Model":
                model_name
        }
    )


    if full_trials.empty:
        raise RuntimeError(
            f"Tabel trial {model_name} kosong."
        )


    initial_trials_full_parts.append(
        full_trials
    )


    # --------------------------------------------------------
    # Menyimpan seluruh trial per model
    # --------------------------------------------------------

    full_trials_path = os.path.join(
        INITIAL_TUNING_DIR,
        (
            f"seluruh_trial_"
            f"{safe_file_name(model_name)}.csv"
        )
    )


    full_trials.to_csv(
        full_trials_path,
        index=False
    )


    full_trials.to_pickle(
        os.path.join(
            INITIAL_TUNING_DIR,
            (
                f"seluruh_trial_"
                f"{safe_file_name(model_name)}.pkl"
            )
        )
    )


    # --------------------------------------------------------
    # Mengambil Top 5 trial terbaik
    #
    # Karena full_trials telah diurutkan berdasarkan
    # Log Loss terendah, lima baris pertama merupakan
    # lima trial terbaik.
    # --------------------------------------------------------

    top5_trials = (
        full_trials
        .head(5)
        .copy()
    )


    top5_trials.insert(
        0,
        "Rank",
        range(
            1,
            len(top5_trials) + 1
        )
    )


    initial_top5_parts.append(
        top5_trials
    )


    # --------------------------------------------------------
    # Menyimpan Top 5 trial per model
    # --------------------------------------------------------

    top5_path = os.path.join(
        INITIAL_TUNING_DIR,
        (
            f"top5_trial_"
            f"{safe_file_name(model_name)}.csv"
        )
    )


    top5_trials.to_csv(
        top5_path,
        index=False
    )


    top5_trials.to_pickle(
        os.path.join(
            INITIAL_TUNING_DIR,
            (
                f"top5_trial_"
                f"{safe_file_name(model_name)}.pkl"
            )
        )
    )


    # --------------------------------------------------------
    # Membentuk ringkasan trial terbaik
    # --------------------------------------------------------

    summary_row = {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            model_name,

        "Jumlah Fitur":
            len(features_before_fs),

        "Best Mean CV Log Loss":
            best_mean_logloss,

        "Mean CV ROC-AUC":
            best_mean_auc,

        "Jumlah Trial Selesai":
            len(
                completed_trials_after_tuning
            ),

        "Waktu Tuning Kumulatif (s)":
            cumulative_time,

        "Hyperparameter Terbaik":
            params_to_text(
                study.best_params
            )
    }


    initial_tuning_summary_rows.append(
        summary_row
    )


    # --------------------------------------------------------
    # Menampilkan Top 5 trial model saat ini
    # --------------------------------------------------------

    print(
        f"\nTOP 5 TRIAL TERBAIK "
        f"— {model_name}"
    )


    display_without_index(
        top5_trials,
        formats={
            "Mean CV Log Loss":
                "{:.6f}",

            "Mean CV ROC-AUC":
                "{:.6f}"
        }
    )


    # --------------------------------------------------------
    # Menampilkan hasil terbaik model saat ini
    # --------------------------------------------------------

    model_summary = pd.DataFrame([
        summary_row
    ])


    print(
        f"\nRINGKASAN TUNING AWAL "
        f"— {model_name}"
    )


    display_without_index(
        model_summary,
        formats={
            "Best Mean CV Log Loss":
                "{:.6f}",

            "Mean CV ROC-AUC":
                "{:.6f}",

            "Waktu Tuning Kumulatif (s)":
                "{:.2f}"
        }
    )


    # --------------------------------------------------------
    # Checkpoint ringkasan setelah setiap model selesai
    # --------------------------------------------------------

    checkpoint_summary = pd.DataFrame(
        initial_tuning_summary_rows
    )


    checkpoint_summary.to_csv(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_tuning_summary_checkpoint.csv"
        ),
        index=False
    )


    checkpoint_summary.to_pickle(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_tuning_summary_checkpoint.pkl"
        )
    )


    # --------------------------------------------------------
    # Menyimpan parameter terbaik seluruh model
    # yang telah selesai
    # --------------------------------------------------------

    with open(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_best_params_checkpoint.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            initial_best_params,
            file,
            ensure_ascii=False,
            indent=4
        )


    # --------------------------------------------------------
    # Menyimpan hasil metrik terbaik seluruh model
    # --------------------------------------------------------

    with open(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_best_results_checkpoint.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            initial_best_results,
            file,
            ensure_ascii=False,
            indent=4
        )


    print(
        "\nCheckpoint tersimpan:"
    )

    print(
        "-",
        full_trials_path
    )

    print(
        "-",
        top5_path
    )

    print(
        "- initial_tuning_summary_checkpoint.csv"
    )

    print(
        "- initial_best_params_checkpoint.json"
    )

    print(
        "- initial_best_results_checkpoint.json"
    )


    gc.collect()


# ============================================================
# MENGGABUNGKAN HASIL SELURUH MODEL
# ============================================================

initial_tuning_trials_full = pd.concat(
    initial_trials_full_parts,
    ignore_index=True
)


initial_tuning_top5 = pd.concat(
    initial_top5_parts,
    ignore_index=True
)


initial_tuning_summary = pd.DataFrame(
    initial_tuning_summary_rows
)


# ------------------------------------------------------------
# Mengurutkan ringkasan sesuai urutan model
# ------------------------------------------------------------

initial_tuning_summary[
    "Model"
] = pd.Categorical(
    initial_tuning_summary["Model"],
    categories=MODEL_ORDER,
    ordered=True
)


initial_tuning_summary = (
    initial_tuning_summary
    .sort_values("Model")
    .reset_index(drop=True)
)


# Mengembalikan tipe kolom Model menjadi string
initial_tuning_summary[
    "Model"
] = initial_tuning_summary[
    "Model"
].astype(str)


# ------------------------------------------------------------
# Menyimpan hasil gabungan seluruh model
# ------------------------------------------------------------

initial_tuning_trials_full.to_csv(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_seluruh_trial.csv"
    ),
    index=False
)


initial_tuning_trials_full.to_pickle(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_seluruh_trial.pkl"
    )
)


initial_tuning_top5.to_csv(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_top5_semua_model.csv"
    ),
    index=False
)


initial_tuning_top5.to_pickle(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_top5_semua_model.pkl"
    )
)


initial_tuning_summary.to_csv(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_summary.csv"
    ),
    index=False
)


initial_tuning_summary.to_pickle(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_tuning_summary.pkl"
    )
)


# ------------------------------------------------------------
# Menyimpan parameter terbaik
# ------------------------------------------------------------

with open(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_best_params.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        initial_best_params,
        file,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# Menyimpan hasil metrik terbaik
# ------------------------------------------------------------

with open(
    os.path.join(
        INITIAL_TUNING_DIR,
        "initial_best_results.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        initial_best_results,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT AKHIR BLOK 3
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "TOP 5 TRIAL TERBAIK "
    "SELURUH MODEL"
)

print(
    "=" * 60
)


display_without_index(
    initial_tuning_top5,
    formats={
        "Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 60
)

print(
    "RINGKASAN HASIL TUNING AWAL "
    "SELURUH MODEL"
)

print(
    "=" * 60
)


display_without_index(
    initial_tuning_summary,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\nKriteria pemilihan trial terbaik:"
)

print(
    "- Mean CV Log Loss terendah"
)

print(
    "- Mean CV ROC-AUC hanya disimpan "
    "sebagai metrik pendamping"
)


print(
    "\nSeluruh hasil tuning awal tersimpan di:"
)

print(
    INITIAL_TUNING_DIR
)

## TUNING AWAL

In [ ]:
# ============================================================
# BLOK 4 — LATIH MODEL HASIL TUNING AWAL
# DAN HITUNG GLOBAL SHAP FEATURE IMPORTANCE
# OBJECTIVE TUNING: MEAN CV LOG LOSS TERENDAH
# DENGAN CHECKPOINT DAN RESUME
# ============================================================

import os
import json
import joblib

# ------------------------------------------------------------
# Folder penyimpanan Blok 4
#
# Folder dibuat berbeda dari eksperimen lama berbasis ROC-AUC
# agar model dan nilai SHAP lama tidak termuat kembali.
# ------------------------------------------------------------

BLOCK4_DIR = os.path.join(
    "./artifacts",
    (
        f"shap_ranking_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

MODEL_SAVE_DIR = os.path.join(
    BLOCK4_DIR,
    "models"
)

SHAP_SAVE_DIR = os.path.join(
    BLOCK4_DIR,
    "shap_values"
)

TABLE_SAVE_DIR = os.path.join(
    BLOCK4_DIR,
    "tables"
)

FIGURE_SAVE_DIR = os.path.join(
    BLOCK4_DIR,
    "figures"
)

for directory in [
    BLOCK4_DIR,
    MODEL_SAVE_DIR,
    SHAP_SAVE_DIR,
    TABLE_SAVE_DIR,
    FIGURE_SAVE_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 4:",
    BLOCK4_DIR
)

print(
    "Objective tuning awal:",
    "Mean CV Log Loss terendah"
)


# ============================================================
# PEMERIKSAAN OBJEK YANG DIBUTUHKAN
# ============================================================

required_block4_objects = [
    "X_train_raw",
    "X_train_rf",
    "X_train_xgb",
    "X_train_cb",
    "y_train",
    "features_before_fs",
    "num_cols",
    "cat_cols",
    "INITIAL_TUNING_DIR",
    "LABEL_DISPLAY_NAME",
    "SHAP_SAMPLE_SIZE"
]

missing_block4_objects = [
    name
    for name in required_block4_objects
    if name not in globals()
]

if missing_block4_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 4: "
        f"{missing_block4_objects}"
    )


# ------------------------------------------------------------
# Validasi panjang data train
# ------------------------------------------------------------

train_lengths = {
    "X_train_raw": len(X_train_raw),
    "X_train_rf": len(X_train_rf),
    "X_train_xgb": len(X_train_xgb),
    "X_train_cb": len(X_train_cb),
    "y_train": len(y_train)
}

if len(set(train_lengths.values())) != 1:
    raise ValueError(
        "Jumlah observasi data train tidak sama: "
        f"{train_lengths}"
    )


# ------------------------------------------------------------
# Validasi daftar fitur
# ------------------------------------------------------------

features_before_fs = list(
    features_before_fs
)

if not features_before_fs:
    raise ValueError(
        "features_before_fs tidak boleh kosong."
    )

missing_features_by_dataset = {}

for dataset_name, dataset in {
    "X_train_rf": X_train_rf,
    "X_train_xgb": X_train_xgb,
    "X_train_cb": X_train_cb
}.items():

    missing_features = [
        feature
        for feature in features_before_fs
        if feature not in dataset.columns
    ]

    if missing_features:
        missing_features_by_dataset[
            dataset_name
        ] = missing_features

if missing_features_by_dataset:
    raise ValueError(
        "Fitur berikut tidak ditemukan pada data train: "
        f"{missing_features_by_dataset}"
    )


# ============================================================
# MEMUAT HYPERPARAMETER HASIL TUNING AWAL
# ============================================================

if (
    "initial_best_params" not in globals()
    or not initial_best_params
):
    initial_params_path = os.path.join(
        INITIAL_TUNING_DIR,
        "initial_best_params.json"
    )

    if not os.path.exists(
        initial_params_path
    ):
        raise FileNotFoundError(
            "Hyperparameter tuning awal berbasis Log Loss "
            "belum tersedia. Jalankan Blok 3 terlebih dahulu."
        )

    with open(
        initial_params_path,
        "r",
        encoding="utf-8"
    ) as file:
        initial_best_params = json.load(
            file
        )

    print(
        "Hyperparameter tuning awal "
        "berhasil dimuat dari file:"
    )

    print(
        initial_params_path
    )


# ------------------------------------------------------------
# Memastikan seluruh model memiliki parameter terbaik
# ------------------------------------------------------------

missing_model_params = [
    model_name
    for model_name in MODEL_ORDER
    if model_name not in initial_best_params
]

if missing_model_params:
    raise KeyError(
        "Hyperparameter terbaik belum tersedia untuk model: "
        f"{missing_model_params}"
    )


# ------------------------------------------------------------
# Memuat hasil metrik tuning awal jika tersedia
# ------------------------------------------------------------

initial_results_path = os.path.join(
    INITIAL_TUNING_DIR,
    "initial_best_results.json"
)

if os.path.exists(
    initial_results_path
):
    with open(
        initial_results_path,
        "r",
        encoding="utf-8"
    ) as file:
        initial_best_results_loaded = json.load(
            file
        )

    print(
        "Hasil metrik tuning awal berhasil dimuat."
    )

else:
    initial_best_results_loaded = {}

    print(
        "File initial_best_results.json tidak ditemukan. "
        "Perhitungan SHAP tetap dapat dilanjutkan."
    )


# ============================================================
# DATASET TRAIN HASIL PREPROCESSING PENUH
# ============================================================

full_prep_train = {
    "Random Forest": (
        X_train_rf[
            features_before_fs
        ]
        .copy()
        .reset_index(drop=True)
    ),

    "XGBoost": (
        X_train_xgb[
            features_before_fs
        ]
        .copy()
        .reset_index(drop=True)
    ),

    "CatBoost": (
        X_train_cb[
            features_before_fs
        ]
        .copy()
        .reset_index(drop=True)
    )
}


y_train_full = (
    pd.Series(y_train)
    .reset_index(drop=True)
    .astype(int)
)


# ============================================================
# MENENTUKAN OBSERVASI UNTUK SHAP
#
# shap_positions merupakan posisi baris, bukan label indeks.
# Hal ini mencegah ketidaksesuaian apabila indeks DataFrame
# berbeda antara data mentah dan data hasil preprocessing.
# ============================================================

n_train = len(
    X_train_raw
)

if (
    SHAP_SAMPLE_SIZE is None
    or SHAP_SAMPLE_SIZE >= n_train
):
    shap_positions = np.arange(
        n_train,
        dtype=int
    )

else:
    random_generator = np.random.default_rng(
        RANDOM_STATE
    )

    shap_positions = np.sort(
        random_generator.choice(
            n_train,
            size=int(SHAP_SAMPLE_SIZE),
            replace=False
        )
    )


shap_positions_path = os.path.join(
    BLOCK4_DIR,
    "shap_positions.npy"
)

np.save(
    shap_positions_path,
    shap_positions
)


X_shap_by_model = {
    model_name: (
        full_prep_train[
            model_name
        ]
        .iloc[
            shap_positions
        ]
        .loc[
            :,
            features_before_fs
        ]
        .copy()
        .reset_index(drop=True)
    )
    for model_name in MODEL_ORDER
}


print(
    "Jumlah observasi data train:",
    n_train
)

print(
    "Jumlah observasi untuk SHAP:",
    len(shap_positions)
)


# ============================================================
# FUNGSI PERHITUNGAN SHAP
# ============================================================

def calculate_rf_shap(
    model,
    X_data
):
    """
    Menghitung nilai SHAP Random Forest untuk kelas positif.
    """

    explainer = shap.TreeExplainer(
        model
    )

    try:
        shap_output = explainer(
            X_data,
            check_additivity=False
        )

        shap_values = shap_output.values

    except Exception:
        shap_values = explainer.shap_values(
            X_data
        )


    # Beberapa versi SHAP mengembalikan list per kelas
    if isinstance(
        shap_values,
        list
    ):
        if len(shap_values) == 2:
            shap_values = shap_values[1]

        else:
            shap_values = shap_values[0]


    shap_values = np.asarray(
        shap_values,
        dtype=float
    )


    # Menangani keluaran tiga dimensi
    if shap_values.ndim == 3:

        # Format:
        # observasi × fitur × kelas
        if shap_values.shape[-1] == 2:
            shap_values = shap_values[
                :,
                :,
                1
            ]

        # Format:
        # kelas × observasi × fitur
        elif shap_values.shape[0] == 2:
            shap_values = shap_values[1]


    return np.asarray(
        shap_values,
        dtype=float
    )


def calculate_xgb_shap(
    model,
    X_data
):
    """
    Menghitung nilai kontribusi SHAP XGBoost.
    """

    dmatrix = xgb.DMatrix(
        X_data,
        enable_categorical=True
    )

    shap_values = (
        model
        .get_booster()
        .predict(
            dmatrix,
            pred_contribs=True
        )
    )

    # Kolom terakhir merupakan expected value
    return np.asarray(
        shap_values[:, :-1],
        dtype=float
    )


def calculate_catboost_shap(
    model,
    X_data
):
    """
    Menghitung nilai SHAP CatBoost.
    """

    selected_cat = [
        col
        for col in X_data.columns
        if col in cat_cols
    ]

    data_pool = Pool(
        X_data,
        cat_features=selected_cat
    )

    shap_values = (
        model
        .get_feature_importance(
            data_pool,
            type="ShapValues"
        )
    )

    # Kolom terakhir merupakan expected value
    return np.asarray(
        shap_values[:, :-1],
        dtype=float
    )


# ============================================================
# MELATIH ATAU MEMUAT MODEL PEMBENTUK SHAP
# ============================================================

initial_models = {}


for model_name in MODEL_ORDER:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "MODEL PEMBENTUK SHAP "
        f"— {model_name}"
    )

    print(
        "=" * 60
    )


    model_file_name = (
        f"model_shap_logloss_"
        f"{safe_file_name(model_name)}.joblib"
    )

    model_path = os.path.join(
        MODEL_SAVE_DIR,
        model_file_name
    )


    # --------------------------------------------------------
    # Memuat model apabila checkpoint baru telah tersedia
    # --------------------------------------------------------

    if os.path.exists(
        model_path
    ):
        print(
            "Memuat model pembentuk SHAP:",
            model_name
        )

        model = joblib.load(
            model_path
        )


    # --------------------------------------------------------
    # Melatih model pada seluruh data train
    # --------------------------------------------------------

    else:
        print(
            "Melatih model pembentuk SHAP:",
            model_name
        )


        best_params = initial_best_params[
            model_name
        ].copy()


        model = build_model(
            model_name=model_name,
            params=best_params,
            use_early_stopping=False
        )


        X_model_train = (
            full_prep_train[
                model_name
            ]
            .loc[
                :,
                features_before_fs
            ]
            .copy()
        )


        if model_name == "Random Forest":

            model.fit(
                X_model_train,
                y_train_full
            )


        elif model_name == "XGBoost":

            model.fit(
                X_model_train,
                y_train_full,
                verbose=False
            )


        elif model_name == "CatBoost":

            initial_cat_features = [
                col
                for col in features_before_fs
                if col in cat_cols
            ]

            model.fit(
                X_model_train,
                y_train_full,
                cat_features=initial_cat_features,
                verbose=False
            )


        else:
            raise ValueError(
                "Model tidak dikenali: "
                f"{model_name}"
            )


        joblib.dump(
            model,
            model_path
        )


        print(
            "Model tersimpan:",
            model_path
        )


    initial_models[
        model_name
    ] = model


print(
    "\nSeluruh model pembentuk SHAP tersedia."
)


# ============================================================
# MENGHITUNG ATAU MEMUAT GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

importance_tables = {}

importance_top5_tables = {}

shap_values_by_model = {}


for model_name in MODEL_ORDER:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "GLOBAL SHAP FEATURE IMPORTANCE "
        f"— {model_name}"
    )

    print(
        "=" * 60
    )


    safe_model = safe_file_name(
        model_name
    )


    shap_values_path = os.path.join(
        SHAP_SAVE_DIR,
        f"shap_values_logloss_{safe_model}.npy"
    )

    importance_csv_path = os.path.join(
        TABLE_SAVE_DIR,
        f"importance_full_logloss_{safe_model}.csv"
    )

    importance_pickle_path = os.path.join(
        TABLE_SAVE_DIR,
        f"importance_full_logloss_{safe_model}.pkl"
    )

    top5_csv_path = os.path.join(
        TABLE_SAVE_DIR,
        f"importance_top5_logloss_{safe_model}.csv"
    )

    top5_pickle_path = os.path.join(
        TABLE_SAVE_DIR,
        f"importance_top5_logloss_{safe_model}.pkl"
    )

    figure_path = os.path.join(
        FIGURE_SAVE_DIR,
        f"global_shap_logloss_{safe_model}.png"
    )


    model = initial_models[
        model_name
    ]

    X_shap = X_shap_by_model[
        model_name
    ]


    # --------------------------------------------------------
    # Memuat hasil SHAP jika checkpoint tersedia
    # --------------------------------------------------------

    if (
        os.path.exists(
            shap_values_path
        )
        and os.path.exists(
            importance_pickle_path
        )
    ):
        print(
            "Memuat nilai SHAP dan tabel importance "
            "dari checkpoint."
        )

        shap_values = np.load(
            shap_values_path
        )

        importance_table = pd.read_pickle(
            importance_pickle_path
        )


    # --------------------------------------------------------
    # Menghitung SHAP jika checkpoint belum tersedia
    # --------------------------------------------------------

    else:
        print(
            "Menghitung SHAP:",
            model_name
        )


        if model_name == "Random Forest":

            shap_values = calculate_rf_shap(
                model=model,
                X_data=X_shap
            )


        elif model_name == "XGBoost":

            shap_values = calculate_xgb_shap(
                model=model,
                X_data=X_shap
            )


        elif model_name == "CatBoost":

            shap_values = calculate_catboost_shap(
                model=model,
                X_data=X_shap
            )


        else:
            raise ValueError(
                "Model tidak dikenali: "
                f"{model_name}"
            )


        if shap_values.shape != X_shap.shape:
            raise ValueError(
                f"Ukuran SHAP {model_name} tidak sesuai. "
                f"SHAP={shap_values.shape}, "
                f"data={X_shap.shape}"
            )


        if not np.isfinite(
            shap_values
        ).all():
            raise ValueError(
                f"Nilai SHAP {model_name} "
                "mengandung NaN atau infinity."
            )


        # Global feature importance:
        # rata-rata nilai absolut SHAP setiap fitur
        mean_abs_shap = (
            np.abs(
                shap_values
            )
            .mean(axis=0)
        )


        total_importance = float(
            mean_abs_shap.sum()
        )


        if np.isclose(
            total_importance,
            0.0
        ):
            normalized_shap = np.zeros_like(
                mean_abs_shap,
                dtype=float
            )

        else:
            normalized_shap = (
                mean_abs_shap
                / total_importance
            )


        importance_table = pd.DataFrame({
            "Feature":
                list(X_shap.columns),

            "Mean Abs SHAP":
                mean_abs_shap.astype(float),

            "Normalized SHAP":
                normalized_shap.astype(float)
        })


        importance_table = (
            importance_table
            .sort_values(
                by=[
                    "Mean Abs SHAP",
                    "Feature"
                ],
                ascending=[
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
        )


        importance_table.insert(
            0,
            "Rank",
            range(
                1,
                len(importance_table) + 1
            )
        )


        # ----------------------------------------------------
        # Menyimpan nilai SHAP dan tabel importance
        # ----------------------------------------------------

        np.save(
            shap_values_path,
            shap_values
        )


        importance_table.to_csv(
            importance_csv_path,
            index=False
        )


        importance_table.to_pickle(
            importance_pickle_path
        )


        print(
            "Nilai SHAP tersimpan:",
            shap_values_path
        )

        print(
            "Tabel importance tersimpan:",
            importance_csv_path
        )


    # --------------------------------------------------------
    # Validasi hasil yang dimuat atau dihitung
    # --------------------------------------------------------

    if shap_values.shape != X_shap.shape:
        raise ValueError(
            "Ukuran SHAP tersimpan untuk "
            f"{model_name} tidak sesuai. "
            f"SHAP={shap_values.shape}, "
            f"data={X_shap.shape}"
        )


    required_importance_cols = [
        "Rank",
        "Feature",
        "Mean Abs SHAP",
        "Normalized SHAP"
    ]


    if list(
        importance_table.columns
    ) != required_importance_cols:
        raise ValueError(
            "Kolom tabel importance "
            f"{model_name} tidak sesuai."
        )


    if set(
        importance_table["Feature"]
    ) != set(
        features_before_fs
    ):
        raise ValueError(
            "Daftar fitur tabel importance "
            f"{model_name} tidak sesuai."
        )


    if len(
        importance_table
    ) != len(
        features_before_fs
    ):
        raise ValueError(
            "Jumlah fitur tabel importance "
            f"{model_name} tidak sesuai."
        )


    # --------------------------------------------------------
    # Menyimpan objek pada notebook
    # --------------------------------------------------------

    importance_tables[
        model_name
    ] = importance_table


    shap_values_by_model[
        model_name
    ] = shap_values


    importance_top5 = (
        importance_table
        .head(5)
        .copy()
    )


    importance_top5_tables[
        model_name
    ] = importance_top5


    importance_top5.to_csv(
        top5_csv_path,
        index=False
    )


    importance_top5.to_pickle(
        top5_pickle_path
    )


    # --------------------------------------------------------
    # Visualisasi global SHAP feature importance
    # --------------------------------------------------------

    plot_data = (
        importance_table
        .sort_values(
            "Mean Abs SHAP",
            ascending=True
        )
    )


    plt.figure(
        figsize=(8, 5)
    )


    plt.barh(
        plot_data["Feature"],
        plot_data["Mean Abs SHAP"]
    )


    plt.title(
        "Global SHAP Feature Importance "
        f"— {model_name}\n"
        f"{LABEL_DISPLAY_NAME}"
    )


    plt.xlabel(
        "Mean Absolute SHAP Value"
    )


    plt.ylabel(
        "Feature"
    )


    plt.tight_layout()


    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


    plt.close()


    print(
        "Gambar tersimpan:",
        figure_path
    )


    # --------------------------------------------------------
    # Menampilkan tabel lengkap ranking SHAP
    # --------------------------------------------------------

    print(
        "\nTABEL LENGKAP RANKING SHAP "
        f"— {model_name}"
    )


    display_without_index(
        importance_table,
        formats={
            "Mean Abs SHAP":
                "{:.6f}",

            "Normalized SHAP":
                "{:.6f}"
        }
    )


    # --------------------------------------------------------
    # Menampilkan tabel ringkasan Top 5
    # --------------------------------------------------------

    print(
        "\nTABEL RINGKAS TOP 5 "
        f"RANKING SHAP — {model_name}"
    )


    display_without_index(
        importance_top5,
        formats={
            "Mean Abs SHAP":
                "{:.6f}",

            "Normalized SHAP":
                "{:.6f}"
        }
    )


    gc.collect()


# ============================================================
# MENGGABUNGKAN TABEL RANKING SELURUH MODEL
# ============================================================

importance_full_parts = []

importance_top5_parts = []


for model_name in MODEL_ORDER:

    importance_full_model = (
        importance_tables[
            model_name
        ]
        .copy()
    )

    importance_full_model.insert(
        0,
        "Model",
        model_name
    )

    importance_full_parts.append(
        importance_full_model
    )


    importance_top5_model = (
        importance_top5_tables[
            model_name
        ]
        .copy()
    )

    importance_top5_model.insert(
        0,
        "Model",
        model_name
    )

    importance_top5_parts.append(
        importance_top5_model
    )


importance_all_models = pd.concat(
    importance_full_parts,
    ignore_index=True
)


importance_top5_all_models = pd.concat(
    importance_top5_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Menyimpan tabel gabungan
# ------------------------------------------------------------

importance_all_models.to_csv(
    os.path.join(
        TABLE_SAVE_DIR,
        "importance_all_models_logloss.csv"
    ),
    index=False
)


importance_all_models.to_pickle(
    os.path.join(
        TABLE_SAVE_DIR,
        "importance_all_models_logloss.pkl"
    )
)


importance_top5_all_models.to_csv(
    os.path.join(
        TABLE_SAVE_DIR,
        "importance_top5_all_models_logloss.csv"
    ),
    index=False
)


importance_top5_all_models.to_pickle(
    os.path.join(
        TABLE_SAVE_DIR,
        "importance_top5_all_models_logloss.pkl"
    )
)


# ============================================================
# MENYIMPAN HYPERPARAMETER MODEL PEMBENTUK SHAP
# ============================================================

with open(
    os.path.join(
        BLOCK4_DIR,
        "shap_model_best_params_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        initial_best_params,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MENYIMPAN METADATA BLOK 4
# ============================================================

block4_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Objective Optimasi":
        "Mean CV Log Loss terendah",

    "Metrik Pendamping":
        "Mean CV ROC-AUC",

    "Jumlah Observasi Train":
        int(n_train),

    "Jumlah Observasi SHAP":
        int(
            len(shap_positions)
        ),

    "Jumlah Fitur":
        int(
            len(features_before_fs)
        ),

    "SHAP Sample Size":
        (
            None
            if SHAP_SAMPLE_SIZE is None
            else int(
                SHAP_SAMPLE_SIZE
            )
        ),

    "Random State":
        int(RANDOM_STATE),

    "Model":
        list(MODEL_ORDER),

    "Fitur":
        list(features_before_fs),

    "Hyperparameter Terbaik":
        initial_best_params
}


with open(
    os.path.join(
        BLOCK4_DIR,
        "block4_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block4_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT AKHIR BLOK 4
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "RINGKASAN TOP 5 GLOBAL SHAP "
    "SELURUH MODEL"
)

print(
    "=" * 60
)


display_without_index(
    importance_top5_all_models,
    formats={
        "Mean Abs SHAP":
            "{:.6f}",

        "Normalized SHAP":
            "{:.6f}"
    }
)


print(
    "\nDasar hyperparameter model pembentuk SHAP:"
)

print(
    "- Mean CV Log Loss terendah"
)

print(
    "- Mean CV ROC-AUC disimpan sebagai metrik pendamping"
)


print(
    "\nSeluruh model, nilai SHAP, tabel, "
    "gambar, dan metadata berhasil disimpan di:"
)

print(
    BLOCK4_DIR
)

## VISUALISASI SHAP

In [ ]:
# ============================================================
# BLOK 5 — AGREGASI RANKING FITUR
# BERDASARKAN GLOBAL SHAP FEATURE IMPORTANCE
# DARI MODEL HASIL OPTIMASI LOG LOSS
# DENGAN PENYIMPANAN DAN RESUME
# ============================================================

import os
import json
import joblib


# ------------------------------------------------------------
# Folder penyimpanan Blok 5
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC
# agar checkpoint lama tidak termuat kembali.
# ------------------------------------------------------------

BLOCK5_DIR = os.path.join(
    "./artifacts",
    (
        f"aggregation_ranking_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

os.makedirs(
    BLOCK5_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Path checkpoint Blok 5
# ------------------------------------------------------------

RANK_TABLE_PATH = os.path.join(
    BLOCK5_DIR,
    "rank_table_logloss.pkl"
)

AGGREGATION_DISPLAY_PATH = os.path.join(
    BLOCK5_DIR,
    "aggregation_display_logloss.pkl"
)

AGGREGATION_RANKINGS_PATH = os.path.join(
    BLOCK5_DIR,
    "aggregation_rankings_logloss.joblib"
)

AGGREGATION_SUMMARY_PATH = os.path.join(
    BLOCK5_DIR,
    "aggregation_ranking_summary_logloss.pkl"
)

AGGREGATION_ORDER_PATH = os.path.join(
    BLOCK5_DIR,
    "aggregation_order_table_logloss.pkl"
)


print(
    "Folder hasil Blok 5:",
    BLOCK5_DIR
)

print(
    "Sumber ranking:",
    "Global SHAP dari model hasil optimasi Mean CV Log Loss"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block5_objects = [
    "features_before_fs",
    "MODEL_ORDER",
    "AGGREGATION_METHODS",
    "LABEL_DISPLAY_NAME"
]

missing_block5_objects = [
    name
    for name in required_block5_objects
    if name not in globals()
]

if missing_block5_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 5: "
        f"{missing_block5_objects}"
    )


features_before_fs = list(
    features_before_fs
)

if not features_before_fs:
    raise ValueError(
        "features_before_fs tidak boleh kosong."
    )


# ============================================================
# MEMUAT IMPORTANCE TABLES DARI BLOK 4
# JIKA VARIABEL TIDAK TERSEDIA
# ============================================================

if (
    "importance_tables" not in globals()
    or not importance_tables
):
    print(
        "Variabel importance_tables tidak tersedia. "
        "Memuat hasil Blok 4 dari file."
    )


    # --------------------------------------------------------
    # Menentukan folder Blok 4 berbasis Log Loss
    # --------------------------------------------------------

    if "BLOCK4_DIR" not in globals():
        BLOCK4_DIR = os.path.join(
            "./artifacts",
            (
                f"shap_ranking_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    TABLE_SAVE_DIR = os.path.join(
        BLOCK4_DIR,
        "tables"
    )


    if not os.path.exists(
        TABLE_SAVE_DIR
    ):
        raise FileNotFoundError(
            "Folder tabel Blok 4 tidak ditemukan: "
            f"{TABLE_SAVE_DIR}"
        )


    importance_tables = {}


    for model_name in MODEL_ORDER:

        safe_model = safe_file_name(
            model_name
        )


        importance_path = os.path.join(
            TABLE_SAVE_DIR,
            (
                f"importance_full_logloss_"
                f"{safe_model}.pkl"
            )
        )


        if not os.path.exists(
            importance_path
        ):
            raise FileNotFoundError(
                f"Tabel importance {model_name} "
                "tidak ditemukan pada path:\n"
                f"{importance_path}\n"
                "Jalankan Blok 4 berbasis Log Loss terlebih dahulu."
            )


        importance_tables[
            model_name
        ] = pd.read_pickle(
            importance_path
        )


    print(
        "Seluruh importance table "
        "berhasil dimuat."
    )


# ============================================================
# VALIDASI IMPORTANCE TABLES
# ============================================================

missing_models = [
    model_name
    for model_name in MODEL_ORDER
    if model_name not in importance_tables
]

if missing_models:
    raise KeyError(
        "Importance table belum tersedia untuk model: "
        f"{missing_models}"
    )


required_importance_columns = [
    "Rank",
    "Feature",
    "Mean Abs SHAP",
    "Normalized SHAP"
]


for model_name in MODEL_ORDER:

    model_importance = importance_tables[
        model_name
    ]


    missing_columns = [
        col
        for col in required_importance_columns
        if col not in model_importance.columns
    ]


    if missing_columns:
        raise ValueError(
            f"Kolom importance {model_name} "
            f"belum sesuai: {missing_columns}"
        )


    if len(
        model_importance
    ) != len(
        features_before_fs
    ):
        raise ValueError(
            f"Jumlah fitur importance {model_name} "
            "tidak sesuai."
        )


    if set(
        model_importance["Feature"]
    ) != set(
        features_before_fs
    ):
        raise ValueError(
            f"Daftar fitur importance {model_name} "
            "tidak sesuai dengan features_before_fs."
        )


    if model_importance[
        "Feature"
    ].duplicated().any():
        raise ValueError(
            f"Terdapat fitur duplikat pada "
            f"importance table {model_name}."
        )


    if model_importance[
        "Rank"
    ].duplicated().any():
        raise ValueError(
            f"Terdapat rank duplikat pada "
            f"importance table {model_name}."
        )


    if not np.isfinite(
        model_importance[
            [
                "Mean Abs SHAP",
                "Normalized SHAP"
            ]
        ].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"Nilai importance {model_name} "
            "mengandung NaN atau infinity."
        )


# ============================================================
# MEMUAT HASIL BLOK 5 JIKA SUDAH TERSEDIA
# ============================================================

block5_complete = all([
    os.path.exists(
        RANK_TABLE_PATH
    ),
    os.path.exists(
        AGGREGATION_DISPLAY_PATH
    ),
    os.path.exists(
        AGGREGATION_RANKINGS_PATH
    ),
    os.path.exists(
        AGGREGATION_SUMMARY_PATH
    ),
    os.path.exists(
        AGGREGATION_ORDER_PATH
    )
])


if block5_complete:

    print(
        "\nHasil Blok 5 sudah tersedia. "
        "Memuat hasil agregasi dari checkpoint."
    )


    rank_table = pd.read_pickle(
        RANK_TABLE_PATH
    )


    aggregation_display = pd.read_pickle(
        AGGREGATION_DISPLAY_PATH
    )


    aggregation_rankings = joblib.load(
        AGGREGATION_RANKINGS_PATH
    )


    aggregation_ranking_summary = pd.read_pickle(
        AGGREGATION_SUMMARY_PATH
    )


    aggregation_order_table = pd.read_pickle(
        AGGREGATION_ORDER_PATH
    )


# ============================================================
# MENGHITUNG AGREGASI RANKING
# ============================================================

else:

    print(
        "\nMenghitung agregasi ranking fitur."
    )


    # ========================================================
    # MEMBENTUK TABEL DASAR RANKING
    # ========================================================

    rank_table = pd.DataFrame({
        "Feature":
            list(features_before_fs)
    })


    # --------------------------------------------------------
    # Menggabungkan ranking dan normalized SHAP
    # Random Forest, XGBoost, dan CatBoost
    # --------------------------------------------------------

    for model_name in MODEL_ORDER:

        model_importance = (
            importance_tables[
                model_name
            ]
            .rename(
                columns={
                    "Normalized SHAP":
                        (
                            f"Normalized SHAP "
                            f"{model_name}"
                        ),

                    "Rank":
                        f"Rank {model_name}"
                }
            )
        )


        required_columns = [
            "Feature",
            f"Normalized SHAP {model_name}",
            f"Rank {model_name}"
        ]


        rank_table = rank_table.merge(
            model_importance[
                required_columns
            ],
            on="Feature",
            how="inner",
            validate="one_to_one"
        )


    if len(
        rank_table
    ) != len(
        features_before_fs
    ):
        raise ValueError(
            "Jumlah fitur setelah penggabungan "
            "ranking tidak sesuai."
        )


    if rank_table[
        "Feature"
    ].duplicated().any():
        raise ValueError(
            "Terdapat fitur duplikat setelah "
            "penggabungan ranking."
        )


    # --------------------------------------------------------
    # Kolom ranking setiap model
    # --------------------------------------------------------

    rank_cols = [
        "Rank Random Forest",
        "Rank XGBoost",
        "Rank CatBoost"
    ]


    # --------------------------------------------------------
    # Kolom normalized SHAP setiap model
    # --------------------------------------------------------

    normalized_cols = [
        "Normalized SHAP Random Forest",
        "Normalized SHAP XGBoost",
        "Normalized SHAP CatBoost"
    ]


    rank_values = (
        rank_table[
            rank_cols
        ]
        .astype(float)
    )


    # --------------------------------------------------------
    # Validasi nilai rank
    # --------------------------------------------------------

    if (
        rank_values
        .le(0)
        .any()
        .any()
    ):
        raise ValueError(
            "Nilai rank harus lebih besar dari nol."
        )


    # --------------------------------------------------------
    # Mean normalized SHAP sebagai tie-break
    #
    # Jika skor agregasi sama, fitur dengan rata-rata
    # normalized SHAP lebih tinggi ditempatkan lebih dahulu.
    # --------------------------------------------------------

    rank_table[
        "Mean Normalized SHAP"
    ] = (
        rank_table[
            normalized_cols
        ]
        .astype(float)
        .mean(axis=1)
    )


    # ========================================================
    # MENGHITUNG SKOR EMPAT METODE AGREGASI
    #
    # Semakin kecil skor agregasi, semakin tinggi
    # peringkat akhir fitur.
    # ========================================================

    # --------------------------------------------------------
    # Arithmetic Mean
    # --------------------------------------------------------

    rank_table[
        "Arithmetic Mean Score"
    ] = (
        rank_values
        .mean(axis=1)
    )


    # --------------------------------------------------------
    # Median
    # --------------------------------------------------------

    rank_table[
        "Median Score"
    ] = (
        rank_values
        .median(axis=1)
    )


    # --------------------------------------------------------
    # Geometric Mean
    # --------------------------------------------------------

    rank_table[
        "Geometric Mean Score"
    ] = np.exp(
        np.log(
            rank_values
        )
        .mean(axis=1)
    )


    # --------------------------------------------------------
    # L2 Norm
    # --------------------------------------------------------

    rank_table[
        "L2 Norm Score"
    ] = np.sqrt(
        np.square(
            rank_values
        )
        .sum(axis=1)
    )


    aggregation_score_cols = {
        "Arithmetic Mean":
            "Arithmetic Mean Score",

        "Median":
            "Median Score",

        "Geometric Mean":
            "Geometric Mean Score",

        "L2 Norm":
            "L2 Norm Score"
    }


    aggregation_rankings = {}


    # ========================================================
    # MEMBENTUK URUTAN FINAL SETIAP METODE
    # ========================================================

    for method_name, score_col in (
        aggregation_score_cols.items()
    ):

        ranking = (
            rank_table
            .sort_values(
                by=[
                    score_col,
                    "Mean Normalized SHAP",
                    "Feature"
                ],
                ascending=[
                    True,
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
            .copy()
        )


        ranking[
            "Final Rank"
        ] = range(
            1,
            len(ranking) + 1
        )


        aggregation_rankings[
            method_name
        ] = ranking


        rank_mapping = (
            ranking
            .set_index("Feature")
            ["Final Rank"]
        )


        rank_table[
            f"Final Rank {method_name}"
        ] = (
            rank_table[
                "Feature"
            ]
            .map(
                rank_mapping
            )
            .astype(int)
        )


    # ========================================================
    # TABEL LENGKAP AGREGASI
    # ========================================================

    aggregation_display = rank_table[
        [
            "Feature",

            "Rank Random Forest",
            "Rank XGBoost",
            "Rank CatBoost",

            "Arithmetic Mean Score",
            "Final Rank Arithmetic Mean",

            "Median Score",
            "Final Rank Median",

            "Geometric Mean Score",
            "Final Rank Geometric Mean",

            "L2 Norm Score",
            "Final Rank L2 Norm"
        ]
    ].copy()


    aggregation_display = (
        aggregation_display
        .sort_values(
            "Final Rank Arithmetic Mean"
        )
        .reset_index(drop=True)
    )


    aggregation_display.insert(
        0,
        "No",
        range(
            1,
            len(aggregation_display) + 1
        )
    )


    # ========================================================
    # TABEL RINGKAS RANKING PER METODE
    # ========================================================

    aggregation_ranking_summary_parts = []


    for method_name in AGGREGATION_METHODS:

        if method_name not in aggregation_rankings:
            raise KeyError(
                "Metode agregasi tidak ditemukan: "
                f"{method_name}"
            )


        method_ranking = (
            aggregation_rankings[
                method_name
            ]
            .sort_values(
                "Final Rank"
            )
        )


        method_summary = (
            method_ranking[
                [
                    "Final Rank",
                    "Feature"
                ]
            ]
            .copy()
        )


        method_summary.insert(
            0,
            "Metode Agregasi",
            method_name
        )


        aggregation_ranking_summary_parts.append(
            method_summary
        )


    aggregation_ranking_summary = pd.concat(
        aggregation_ranking_summary_parts,
        ignore_index=True
    )


    # ========================================================
    # TABEL URUTAN SATU BARIS PER METODE
    # ========================================================

    aggregation_order_rows = []


    for method_name in AGGREGATION_METHODS:

        ordered_features = (
            aggregation_rankings[
                method_name
            ]
            .sort_values(
                "Final Rank"
            )
            ["Feature"]
            .tolist()
        )


        aggregation_order_rows.append({
            "Metode Agregasi":
                method_name,

            "Urutan Fitur":
                " > ".join(
                    ordered_features
                )
        })


    aggregation_order_table = pd.DataFrame(
        aggregation_order_rows
    )


    # ========================================================
    # MENYIMPAN HASIL BLOK 5
    # ========================================================

    rank_table.to_pickle(
        RANK_TABLE_PATH
    )


    rank_table.to_csv(
        os.path.join(
            BLOCK5_DIR,
            "rank_table_logloss.csv"
        ),
        index=False
    )


    aggregation_display.to_pickle(
        AGGREGATION_DISPLAY_PATH
    )


    aggregation_display.to_csv(
        os.path.join(
            BLOCK5_DIR,
            "aggregation_display_logloss.csv"
        ),
        index=False
    )


    joblib.dump(
        aggregation_rankings,
        AGGREGATION_RANKINGS_PATH
    )


    aggregation_ranking_summary.to_pickle(
        AGGREGATION_SUMMARY_PATH
    )


    aggregation_ranking_summary.to_csv(
        os.path.join(
            BLOCK5_DIR,
            "aggregation_ranking_summary_logloss.csv"
        ),
        index=False
    )


    aggregation_order_table.to_pickle(
        AGGREGATION_ORDER_PATH
    )


    aggregation_order_table.to_csv(
        os.path.join(
            BLOCK5_DIR,
            "aggregation_order_table_logloss.csv"
        ),
        index=False
    )


    # --------------------------------------------------------
    # Metadata Blok 5
    # --------------------------------------------------------

    block5_metadata = {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Sumber Ranking":
            (
                "Global SHAP Feature Importance "
                "dari model hasil optimasi "
                "Mean CV Log Loss terendah"
            ),

        "Objective Optimasi Model":
            "Mean CV Log Loss terendah",

        "Dasar Peringkat Fitur":
            "Mean Absolute SHAP Value",

        "Tie-break Agregasi":
            (
                "Mean Normalized SHAP tertinggi, "
                "kemudian nama fitur"
            ),

        "Jumlah Fitur":
            int(
                len(features_before_fs)
            ),

        "Model":
            list(
                MODEL_ORDER
            ),

        "Metode Agregasi":
            list(
                AGGREGATION_METHODS
            ),

        "Fitur":
            list(
                features_before_fs
            )
    }


    with open(
        os.path.join(
            BLOCK5_DIR,
            "block5_metadata_logloss.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            block5_metadata,
            file,
            ensure_ascii=False,
            indent=4
        )


    print(
        "\nHasil Blok 5 berhasil disimpan."
    )


# ============================================================
# VALIDASI HASIL BLOK 5
# ============================================================

required_rank_columns = [
    "Feature",
    "Rank Random Forest",
    "Rank XGBoost",
    "Rank CatBoost",
    "Arithmetic Mean Score",
    "Final Rank Arithmetic Mean",
    "Median Score",
    "Final Rank Median",
    "Geometric Mean Score",
    "Final Rank Geometric Mean",
    "L2 Norm Score",
    "Final Rank L2 Norm"
]


missing_rank_columns = [
    col
    for col in required_rank_columns
    if col not in rank_table.columns
]


if missing_rank_columns:
    raise ValueError(
        "Kolom rank_table belum lengkap: "
        f"{missing_rank_columns}"
    )


if len(
    rank_table
) != len(
    features_before_fs
):
    raise ValueError(
        "Jumlah baris rank_table tidak sesuai."
    )


for method_name in AGGREGATION_METHODS:

    final_rank_col = (
        f"Final Rank {method_name}"
    )


    expected_ranks = set(
        range(
            1,
            len(features_before_fs) + 1
        )
    )


    actual_ranks = set(
        rank_table[
            final_rank_col
        ].astype(int)
    )


    if actual_ranks != expected_ranks:
        raise ValueError(
            f"Final rank {method_name} "
            "tidak membentuk urutan lengkap."
        )


# ============================================================
# MEMERIKSA APAKAH URUTAN KEEMPAT METODE SAMA
# ============================================================

final_rank_cols = [
    f"Final Rank {method_name}"
    for method_name in AGGREGATION_METHODS
]


all_rankings_identical = (
    rank_table[
        final_rank_cols
    ]
    .nunique(axis=1)
    .eq(1)
    .all()
)


if all_rankings_identical:

    print(
        "\nKeempat metode agregasi "
        "menghasilkan urutan fitur yang sama."
    )

else:

    print(
        "\nTerdapat perbedaan urutan fitur "
        "antar-metode agregasi."
    )


# ============================================================
# OUTPUT TABEL LENGKAP
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "TABEL LENGKAP HASIL "
    "AGREGASI PERINGKAT FITUR"
)

print(
    "=" * 60
)


display_without_index(
    aggregation_display,
    formats={
        "Arithmetic Mean Score":
            "{:.3f}",

        "Median Score":
            "{:.3f}",

        "Geometric Mean Score":
            "{:.3f}",

        "L2 Norm Score":
            "{:.3f}"
    }
)


# ============================================================
# OUTPUT TABEL RINGKAS
# Satu baris untuk setiap metode agregasi
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "TABEL RINGKAS URUTAN FITUR "
    "SETIAP METODE AGREGASI"
)

print(
    "=" * 60
)


display_without_index(
    aggregation_order_table
)


print(
    "\nSeluruh hasil agregasi ranking "
    "tersimpan di:"
)

print(
    BLOCK5_DIR
)

## HASIL AGREGASI

In [ ]:
# ============================================================
# BLOK 6 — TUNING SETIAP TOP-K
# PADA SETIAP METODE AGREGASI DAN MODEL
#
# OBJECTIVE:
# MEAN CV LOG LOSS TERENDAH
#
# Mean CV ROC-AUC tetap dihitung dan disimpan
# sebagai metrik pendamping.
#
# Setiap Top-K memperoleh hyperparameter terbaiknya sendiri.
# Top-K maksimum merepresentasikan seluruh fitur.
# Dilengkapi checkpoint dan resume Optuna.
# ============================================================

import os
import json
import joblib


# ============================================================
# FOLDER PENYIMPANAN BLOK 6
#
# Folder dibuat berbeda dari eksperimen lama berbasis ROC-AUC
# agar database dan hasil lama tidak tercampur.
# ============================================================

BLOCK6_DIR = os.path.join(
    "./artifacts",
    (
        f"topk_tuning_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

TOPK_TRIAL_DIR = os.path.join(
    BLOCK6_DIR,
    "trial_tables"
)

os.makedirs(
    BLOCK6_DIR,
    exist_ok=True
)

os.makedirs(
    TOPK_TRIAL_DIR,
    exist_ok=True
)


# ============================================================
# DATABASE SQLITE OPTUNA
#
# Database menyimpan progres hingga tingkat trial sehingga
# tuning dapat dilanjutkan ketika session terputus.
# ============================================================

TOPK_OPTUNA_DB_PATH = os.path.join(
    BLOCK6_DIR,
    "topk_tuning_logloss_optuna.db"
)

TOPK_OPTUNA_STORAGE = (
    f"sqlite:///{TOPK_OPTUNA_DB_PATH}"
)


print(
    "Folder hasil Blok 6:",
    BLOCK6_DIR
)

print(
    "Database Optuna:",
    TOPK_OPTUNA_DB_PATH
)

print(
    "Objective optimasi:",
    "Mean CV Log Loss terendah"
)

print(
    "Metrik pendamping:",
    "Mean CV ROC-AUC"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block6_objects = [
    "X_train_raw",
    "y_train",
    "features_before_fs",
    "MODEL_ORDER",
    "AGGREGATION_METHODS",
    "LABEL_DISPLAY_NAME",
    "MIN_K",
    "MAX_K",
    "N_TRIALS_TOPK",
    "RANDOM_STATE",
    "cv_splits"
]

missing_block6_objects = [
    name
    for name in required_block6_objects
    if name not in globals()
]

if missing_block6_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 6: "
        f"{missing_block6_objects}"
    )


features_before_fs = list(
    features_before_fs
)


if not features_before_fs:
    raise ValueError(
        "features_before_fs tidak boleh kosong."
    )


if MIN_K < 1:
    raise ValueError(
        "MIN_K minimal bernilai 1."
    )


if MAX_K != len(
    features_before_fs
):
    raise ValueError(
        "MAX_K harus sama dengan jumlah seluruh fitur."
    )


if MIN_K > MAX_K:
    raise ValueError(
        "MIN_K tidak boleh lebih besar daripada MAX_K."
    )


# ============================================================
# MEMUAT AGGREGATION RANKINGS DARI BLOK 5
# JIKA VARIABEL TIDAK TERSEDIA
# ============================================================

if (
    "aggregation_rankings" not in globals()
    or not aggregation_rankings
):
    print(
        "Variabel aggregation_rankings tidak tersedia. "
        "Memuat hasil Blok 5 berbasis Log Loss."
    )


    if "BLOCK5_DIR" not in globals():
        BLOCK5_DIR = os.path.join(
            "./artifacts",
            (
                f"aggregation_ranking_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    aggregation_rankings_path = os.path.join(
        BLOCK5_DIR,
        "aggregation_rankings_logloss.joblib"
    )


    if not os.path.exists(
        aggregation_rankings_path
    ):
        raise FileNotFoundError(
            "Hasil agregasi ranking berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{aggregation_rankings_path}\n"
            "Jalankan Blok 5 terlebih dahulu."
        )


    aggregation_rankings = joblib.load(
        aggregation_rankings_path
    )


    print(
        "Aggregation rankings berhasil dimuat."
    )


# ============================================================
# VALIDASI AGGREGATION RANKINGS
# ============================================================

missing_aggregation_methods = [
    method_name
    for method_name in AGGREGATION_METHODS
    if method_name not in aggregation_rankings
]

if missing_aggregation_methods:
    raise KeyError(
        "Ranking belum tersedia untuk metode agregasi: "
        f"{missing_aggregation_methods}"
    )


for method_name in AGGREGATION_METHODS:

    method_ranking = aggregation_rankings[
        method_name
    ]


    required_ranking_columns = [
        "Feature",
        "Final Rank"
    ]


    missing_columns = [
        col
        for col in required_ranking_columns
        if col not in method_ranking.columns
    ]


    if missing_columns:
        raise ValueError(
            f"Kolom ranking {method_name} belum lengkap: "
            f"{missing_columns}"
        )


    if len(
        method_ranking
    ) != len(
        features_before_fs
    ):
        raise ValueError(
            f"Jumlah fitur ranking {method_name} tidak sesuai."
        )


    if set(
        method_ranking["Feature"]
    ) != set(
        features_before_fs
    ):
        raise ValueError(
            f"Daftar fitur ranking {method_name} "
            "tidak sesuai dengan features_before_fs."
        )


    if method_ranking[
        "Feature"
    ].duplicated().any():
        raise ValueError(
            f"Terdapat fitur duplikat pada ranking "
            f"{method_name}."
        )


# ============================================================
# FUNGSI PENDUKUNG BLOK 6
# ============================================================

def make_topk_study_name(
    method_name,
    model_name,
    k
):
    """
    Membentuk nama study Optuna yang unik untuk setiap
    metode agregasi, model, dan Top-K.
    """

    return (
        f"topk_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}_"
        f"{safe_file_name(method_name)}_"
        f"{safe_file_name(model_name)}_"
        f"Top_{k}"
    )


def get_complete_trials(
    study
):
    """
    Mengambil trial Optuna yang telah selesai dan memiliki
    nilai objective.
    """

    return [
        trial
        for trial in study.trials
        if (
            trial.state == TrialState.COMPLETE
            and trial.value is not None
            and np.isfinite(
                float(trial.value)
            )
        )
    ]


def get_trial_duration_seconds(
    trial
):
    """
    Mengambil waktu komputasi satu trial dalam detik.
    """

    if trial.duration is None:
        return np.inf

    return float(
        trial.duration.total_seconds()
    )


def calculate_study_time_seconds(
    study
):
    """
    Menghitung total waktu seluruh trial yang selesai
    pada satu study.
    """

    durations = [
        get_trial_duration_seconds(
            trial
        )
        for trial in get_complete_trials(
            study
        )
    ]


    finite_durations = [
        duration
        for duration in durations
        if np.isfinite(
            duration
        )
    ]


    return float(
        np.sum(
            finite_durations
        )
    )


def select_best_logloss_trial(
    study,
    tolerance=1e-12
):
    """
    Memilih trial terbaik berdasarkan:

    1. Mean CV Log Loss terendah;
    2. waktu komputasi lebih rendah apabila Log Loss sama;
    3. nomor trial lebih rendah apabila keduanya sama.
    """

    complete_trials = get_complete_trials(
        study
    )


    if not complete_trials:
        raise RuntimeError(
            f"Tidak terdapat trial selesai pada "
            f"{study.study_name}."
        )


    minimum_logloss = min(
        float(trial.value)
        for trial in complete_trials
    )


    tied_trials = [
        trial
        for trial in complete_trials
        if np.isclose(
            float(trial.value),
            minimum_logloss,
            rtol=0.0,
            atol=tolerance
        )
    ]


    selected_trial = min(
        tied_trials,
        key=lambda trial: (
            get_trial_duration_seconds(
                trial
            ),
            trial.number
        )
    )


    return selected_trial


def validate_study_features(
    study,
    selected_features
):
    """
    Memastikan study lama menggunakan daftar dan urutan fitur
    yang sama dengan ranking saat ini.
    """

    saved_features_json = (
        study.user_attrs.get(
            "selected_features_json"
        )
    )


    current_features_json = json.dumps(
        list(selected_features),
        ensure_ascii=False
    )


    if saved_features_json is None:

        study.set_user_attr(
            "selected_features_json",
            current_features_json
        )


    elif saved_features_json != current_features_json:

        raise ValueError(
            "Daftar fitur pada study lama berbeda "
            "dengan ranking saat ini.\n"
            f"Study: {study.study_name}\n"
            "Gunakan database baru atau hapus "
            "database Blok 6 lama."
        )


def validate_study_objective(
    study
):
    """
    Memastikan study menggunakan objective Log Loss
    dengan arah minimization.
    """

    saved_objective = study.user_attrs.get(
        "optimization_objective"
    )


    current_objective = (
        "Mean CV Log Loss"
    )


    if saved_objective is None:

        study.set_user_attr(
            "optimization_objective",
            current_objective
        )


    elif saved_objective != current_objective:

        raise ValueError(
            "Objective study lama berbeda dengan "
            "objective Blok 6 saat ini.\n"
            f"Study: {study.study_name}\n"
            "Gunakan database baru."
        )


# ============================================================
# TUNING SELURUH KOMBINASI
# ============================================================

topk_studies = {}

topk_trials_full_parts = []

topk_best_rows = []


total_combinations = (
    len(AGGREGATION_METHODS)
    * len(MODEL_ORDER)
    * (MAX_K - MIN_K + 1)
)


combination_counter = 0


for method_name in AGGREGATION_METHODS:

    ordered_features = (
        aggregation_rankings[
            method_name
        ]
        .sort_values(
            "Final Rank"
        )
        ["Feature"]
        .tolist()
    )


    if len(
        ordered_features
    ) != MAX_K:
        raise ValueError(
            f"Jumlah fitur urutan {method_name} "
            f"tidak sama dengan MAX_K."
        )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "METODE AGREGASI:",
        method_name
    )

    print(
        "=" * 70
    )


    for model_name in MODEL_ORDER:

        print(
            f"\nMODEL: {model_name}"
        )


        for k in range(
            MIN_K,
            MAX_K + 1
        ):

            combination_counter += 1


            selected_features = list(
                ordered_features[:k]
            )


            study_key = (
                method_name,
                model_name,
                k
            )


            study_name = make_topk_study_name(
                method_name=method_name,
                model_name=model_name,
                k=k
            )


            # =================================================
            # MEMBUAT ATAU MEMUAT STUDY
            #
            # direction="minimize" karena objective berupa
            # Mean CV Log Loss.
            # =================================================

            study = optuna.create_study(
                study_name=study_name,
                direction="minimize",
                sampler=TPESampler(
                    seed=RANDOM_STATE
                ),
                storage=TOPK_OPTUNA_STORAGE,
                load_if_exists=True
            )


            validate_study_features(
                study=study,
                selected_features=selected_features
            )


            validate_study_objective(
                study=study
            )


            complete_trials_before = (
                get_complete_trials(
                    study
                )
            )


            n_completed_before = len(
                complete_trials_before
            )


            remaining_trials = max(
                0,
                N_TRIALS_TOPK
                - n_completed_before
            )


            print(
                f"\n[{combination_counter}/"
                f"{total_combinations}] "
                f"{model_name} — "
                f"{method_name} — Top-{k}"
            )


            print(
                "Trial selesai :",
                n_completed_before
            )


            print(
                "Sisa trial    :",
                remaining_trials
            )


            # =================================================
            # MELANJUTKAN TRIAL YANG BELUM SELESAI
            # =================================================

            if remaining_trials > 0:

                study.optimize(
                    create_logloss_objective(
                        model_name=model_name,
                        selected_features=selected_features
                    ),
                    n_trials=remaining_trials,
                    n_jobs=1,
                    gc_after_trial=True
                )


            else:

                print(
                    "Status        : "
                    "selesai, tuning dilewati."
                )


            topk_studies[
                study_key
            ] = study


            # =================================================
            # MEMERIKSA JUMLAH TRIAL SETELAH TUNING
            # =================================================

            complete_trials_after = (
                get_complete_trials(
                    study
                )
            )


            n_completed_after = len(
                complete_trials_after
            )


            if n_completed_after == 0:

                raise RuntimeError(
                    "Tidak ada trial selesai pada "
                    f"{study_name}."
                )


            # =================================================
            # MEMILIH TRIAL TERBAIK
            #
            # Tidak langsung menggunakan study.best_trial agar
            # waktu komputasi dapat digunakan ketika terdapat
            # nilai Log Loss yang sama.
            # =================================================

            selected_trial = (
                select_best_logloss_trial(
                    study
                )
            )


            best_mean_logloss = float(
                selected_trial.value
            )


            best_mean_auc = float(
                selected_trial.user_attrs.get(
                    "mean_cv_roc_auc",
                    np.nan
                )
            )


            selected_trial_time = (
                get_trial_duration_seconds(
                    selected_trial
                )
            )


            cumulative_time = (
                calculate_study_time_seconds(
                    study
                )
            )


            # =================================================
            # TABEL SELURUH TRIAL KOMBINASI
            #
            # Disimpan, tetapi tidak ditampilkan pada setiap
            # kombinasi agar output notebook tidak terlalu
            # panjang.
            # =================================================

            trial_table = study_trials_table(
                study=study,
                model_name=model_name,
                context_columns={
                    "Skenario Pelabelan":
                        LABEL_DISPLAY_NAME,

                    "Metode Agregasi":
                        method_name,

                    "Model":
                        model_name,

                    "Top-K":
                        k,

                    "Fitur":
                        ", ".join(
                            selected_features
                        )
                }
            )


            if trial_table.empty:

                raise RuntimeError(
                    f"Tabel trial kosong pada {study_name}."
                )


            topk_trials_full_parts.append(
                trial_table
            )


            trial_file_name = (
                f"trial_logloss_"
                f"{safe_file_name(method_name)}_"
                f"{safe_file_name(model_name)}_"
                f"Top_{k}.csv"
            )


            trial_table_path = os.path.join(
                TOPK_TRIAL_DIR,
                trial_file_name
            )


            trial_table.to_csv(
                trial_table_path,
                index=False
            )


            trial_table.to_pickle(
                trial_table_path.replace(
                    ".csv",
                    ".pkl"
                )
            )


            # =================================================
            # MENYIMPAN HASIL TERBAIK SETIAP KOMBINASI
            # =================================================

            best_row = {
                "Skenario Pelabelan":
                    LABEL_DISPLAY_NAME,

                "Metode Agregasi":
                    method_name,

                "Model":
                    model_name,

                "Top-K":
                    int(k),

                "Fitur":
                    ", ".join(
                        selected_features
                    ),

                "Best Mean CV Log Loss":
                    best_mean_logloss,

                "Mean CV ROC-AUC":
                    best_mean_auc,

                "Selected Trial":
                    int(
                        selected_trial.number
                    ),

                "Waktu Trial Terpilih (s)":
                    selected_trial_time,

                "Jumlah Trial Selesai":
                    int(
                        n_completed_after
                    ),

                "Waktu Tuning Kumulatif (s)":
                    cumulative_time,

                "Hyperparameter Terbaik":
                    params_to_text(
                        selected_trial.params
                    )
            }


            topk_best_rows.append(
                best_row
            )


            # =================================================
            # CHECKPOINT SETELAH SATU TOP-K SELESAI
            # =================================================

            checkpoint_df = pd.DataFrame(
                topk_best_rows
            )


            checkpoint_df.to_csv(
                os.path.join(
                    BLOCK6_DIR,
                    "topk_best_logloss_checkpoint.csv"
                ),
                index=False
            )


            checkpoint_df.to_pickle(
                os.path.join(
                    BLOCK6_DIR,
                    "topk_best_logloss_checkpoint.pkl"
                )
            )


            print(
                "Best Log Loss :",
                f"{best_mean_logloss:.6f}"
            )


            print(
                "Mean ROC-AUC  :",
                (
                    f"{best_mean_auc:.6f}"
                    if np.isfinite(best_mean_auc)
                    else "tidak tersedia"
                )
            )


            print(
                "Trial terpilih:",
                selected_trial.number
            )


            print(
                "Waktu trial   :",
                (
                    f"{selected_trial_time:.2f} detik"
                    if np.isfinite(
                        selected_trial_time
                    )
                    else "tidak tersedia"
                )
            )


            print(
                "Waktu kumulatif:",
                f"{cumulative_time:.2f} detik"
            )


            print(
                "Checkpoint    : tersimpan"
            )


            gc.collect()


# ============================================================
# MENGGABUNGKAN SELURUH HASIL
# ============================================================

if not topk_trials_full_parts:

    raise RuntimeError(
        "Tidak terdapat tabel trial yang dapat digabungkan."
    )


topk_trials_full = pd.concat(
    topk_trials_full_parts,
    ignore_index=True
)


topk_best_full = pd.DataFrame(
    topk_best_rows
)


if topk_best_full.empty:

    raise RuntimeError(
        "Hasil terbaik Top-K kosong."
    )


# ------------------------------------------------------------
# Mengatur urutan metode dan model
# ------------------------------------------------------------

topk_best_full[
    "Metode Agregasi"
] = pd.Categorical(
    topk_best_full[
        "Metode Agregasi"
    ],
    categories=AGGREGATION_METHODS,
    ordered=True
)


topk_best_full[
    "Model"
] = pd.Categorical(
    topk_best_full[
        "Model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


topk_best_full = (
    topk_best_full
    .sort_values(
        by=[
            "Metode Agregasi",
            "Model",
            "Top-K"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


topk_best_full[
    "Metode Agregasi"
] = topk_best_full[
    "Metode Agregasi"
].astype(str)


topk_best_full[
    "Model"
] = topk_best_full[
    "Model"
].astype(str)


# ============================================================
# VALIDASI JUMLAH KOMBINASI
# ============================================================

expected_combinations = (
    len(AGGREGATION_METHODS)
    * len(MODEL_ORDER)
    * (MAX_K - MIN_K + 1)
)


if len(
    topk_best_full
) != expected_combinations:

    raise ValueError(
        "Jumlah hasil kombinasi belum sesuai. "
        f"Ditemukan={len(topk_best_full)}, "
        f"seharusnya={expected_combinations}."
    )


duplicate_combinations = (
    topk_best_full
    .duplicated(
        subset=[
            "Metode Agregasi",
            "Model",
            "Top-K"
        ]
    )
    .sum()
)


if duplicate_combinations > 0:

    raise ValueError(
        "Terdapat kombinasi metode, model, dan Top-K "
        "yang duplikat."
    )


# ============================================================
# MENYIMPAN HASIL FINAL BLOK 6
# ============================================================

topk_trials_full.to_csv(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_seluruh_trial.csv"
    ),
    index=False
)


topk_trials_full.to_pickle(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_seluruh_trial.pkl"
    )
)


topk_best_full.to_csv(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_full.csv"
    ),
    index=False
)


topk_best_full.to_pickle(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_full.pkl"
    )
)


# ============================================================
# TABEL RINGKAS HASIL SETIAP TOP-K
# ============================================================

topk_best_summary = topk_best_full[
    [
        "Metode Agregasi",
        "Model",
        "Top-K",
        "Best Mean CV Log Loss",
        "Mean CV ROC-AUC",
        "Selected Trial",
        "Waktu Trial Terpilih (s)",
        "Jumlah Trial Selesai",
        "Waktu Tuning Kumulatif (s)"
    ]
].copy()


topk_best_summary.to_csv(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_summary.csv"
    ),
    index=False
)


topk_best_summary.to_pickle(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_summary.pkl"
    )
)


# ============================================================
# MENYIMPAN HYPERPARAMETER TERBAIK DALAM FORMAT JSON
# ============================================================

topk_best_params_json = {}


for _, row in topk_best_full.iterrows():

    key = (
        f"{row['Metode Agregasi']} | "
        f"{row['Model']} | "
        f"Top-{int(row['Top-K'])}"
    )


    topk_best_params_json[
        key
    ] = {
        "Fitur":
            [
                feature.strip()
                for feature in str(
                    row["Fitur"]
                ).split(",")
            ],

        "Best Mean CV Log Loss":
            float(
                row[
                    "Best Mean CV Log Loss"
                ]
            ),

        "Mean CV ROC-AUC":
            (
                None
                if pd.isna(
                    row["Mean CV ROC-AUC"]
                )
                else float(
                    row["Mean CV ROC-AUC"]
                )
            ),

        "Selected Trial":
            int(
                row["Selected Trial"]
            ),

        "Hyperparameter Terbaik":
            json.loads(
                row[
                    "Hyperparameter Terbaik"
                ]
            )
    }


with open(
    os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_params.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        topk_best_params_json,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# METADATA BLOK 6
# ============================================================

block6_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Objective Optimasi":
        "Mean CV Log Loss terendah",

    "Arah Optimasi":
        "minimize",

    "Metrik Pendamping":
        "Mean CV ROC-AUC",

    "Tie-break Trial":
        (
            "Waktu trial lebih rendah apabila "
            "Mean CV Log Loss sama"
        ),

    "Jumlah Metode Agregasi":
        int(
            len(AGGREGATION_METHODS)
        ),

    "Jumlah Model":
        int(
            len(MODEL_ORDER)
        ),

    "Minimum Top-K":
        int(
            MIN_K
        ),

    "Maximum Top-K":
        int(
            MAX_K
        ),

    "Trial per Top-K":
        int(
            N_TRIALS_TOPK
        ),

    "Jumlah Kombinasi":
        int(
            expected_combinations
        ),

    "Cross-Validation Fold":
        int(
            N_SPLITS
        ),

    "Metode Agregasi":
        list(
            AGGREGATION_METHODS
        ),

    "Model":
        list(
            MODEL_ORDER
        )
}


with open(
    os.path.join(
        BLOCK6_DIR,
        "block6_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block6_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT LENGKAP HASIL TERBAIK SETIAP TOP-K
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "TABEL LENGKAP HASIL TERBAIK "
    "SETIAP TOP-K"
)

print(
    "=" * 70
)


display_without_index(
    topk_best_full,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Trial Terpilih (s)":
            "{:.2f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


# ============================================================
# OUTPUT RINGKAS HASIL SETIAP TOP-K
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "TABEL RINGKAS HASIL TERBAIK "
    "SETIAP TOP-K"
)

print(
    "=" * 70
)


display_without_index(
    topk_best_summary,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Trial Terpilih (s)":
            "{:.2f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\nKriteria pemilihan hyperparameter "
    "pada setiap Top-K:"
)

print(
    "- Mean CV Log Loss terendah"
)

print(
    "- Waktu trial lebih rendah apabila "
    "Mean CV Log Loss sama"
)

print(
    "- Mean CV ROC-AUC disimpan sebagai "
    "metrik pendamping"
)


print(
    "\nSeluruh hasil tuning Top-K "
    "berhasil disimpan di:"
)

print(
    BLOCK6_DIR
)

## TRIAL

In [ ]:
# ============================================================
# BLOK 7 — PEMILIHAN TOP-K TERBAIK,
# METODE AGREGASI TERBAIK,
# DAN HYPERPARAMETER FINAL
#
# KRITERIA UTAMA:
# 1. MEAN CV LOG LOSS TERENDAH
# 2. WAKTU KOMPUTASI LEBIH RENDAH JIKA NILAI SAMA
#
# Mean CV ROC-AUC tetap disimpan sebagai metrik pendamping.
# Dilengkapi penyimpanan dan resume.
# ============================================================

import os
import json
import joblib


# ============================================================
# FOLDER PENYIMPANAN BLOK 7
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC.
# ============================================================

BLOCK7_DIR = os.path.join(
    "./artifacts",
    (
        f"final_combination_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

os.makedirs(
    BLOCK7_DIR,
    exist_ok=True
)


print(
    "Folder hasil Blok 7:",
    BLOCK7_DIR
)

print(
    "Kriteria pemilihan:",
    "Mean CV Log Loss terendah"
)

print(
    "Metrik pendamping:",
    "Mean CV ROC-AUC"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block7_objects = [
    "MODEL_ORDER",
    "AGGREGATION_METHODS",
    "PARAMETER_ORDER",
    "LABEL_DISPLAY_NAME",
    "MIN_K",
    "MAX_K",
    "N_TRIALS_TOPK"
]


missing_block7_objects = [
    name
    for name in required_block7_objects
    if name not in globals()
]


if missing_block7_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 7: "
        f"{missing_block7_objects}"
    )


# ============================================================
# MEMUAT HASIL BLOK 6
# JIKA VARIABEL TIDAK TERSEDIA
# ============================================================

if (
    "topk_best_full" not in globals()
    or topk_best_full is None
    or len(topk_best_full) == 0
):

    print(
        "Variabel topk_best_full tidak tersedia. "
        "Memuat hasil Blok 6 berbasis Log Loss."
    )


    if "BLOCK6_DIR" not in globals():

        BLOCK6_DIR = os.path.join(
            "./artifacts",
            (
                f"topk_tuning_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    topk_best_path = os.path.join(
        BLOCK6_DIR,
        "topk_logloss_best_full.pkl"
    )


    if not os.path.exists(
        topk_best_path
    ):

        raise FileNotFoundError(
            "Hasil tuning Top-K berbasis Log Loss "
            "belum tersedia pada path:\n"
            f"{topk_best_path}\n"
            "Jalankan Blok 6 hingga selesai."
        )


    topk_best_full = pd.read_pickle(
        topk_best_path
    )


    print(
        "Hasil Blok 6 berhasil dimuat."
    )


# ============================================================
# MEMUAT HASIL BLOK 5
# JIKA VARIABEL TIDAK TERSEDIA
# ============================================================

if (
    "aggregation_rankings" not in globals()
    or not aggregation_rankings
):

    print(
        "Variabel aggregation_rankings tidak tersedia. "
        "Memuat hasil Blok 5 berbasis Log Loss."
    )


    if "BLOCK5_DIR" not in globals():

        BLOCK5_DIR = os.path.join(
            "./artifacts",
            (
                f"aggregation_ranking_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    aggregation_rankings_path = os.path.join(
        BLOCK5_DIR,
        "aggregation_rankings_logloss.joblib"
    )


    if not os.path.exists(
        aggregation_rankings_path
    ):

        raise FileNotFoundError(
            "Hasil agregasi ranking berbasis Log Loss "
            "belum tersedia pada path:\n"
            f"{aggregation_rankings_path}\n"
            "Jalankan Blok 5 terlebih dahulu."
        )


    aggregation_rankings = joblib.load(
        aggregation_rankings_path
    )


    print(
        "Hasil Blok 5 berhasil dimuat."
    )


# ============================================================
# VALIDASI HASIL BLOK 6
# ============================================================

required_topk_columns = [
    "Skenario Pelabelan",
    "Metode Agregasi",
    "Model",
    "Top-K",
    "Fitur",
    "Best Mean CV Log Loss",
    "Mean CV ROC-AUC",
    "Selected Trial",
    "Waktu Trial Terpilih (s)",
    "Jumlah Trial Selesai",
    "Waktu Tuning Kumulatif (s)",
    "Hyperparameter Terbaik"
]


missing_topk_columns = [
    column
    for column in required_topk_columns
    if column not in topk_best_full.columns
]


if missing_topk_columns:

    raise ValueError(
        "Kolom hasil Blok 6 belum sesuai: "
        f"{missing_topk_columns}"
    )


# ------------------------------------------------------------
# Validasi nilai numerik utama
# ------------------------------------------------------------

numeric_validation_columns = [
    "Top-K",
    "Best Mean CV Log Loss",
    "Mean CV ROC-AUC",
    "Waktu Trial Terpilih (s)",
    "Jumlah Trial Selesai",
    "Waktu Tuning Kumulatif (s)"
]


for column in numeric_validation_columns:

    topk_best_full[column] = pd.to_numeric(
        topk_best_full[column],
        errors="coerce"
    )


if topk_best_full[
    "Best Mean CV Log Loss"
].isna().any():

    raise ValueError(
        "Terdapat Mean CV Log Loss yang kosong "
        "atau tidak valid pada hasil Blok 6."
    )


if not np.isfinite(
    topk_best_full[
        "Best Mean CV Log Loss"
    ].to_numpy(dtype=float)
).all():

    raise ValueError(
        "Terdapat Mean CV Log Loss yang tidak finite."
    )


# ------------------------------------------------------------
# Validasi jumlah kombinasi
# ------------------------------------------------------------

expected_combinations = (
    len(AGGREGATION_METHODS)
    * len(MODEL_ORDER)
    * (MAX_K - MIN_K + 1)
)


if len(
    topk_best_full
) != expected_combinations:

    raise ValueError(
        "Blok 6 belum menyelesaikan seluruh kombinasi. "
        f"Ditemukan={len(topk_best_full)}, "
        f"seharusnya={expected_combinations}."
    )


# ------------------------------------------------------------
# Validasi kombinasi duplikat
# ------------------------------------------------------------

duplicate_combinations = (
    topk_best_full
    .duplicated(
        subset=[
            "Metode Agregasi",
            "Model",
            "Top-K"
        ]
    )
    .sum()
)


if duplicate_combinations > 0:

    raise ValueError(
        "Terdapat kombinasi metode agregasi, model, "
        "dan Top-K yang duplikat."
    )


# ------------------------------------------------------------
# Validasi jumlah trial
# ------------------------------------------------------------

if (
    topk_best_full[
        "Jumlah Trial Selesai"
    ].min()
    < N_TRIALS_TOPK
):

    incomplete_rows = (
        topk_best_full
        .loc[
            topk_best_full[
                "Jumlah Trial Selesai"
            ] < N_TRIALS_TOPK,
            [
                "Metode Agregasi",
                "Model",
                "Top-K",
                "Jumlah Trial Selesai"
            ]
        ]
        .copy()
    )


    raise ValueError(
        "Masih terdapat kombinasi Top-K "
        "yang belum mencapai jumlah trial final.\n"
        f"{incomplete_rows}"
    )


# ------------------------------------------------------------
# Validasi metode agregasi dan model
# ------------------------------------------------------------

unknown_methods = sorted(
    set(
        topk_best_full[
            "Metode Agregasi"
        ].astype(str)
    )
    - set(
        AGGREGATION_METHODS
    )
)


if unknown_methods:

    raise ValueError(
        "Terdapat metode agregasi yang tidak dikenali: "
        f"{unknown_methods}"
    )


unknown_models = sorted(
    set(
        topk_best_full[
            "Model"
        ].astype(str)
    )
    - set(
        MODEL_ORDER
    )
)


if unknown_models:

    raise ValueError(
        "Terdapat model yang tidak dikenali: "
        f"{unknown_models}"
    )


# ============================================================
# FUNGSI PEMILIHAN DENGAN TOLERANSI
# ============================================================

LOGLOSS_TIE_TOLERANCE = 1e-12


def choose_best_topk(
    group,
    tolerance=LOGLOSS_TIE_TOLERANCE
):
    """
    Memilih Top-K terbaik pada satu kombinasi
    metode agregasi dan model.

    Urutan pemilihan:
    1. Best Mean CV Log Loss terendah;
    2. waktu tuning kumulatif lebih rendah jika Log Loss sama;
    3. Top-K lebih kecil jika keduanya sama.
    """

    group = group.copy()


    minimum_logloss = float(
        group[
            "Best Mean CV Log Loss"
        ].min()
    )


    tied_group = group.loc[
        np.isclose(
            group[
                "Best Mean CV Log Loss"
            ].astype(float),
            minimum_logloss,
            rtol=0.0,
            atol=tolerance
        )
    ].copy()


    tied_group = (
        tied_group
        .sort_values(
            by=[
                "Waktu Tuning Kumulatif (s)",
                "Top-K"
            ],
            ascending=[
                True,
                True
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )


    return tied_group.iloc[0]


def choose_best_aggregation(
    aggregation_table,
    tolerance=LOGLOSS_TIE_TOLERANCE
):
    """
    Memilih metode agregasi terbaik.

    Urutan pemilihan:
    1. rata-rata Mean CV Log Loss terendah;
    2. total waktu tiga konfigurasi terpilih lebih rendah;
    3. nama metode agregasi secara alfabetis.
    """

    aggregation_table = aggregation_table.copy()


    minimum_mean_logloss = float(
        aggregation_table[
            "Rata-rata Mean CV Log Loss"
        ].min()
    )


    tied_methods = aggregation_table.loc[
        np.isclose(
            aggregation_table[
                "Rata-rata Mean CV Log Loss"
            ].astype(float),
            minimum_mean_logloss,
            rtol=0.0,
            atol=tolerance
        )
    ].copy()


    tied_methods = (
        tied_methods
        .sort_values(
            by=[
                "Total Waktu Tiga Konfigurasi Terpilih (s)",
                "Metode Agregasi"
            ],
            ascending=[
                True,
                True
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )


    return tied_methods.iloc[0]


# ============================================================
# 1. MEMILIH TOP-K TERBAIK
# SETIAP METODE AGREGASI DAN MODEL
#
# Kriteria:
# 1. Mean CV Log Loss terendah
# 2. Waktu tuning kumulatif lebih rendah jika sama
# 3. Top-K lebih kecil jika masih sama
# ============================================================

best_topk_rows = []


for method_name in AGGREGATION_METHODS:

    for model_name in MODEL_ORDER:

        subset = (
            topk_best_full
            .loc[
                (
                    topk_best_full[
                        "Metode Agregasi"
                    ] == method_name
                )
                &
                (
                    topk_best_full[
                        "Model"
                    ] == model_name
                )
            ]
            .copy()
        )


        if subset.empty:

            raise ValueError(
                "Hasil Top-K tidak ditemukan untuk "
                f"{method_name} — {model_name}."
            )


        best_row = choose_best_topk(
            subset
        )


        best_topk_rows.append(
            best_row.to_dict()
        )


best_topk_per_method_model = pd.DataFrame(
    best_topk_rows
)


# ------------------------------------------------------------
# Mengatur urutan metode dan model
# ------------------------------------------------------------

best_topk_per_method_model[
    "Metode Agregasi"
] = pd.Categorical(
    best_topk_per_method_model[
        "Metode Agregasi"
    ],
    categories=AGGREGATION_METHODS,
    ordered=True
)


best_topk_per_method_model[
    "Model"
] = pd.Categorical(
    best_topk_per_method_model[
        "Model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


best_topk_per_method_model = (
    best_topk_per_method_model
    .sort_values(
        by=[
            "Metode Agregasi",
            "Model"
        ]
    )
    .reset_index(drop=True)
)


best_topk_per_method_model[
    "Metode Agregasi"
] = best_topk_per_method_model[
    "Metode Agregasi"
].astype(str)


best_topk_per_method_model[
    "Model"
] = best_topk_per_method_model[
    "Model"
].astype(str)


# ------------------------------------------------------------
# Menambahkan nomor
# ------------------------------------------------------------

best_topk_per_method_model.insert(
    0,
    "No",
    range(
        1,
        len(
            best_topk_per_method_model
        ) + 1
    )
)


# ------------------------------------------------------------
# Validasi jumlah hasil
# ------------------------------------------------------------

expected_best_topk_rows = (
    len(AGGREGATION_METHODS)
    * len(MODEL_ORDER)
)


if len(
    best_topk_per_method_model
) != expected_best_topk_rows:

    raise ValueError(
        "Jumlah Top-K terbaik per metode dan model "
        "tidak sesuai."
    )


# ------------------------------------------------------------
# Tabel ringkas Top-K terbaik
# ------------------------------------------------------------

best_topk_summary = (
    best_topk_per_method_model[
        [
            "No",
            "Metode Agregasi",
            "Model",
            "Top-K",
            "Best Mean CV Log Loss",
            "Mean CV ROC-AUC",
            "Waktu Tuning Kumulatif (s)"
        ]
    ]
    .copy()
)


# ============================================================
# 2. MEMBANDINGKAN METODE AGREGASI
#
# Setiap metode memiliki tiga konfigurasi Top-K terbaik:
# - Random Forest
# - XGBoost
# - CatBoost
#
# Rata-rata Log Loss:
# rata-rata Mean CV Log Loss dari tiga konfigurasi terpilih.
#
# Total waktu:
# jumlah waktu tuning dari tiga konfigurasi Top-K terpilih,
# bukan waktu seluruh pengujian Top-1 sampai Top-11.
# ============================================================

aggregation_comparison = (
    best_topk_per_method_model
    .groupby(
        "Metode Agregasi",
        as_index=False,
        sort=False
    )
    .agg(
        **{
            "Rata-rata Mean CV Log Loss": (
                "Best Mean CV Log Loss",
                "mean"
            ),

            "Rata-rata Mean CV ROC-AUC": (
                "Mean CV ROC-AUC",
                "mean"
            ),

            "Total Waktu Tiga Konfigurasi Terpilih (s)": (
                "Waktu Tuning Kumulatif (s)",
                "sum"
            ),

            "Jumlah Model": (
                "Model",
                "nunique"
            )
        }
    )
)


# ------------------------------------------------------------
# Memastikan setiap metode memiliki tiga model
# ------------------------------------------------------------

if not (
    aggregation_comparison[
        "Jumlah Model"
    ] == len(MODEL_ORDER)
).all():

    raise ValueError(
        "Tidak semua metode agregasi memiliki "
        "hasil untuk seluruh model."
    )


# ------------------------------------------------------------
# Menentukan metode agregasi terpilih
# ------------------------------------------------------------

selected_aggregation_row = (
    choose_best_aggregation(
        aggregation_comparison
    )
)


selected_aggregation_method = str(
    selected_aggregation_row[
        "Metode Agregasi"
    ]
)


# ------------------------------------------------------------
# Memberi status metode terpilih
# ------------------------------------------------------------

aggregation_comparison[
    "Status"
] = np.where(
    aggregation_comparison[
        "Metode Agregasi"
    ] == selected_aggregation_method,
    "Terpilih",
    "Tidak Terpilih"
)


# ------------------------------------------------------------
# Mengurutkan tabel perbandingan
# ------------------------------------------------------------

aggregation_comparison = (
    aggregation_comparison
    .sort_values(
        by=[
            "Rata-rata Mean CV Log Loss",
            "Total Waktu Tiga Konfigurasi Terpilih (s)",
            "Metode Agregasi"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


aggregation_comparison.insert(
    0,
    "Peringkat",
    range(
        1,
        len(
            aggregation_comparison
        ) + 1
    )
)


# ------------------------------------------------------------
# Tabel ringkas perbandingan agregasi
# ------------------------------------------------------------

aggregation_comparison_summary = (
    aggregation_comparison[
        [
            "Peringkat",
            "Metode Agregasi",
            "Rata-rata Mean CV Log Loss",
            "Rata-rata Mean CV ROC-AUC",
            "Total Waktu Tiga Konfigurasi Terpilih (s)",
            "Status"
        ]
    ]
    .copy()
)


print(
    "\nMetode agregasi terpilih:",
    selected_aggregation_method
)

print(
    "Rata-rata Mean CV Log Loss:",
    (
        f"{float(selected_aggregation_row['Rata-rata Mean CV Log Loss']):.6f}"
    )
)

print(
    "Rata-rata Mean CV ROC-AUC:",
    (
        f"{float(selected_aggregation_row['Rata-rata Mean CV ROC-AUC']):.6f}"
    )
)

print(
    "Total waktu tiga konfigurasi:",
    (
        f"{float(selected_aggregation_row['Total Waktu Tiga Konfigurasi Terpilih (s)']):.2f} detik"
    )
)


# ============================================================
# 3. MENGAMBIL KOMBINASI FINAL SETIAP MODEL
# ============================================================

selected_combinations = (
    best_topk_per_method_model
    .loc[
        best_topk_per_method_model[
            "Metode Agregasi"
        ] == selected_aggregation_method
    ]
    .copy()
)


# ------------------------------------------------------------
# Mengurutkan berdasarkan MODEL_ORDER
# ------------------------------------------------------------

selected_combinations[
    "Model"
] = pd.Categorical(
    selected_combinations[
        "Model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


selected_combinations = (
    selected_combinations
    .sort_values(
        "Model"
    )
    .reset_index(drop=True)
)


selected_combinations[
    "Model"
] = selected_combinations[
    "Model"
].astype(str)


# ------------------------------------------------------------
# Mengatur ulang nomor
# ------------------------------------------------------------

selected_combinations[
    "No"
] = range(
    1,
    len(
        selected_combinations
    ) + 1
)


selected_combinations = selected_combinations[
    [
        "No",
        "Skenario Pelabelan",
        "Metode Agregasi",
        "Model",
        "Top-K",
        "Fitur",
        "Best Mean CV Log Loss",
        "Mean CV ROC-AUC",
        "Selected Trial",
        "Jumlah Trial Selesai",
        "Waktu Trial Terpilih (s)",
        "Waktu Tuning Kumulatif (s)",
        "Hyperparameter Terbaik"
    ]
]


# ------------------------------------------------------------
# Validasi jumlah model
# ------------------------------------------------------------

if len(
    selected_combinations
) != len(
    MODEL_ORDER
):

    raise ValueError(
        "Jumlah konfigurasi model terpilih tidak sesuai."
    )


# ------------------------------------------------------------
# Tabel ringkas kombinasi final
# ------------------------------------------------------------

selected_combination_summary = (
    selected_combinations[
        [
            "No",
            "Metode Agregasi",
            "Model",
            "Top-K",
            "Best Mean CV Log Loss",
            "Mean CV ROC-AUC",
            "Waktu Tuning Kumulatif (s)"
        ]
    ]
    .copy()
)


# ============================================================
# 4. MENYIMPAN TOP-K, FITUR, DAN PARAMETER FINAL
# ============================================================

selected_features_by_model = {}

selected_k_by_model = {}

best_params_by_model = {}

selected_logloss_by_model = {}

selected_auc_by_model = {}


selected_method_order = (
    aggregation_rankings[
        selected_aggregation_method
    ]
    .sort_values(
        "Final Rank"
    )
    ["Feature"]
    .tolist()
)


for _, row in selected_combinations.iterrows():

    model_name = row[
        "Model"
    ]


    selected_k = int(
        row[
            "Top-K"
        ]
    )


    selected_features = list(
        selected_method_order[
            :selected_k
        ]
    )


    # --------------------------------------------------------
    # Validasi fitur dari hasil Blok 6
    # --------------------------------------------------------

    features_from_result = [
        feature.strip()
        for feature in str(
            row[
                "Fitur"
            ]
        ).split(",")
    ]


    if selected_features != features_from_result:

        raise ValueError(
            f"Daftar fitur {model_name} tidak konsisten "
            "antara ranking Blok 5 dan hasil Blok 6."
        )


    selected_features_by_model[
        model_name
    ] = selected_features


    selected_k_by_model[
        model_name
    ] = selected_k


    selected_logloss_by_model[
        model_name
    ] = float(
        row[
            "Best Mean CV Log Loss"
        ]
    )


    selected_auc_by_model[
        model_name
    ] = (
        None
        if pd.isna(
            row[
                "Mean CV ROC-AUC"
            ]
        )
        else float(
            row[
                "Mean CV ROC-AUC"
            ]
        )
    )


    parameter_value = row[
        "Hyperparameter Terbaik"
    ]


    if isinstance(
        parameter_value,
        dict
    ):

        parsed_params = (
            parameter_value.copy()
        )


    else:

        parsed_params = json.loads(
            parameter_value
        )


    best_params_by_model[
        model_name
    ] = parsed_params


# ------------------------------------------------------------
# Validasi seluruh model tersedia
# ------------------------------------------------------------

for model_name in MODEL_ORDER:

    if model_name not in selected_features_by_model:

        raise ValueError(
            f"Fitur final {model_name} belum tersedia."
        )


    if model_name not in selected_k_by_model:

        raise ValueError(
            f"Top-K final {model_name} belum tersedia."
        )


    if model_name not in best_params_by_model:

        raise ValueError(
            f"Hyperparameter final {model_name} belum tersedia."
        )


    if model_name not in selected_logloss_by_model:

        raise ValueError(
            f"Log Loss final {model_name} belum tersedia."
        )


# ============================================================
# 5. TABEL FITUR FINAL SETIAP MODEL
# ============================================================

selected_feature_rows = []


for model_name in MODEL_ORDER:

    selected_features = (
        selected_features_by_model[
            model_name
        ]
    )


    for feature_rank, feature in enumerate(
        selected_features,
        start=1
    ):

        selected_feature_rows.append({
            "Model":
                model_name,

            "Top-K":
                selected_k_by_model[
                    model_name
                ],

            "Peringkat Fitur":
                feature_rank,

            "Fitur":
                feature
        })


selected_feature_table = pd.DataFrame(
    selected_feature_rows
)


selected_feature_summary_rows = []


for model_name in MODEL_ORDER:

    selected_feature_summary_rows.append({
        "Model":
            model_name,

        "Top-K":
            selected_k_by_model[
                model_name
            ],

        "Fitur Terpilih":
            ", ".join(
                selected_features_by_model[
                    model_name
                ]
            )
    })


selected_feature_summary = pd.DataFrame(
    selected_feature_summary_rows
)


# ============================================================
# 6. TABEL HYPERPARAMETER FINAL
# ============================================================

final_hyperparameter_rows = []


for model_name in MODEL_ORDER:

    selected_result = (
        selected_combinations
        .loc[
            selected_combinations[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )


    hyperparameter_row = {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Metode Agregasi":
            selected_aggregation_method,

        "Model":
            model_name,

        "Top-K":
            selected_k_by_model[
                model_name
            ],

        "Fitur Terpilih":
            ", ".join(
                selected_features_by_model[
                    model_name
                ]
            ),

        "Best Mean CV Log Loss":
            selected_result[
                "Best Mean CV Log Loss"
            ],

        "Mean CV ROC-AUC":
            selected_result[
                "Mean CV ROC-AUC"
            ],

        "Waktu Tuning Kumulatif (s)":
            selected_result[
                "Waktu Tuning Kumulatif (s)"
            ]
    }


    for parameter in PARAMETER_ORDER[
        model_name
    ]:

        hyperparameter_row[
            parameter
        ] = (
            best_params_by_model[
                model_name
            ]
            .get(
                parameter
            )
        )


    final_hyperparameter_rows.append(
        hyperparameter_row
    )


final_hyperparameter_table = pd.DataFrame(
    final_hyperparameter_rows
)


# ------------------------------------------------------------
# Tabel hyperparameter vertikal
# ------------------------------------------------------------

final_hyperparameter_vertical_rows = []


for model_name in MODEL_ORDER:

    for parameter in PARAMETER_ORDER[
        model_name
    ]:

        final_hyperparameter_vertical_rows.append({
            "Model":
                model_name,

            "Top-K":
                selected_k_by_model[
                    model_name
                ],

            "Hyperparameter":
                parameter,

            "Nilai Terbaik":
                best_params_by_model[
                    model_name
                ].get(
                    parameter
                )
        })


final_hyperparameter_vertical = pd.DataFrame(
    final_hyperparameter_vertical_rows
)


# ------------------------------------------------------------
# Ringkasan hyperparameter final
# ------------------------------------------------------------

final_hyperparameter_summary = (
    selected_combinations[
        [
            "Skenario Pelabelan",
            "Metode Agregasi",
            "Model",
            "Top-K",
            "Fitur",
            "Best Mean CV Log Loss",
            "Mean CV ROC-AUC",
            "Waktu Tuning Kumulatif (s)",
            "Hyperparameter Terbaik"
        ]
    ]
    .copy()
    .rename(
        columns={
            "Fitur":
                "Fitur Terpilih"
        }
    )
)


# ============================================================
# 7. MENYIMPAN SELURUH HASIL BLOK 7
# ============================================================

tables_to_save = {
    "best_topk_per_method_model_logloss":
        best_topk_per_method_model,

    "best_topk_summary_logloss":
        best_topk_summary,

    "aggregation_comparison_logloss":
        aggregation_comparison,

    "aggregation_comparison_summary_logloss":
        aggregation_comparison_summary,

    "selected_combinations_logloss":
        selected_combinations,

    "selected_combination_summary_logloss":
        selected_combination_summary,

    "selected_feature_table_logloss":
        selected_feature_table,

    "selected_feature_summary_logloss":
        selected_feature_summary,

    "final_hyperparameter_table_logloss":
        final_hyperparameter_table,

    "final_hyperparameter_vertical_logloss":
        final_hyperparameter_vertical,

    "final_hyperparameter_summary_logloss":
        final_hyperparameter_summary
}


for file_name, dataframe in (
    tables_to_save.items()
):

    dataframe.to_csv(
        os.path.join(
            BLOCK7_DIR,
            f"{file_name}.csv"
        ),
        index=False
    )


    dataframe.to_pickle(
        os.path.join(
            BLOCK7_DIR,
            f"{file_name}.pkl"
        )
    )


# ============================================================
# MENYIMPAN DICTIONARY SEBAGAI JSON
# ============================================================

with open(
    os.path.join(
        BLOCK7_DIR,
        "selected_features_by_model_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        selected_features_by_model,
        file,
        ensure_ascii=False,
        indent=4
    )


with open(
    os.path.join(
        BLOCK7_DIR,
        "selected_k_by_model_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        selected_k_by_model,
        file,
        ensure_ascii=False,
        indent=4
    )


with open(
    os.path.join(
        BLOCK7_DIR,
        "best_params_by_model_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        best_params_by_model,
        file,
        ensure_ascii=False,
        indent=4
    )


with open(
    os.path.join(
        BLOCK7_DIR,
        "selected_metrics_by_model_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            model_name: {
                "Best Mean CV Log Loss":
                    selected_logloss_by_model[
                        model_name
                    ],

                "Mean CV ROC-AUC":
                    selected_auc_by_model[
                        model_name
                    ]
            }
            for model_name in MODEL_ORDER
        },
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MENYIMPAN SELURUH OBJEK UTAMA
# ============================================================

block7_results = {
    "selection_criterion":
        "Mean CV Log Loss terendah",

    "selected_aggregation_method":
        selected_aggregation_method,

    "selected_features_by_model":
        selected_features_by_model,

    "selected_k_by_model":
        selected_k_by_model,

    "best_params_by_model":
        best_params_by_model,

    "selected_logloss_by_model":
        selected_logloss_by_model,

    "selected_auc_by_model":
        selected_auc_by_model,

    "best_topk_per_method_model":
        best_topk_per_method_model,

    "aggregation_comparison":
        aggregation_comparison,

    "selected_combinations":
        selected_combinations,

    "final_hyperparameter_table":
        final_hyperparameter_table
}


joblib.dump(
    block7_results,
    os.path.join(
        BLOCK7_DIR,
        "block7_results_logloss.joblib"
    )
)


# ============================================================
# METADATA BLOK 7
# ============================================================

block7_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Kriteria Pemilihan Top-K":
        "Mean CV Log Loss terendah",

    "Tie-break Top-K":
        (
            "Waktu tuning kumulatif lebih rendah, "
            "kemudian jumlah fitur lebih sedikit"
        ),

    "Kriteria Pemilihan Metode Agregasi":
        "Rata-rata Mean CV Log Loss tiga model terendah",

    "Tie-break Metode Agregasi":
        (
            "Total waktu tiga konfigurasi "
            "Top-K terpilih lebih rendah"
        ),

    "Metrik Pendamping":
        "Mean CV ROC-AUC",

    "Metode Agregasi Terpilih":
        selected_aggregation_method,

    "Rata-rata Mean CV Log Loss Metode Terpilih":
        float(
            selected_aggregation_row[
                "Rata-rata Mean CV Log Loss"
            ]
        ),

    "Rata-rata Mean CV ROC-AUC Metode Terpilih":
        (
            None
            if pd.isna(
                selected_aggregation_row[
                    "Rata-rata Mean CV ROC-AUC"
                ]
            )
            else float(
                selected_aggregation_row[
                    "Rata-rata Mean CV ROC-AUC"
                ]
            )
        ),

    "Total Waktu Tiga Konfigurasi Terpilih (s)":
        float(
            selected_aggregation_row[
                "Total Waktu Tiga Konfigurasi Terpilih (s)"
            ]
        ),

    "Jumlah Model":
        int(
            len(
                MODEL_ORDER
            )
        ),

    "Top-K per Model":
        selected_k_by_model,

    "Best Mean CV Log Loss per Model":
        selected_logloss_by_model,

    "Mean CV ROC-AUC per Model":
        selected_auc_by_model
}


with open(
    os.path.join(
        BLOCK7_DIR,
        "block7_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block7_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 8. OUTPUT BLOK 7
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "TABEL LENGKAP TOP-K TERBAIK "
    "SETIAP METODE DAN MODEL"
)

print(
    "=" * 70
)


display_without_index(
    best_topk_per_method_model,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Trial Terpilih (s)":
            "{:.2f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL RINGKAS TOP-K TERBAIK "
    "SETIAP METODE DAN MODEL"
)

print(
    "=" * 70
)


display_without_index(
    best_topk_summary,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL PERBANDINGAN METODE AGREGASI"
)

print(
    "=" * 70
)


display_without_index(
    aggregation_comparison_summary,
    formats={
        "Rata-rata Mean CV Log Loss":
            "{:.6f}",

        "Rata-rata Mean CV ROC-AUC":
            "{:.6f}",

        "Total Waktu Tiga Konfigurasi Terpilih (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL KOMBINASI FINAL SETIAP MODEL"
)

print(
    "=" * 70
)


display_without_index(
    selected_combinations,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Trial Terpilih (s)":
            "{:.2f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL RINGKAS FITUR FINAL "
    "SETIAP MODEL"
)

print(
    "=" * 70
)


display_without_index(
    selected_feature_summary
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL LENGKAP HYPERPARAMETER FINAL"
)

print(
    "=" * 70
)


display_without_index(
    final_hyperparameter_table,
    formats={
        "Best Mean CV Log Loss":
            "{:.6f}",

        "Mean CV ROC-AUC":
            "{:.6f}",

        "Waktu Tuning Kumulatif (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "TABEL HYPERPARAMETER FINAL "
    "FORMAT VERTIKAL"
)

print(
    "=" * 70
)


display_without_index(
    final_hyperparameter_vertical
)


print(
    "\nKriteria pemilihan final:"
)

print(
    "- Top-K terbaik dipilih berdasarkan "
    "Mean CV Log Loss terendah"
)

print(
    "- Metode agregasi terbaik dipilih berdasarkan "
    "rata-rata Mean CV Log Loss tiga model terendah"
)

print(
    "- Waktu komputasi digunakan apabila "
    "nilai Log Loss sama"
)

print(
    "- Mean CV ROC-AUC disimpan sebagai "
    "metrik pendamping"
)


print(
    "\nSeluruh hasil Blok 7 "
    "berhasil disimpan di:"
)

print(
    BLOCK7_DIR
)

## TOP-K

In [ ]:
# ============================================================
# MEMUAT HASIL BLOK 7
# HASIL PEMILIHAN BERBASIS MEAN CV LOG LOSS
# ============================================================

import os
import joblib


# ------------------------------------------------------------
# Folder hasil Blok 7 berbasis Log Loss
# ------------------------------------------------------------

BLOCK7_DIR = os.path.join(
    "./artifacts",
    (
        f"final_combination_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)


# ------------------------------------------------------------
# Path file hasil utama Blok 7
# ------------------------------------------------------------

block7_results_path = os.path.join(
    BLOCK7_DIR,
    "block7_results_logloss.joblib"
)


# ------------------------------------------------------------
# Memeriksa keberadaan file
# ------------------------------------------------------------

if not os.path.exists(
    block7_results_path
):
    raise FileNotFoundError(
        "File hasil Blok 7 berbasis Log Loss "
        "tidak ditemukan pada path:\n"
        f"{block7_results_path}\n"
        "Jalankan Blok 7 terlebih dahulu."
    )


# ------------------------------------------------------------
# Memuat seluruh hasil Blok 7
# ------------------------------------------------------------

block7_results = joblib.load(
    block7_results_path
)


# ------------------------------------------------------------
# Validasi isi file
# ------------------------------------------------------------

required_block7_keys = [
    "selection_criterion",
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model",
    "best_params_by_model",
    "selected_logloss_by_model",
    "selected_auc_by_model",
    "best_topk_per_method_model",
    "aggregation_comparison",
    "selected_combinations",
    "final_hyperparameter_table"
]


missing_block7_keys = [
    key
    for key in required_block7_keys
    if key not in block7_results
]


if missing_block7_keys:
    raise KeyError(
        "Isi file hasil Blok 7 belum lengkap. "
        f"Key yang tidak ditemukan: {missing_block7_keys}"
    )


# ------------------------------------------------------------
# Mengambil objek utama
# ------------------------------------------------------------

selection_criterion = (
    block7_results[
        "selection_criterion"
    ]
)


selected_aggregation_method = (
    block7_results[
        "selected_aggregation_method"
    ]
)


selected_features_by_model = (
    block7_results[
        "selected_features_by_model"
    ]
)


selected_k_by_model = (
    block7_results[
        "selected_k_by_model"
    ]
)


best_params_by_model = (
    block7_results[
        "best_params_by_model"
    ]
)


selected_logloss_by_model = (
    block7_results[
        "selected_logloss_by_model"
    ]
)


selected_auc_by_model = (
    block7_results[
        "selected_auc_by_model"
    ]
)


best_topk_per_method_model = (
    block7_results[
        "best_topk_per_method_model"
    ]
)


aggregation_comparison = (
    block7_results[
        "aggregation_comparison"
    ]
)


selected_combinations = (
    block7_results[
        "selected_combinations"
    ]
)


final_hyperparameter_table = (
    block7_results[
        "final_hyperparameter_table"
    ]
)


# ------------------------------------------------------------
# Validasi seluruh model tersedia
# ------------------------------------------------------------

for model_name in MODEL_ORDER:

    if model_name not in selected_features_by_model:
        raise KeyError(
            f"Fitur terpilih untuk {model_name} "
            "tidak ditemukan."
        )

    if model_name not in selected_k_by_model:
        raise KeyError(
            f"Top-K terpilih untuk {model_name} "
            "tidak ditemukan."
        )

    if model_name not in best_params_by_model:
        raise KeyError(
            f"Hyperparameter terbaik untuk {model_name} "
            "tidak ditemukan."
        )

    if model_name not in selected_logloss_by_model:
        raise KeyError(
            f"Mean CV Log Loss untuk {model_name} "
            "tidak ditemukan."
        )

    if model_name not in selected_auc_by_model:
        raise KeyError(
            f"Mean CV ROC-AUC untuk {model_name} "
            "tidak ditemukan."
        )


# ------------------------------------------------------------
# Menampilkan ringkasan hasil yang dimuat
# ------------------------------------------------------------

print(
    "Hasil Blok 7 berhasil dimuat."
)

print(
    "Path:",
    block7_results_path
)

print(
    "Kriteria pemilihan:",
    selection_criterion
)

print(
    "Metode agregasi terpilih:",
    selected_aggregation_method
)


print(
    "\nTop-K terpilih setiap model:"
)

for model_name in MODEL_ORDER:

    print(
        f"- {model_name}: "
        f"Top-{selected_k_by_model[model_name]}"
    )


print(
    "\nMean CV Log Loss konfigurasi terpilih:"
)

for model_name in MODEL_ORDER:

    print(
        f"- {model_name}: "
        f"{selected_logloss_by_model[model_name]:.6f}"
    )


print(
    "\nMean CV ROC-AUC konfigurasi terpilih:"
)

for model_name in MODEL_ORDER:

    auc_value = selected_auc_by_model[
        model_name
    ]

    if auc_value is None:
        auc_text = "tidak tersedia"
    else:
        auc_text = f"{auc_value:.6f}"

    print(
        f"- {model_name}: {auc_text}"
    )

## UJI PATIENCE

In [ ]:
# ============================================================
# BLOK 7A — UJI DAN PEMILIHAN EARLY STOPPING PATIENCE
# UNTUK XGBOOST DAN CATBOOST
#
# LETAK:
# Setelah memuat hasil Blok 7
# dan sebelum menjalankan Blok 8.
#
# Pemilihan dilakukan menggunakan Stratified 5-Fold CV
# pada data train sehingga data validation dan test
# tetap tidak digunakan dalam pemilihan patience.
#
# Kriteria utama:
# Mean CV Log Loss terendah.
#
# Tie-break:
# 1. Waktu komputasi lebih singkat.
# 2. Nilai patience lebih kecil.
# ============================================================

import os
import gc
import copy
import json
import time
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# ============================================================
# KONFIGURASI UJI PATIENCE
# ============================================================

PATIENCE_CANDIDATES = [
    10,
    20,
    30,
    50,
    100
]

MAX_BOOSTING_ITERATIONS = 1000

RESET_DOWNSTREAM_CHECKPOINTS = True


# ============================================================
# FOLDER PENYIMPANAN HASIL UJI PATIENCE
# ============================================================

PATIENCE_DIR = os.path.join(
    "./artifacts",
    (
        f"patience_selection_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

PATIENCE_TABLE_DIR = os.path.join(
    PATIENCE_DIR,
    "tables"
)

PATIENCE_CHECKPOINT_DIR = os.path.join(
    PATIENCE_DIR,
    "checkpoints"
)

for directory in [
    PATIENCE_DIR,
    PATIENCE_TABLE_DIR,
    PATIENCE_CHECKPOINT_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil uji patience:",
    PATIENCE_DIR
)

print(
    "Kandidat patience:",
    PATIENCE_CANDIDATES
)

print(
    "Batas maksimum boosting:",
    MAX_BOOSTING_ITERATIONS
)


# ============================================================
# PEMERIKSAAN OBJEK WAJIB
# ============================================================

required_patience_objects = [
    "X_train_raw",
    "y_train",
    "cv_splits",
    "N_SPLITS",
    "RANDOM_STATE",
    "selected_features_by_model",
    "best_params_by_model",
    "prepare_fold_data"
]

missing_patience_objects = [
    object_name
    for object_name in required_patience_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]

if missing_patience_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk uji patience: "
        f"{missing_patience_objects}"
    )


if len(cv_splits) != N_SPLITS:
    raise ValueError(
        "Jumlah cv_splits tidak sesuai dengan N_SPLITS."
    )


for model_name in [
    "XGBoost",
    "CatBoost"
]:

    if model_name not in selected_features_by_model:
        raise KeyError(
            f"Fitur terpilih {model_name} tidak tersedia."
        )

    if model_name not in best_params_by_model:
        raise KeyError(
            f"Hyperparameter {model_name} tidak tersedia."
        )

    if not isinstance(
        best_params_by_model[model_name],
        dict
    ):
        raise TypeError(
            f"Hyperparameter {model_name} harus berupa dictionary."
        )


# ============================================================
# MENYIMPAN HYPERPARAMETER ASLI HASIL TUNING
# ============================================================

best_params_by_model_original = copy.deepcopy(
    best_params_by_model
)


# ============================================================
# MENYESUAIKAN BATAS MAKSIMUM BOOSTING
#
# Nilai hasil tuning asli tetap tersimpan pada
# best_params_by_model_original.
#
# Nilai minimal 1000 hanya menjadi batas maksimum operasional
# agar early stopping mempunyai ruang yang cukup.
# ============================================================

# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_original_n_estimators = int(
    best_params_by_model_original[
        "XGBoost"
    ].get(
        "n_estimators",
        MAX_BOOSTING_ITERATIONS
    )
)

best_params_by_model[
    "XGBoost"
][
    "n_estimators"
] = max(
    xgb_original_n_estimators,
    MAX_BOOSTING_ITERATIONS
)


# ------------------------------------------------------------
# CatBoost
#
# Kode dibuat kompatibel jika hasil tuning CatBoost memakai
# key "iterations" maupun "n_estimators".
# ------------------------------------------------------------

catboost_original_params = (
    best_params_by_model_original[
        "CatBoost"
    ]
)

if "iterations" in catboost_original_params:

    catboost_iteration_key = "iterations"

    catboost_original_iterations = int(
        catboost_original_params[
            "iterations"
        ]
    )

elif "n_estimators" in catboost_original_params:

    catboost_iteration_key = "n_estimators"

    catboost_original_iterations = int(
        catboost_original_params[
            "n_estimators"
        ]
    )

else:

    catboost_iteration_key = "iterations"

    catboost_original_iterations = (
        MAX_BOOSTING_ITERATIONS
    )


# Menghapus kemungkinan dua nama parameter yang bentrok
best_params_by_model[
    "CatBoost"
].pop(
    "iterations",
    None
)

best_params_by_model[
    "CatBoost"
].pop(
    "n_estimators",
    None
)


# Menggunakan iterations sebagai nama parameter final CatBoost
best_params_by_model[
    "CatBoost"
][
    "iterations"
] = max(
    catboost_original_iterations,
    MAX_BOOSTING_ITERATIONS
)


print(
    "\nPenyesuaian batas maksimum boosting:"
)

print(
    "- XGBoost hasil tuning:",
    xgb_original_n_estimators
)

print(
    "- XGBoost batas operasional:",
    best_params_by_model[
        "XGBoost"
    ][
        "n_estimators"
    ]
)

print(
    "- CatBoost hasil tuning:",
    catboost_original_iterations
)

print(
    "- CatBoost batas operasional:",
    best_params_by_model[
        "CatBoost"
    ][
        "iterations"
    ]
)


# ============================================================
# FUNGSI MEMBENTUK MODEL KHUSUS UJI PATIENCE
# ============================================================

def build_patience_test_model(
    model_name,
    params,
    patience
):
    """
    Membentuk XGBoost atau CatBoost untuk satu kandidat
    patience tertentu.
    """

    model_params = copy.deepcopy(
        params
    )


    if model_name == "XGBoost":

        model_params[
            "n_estimators"
        ] = max(
            int(
                model_params.get(
                    "n_estimators",
                    MAX_BOOSTING_ITERATIONS
                )
            ),
            MAX_BOOSTING_ITERATIONS
        )


        return XGBClassifier(
            **model_params,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            enable_categorical=True,
            early_stopping_rounds=int(
                patience
            ),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        )


    if model_name == "CatBoost":

        model_params.pop(
            "n_estimators",
            None
        )


        model_params[
            "iterations"
        ] = max(
            int(
                model_params.get(
                    "iterations",
                    MAX_BOOSTING_ITERATIONS
                )
            ),
            MAX_BOOSTING_ITERATIONS
        )


        return CatBoostClassifier(
            **model_params,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=RANDOM_STATE,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            od_type="Iter",
            od_wait=int(
                patience
            ),
            use_best_model=True
        )


    raise ValueError(
        f"Model tidak didukung untuk uji patience: "
        f"{model_name}"
    )


# ============================================================
# FUNGSI MENGAMBIL BEST ITERATION
# ============================================================

def get_model_best_iteration(
    model_name,
    model
):
    """
    Mengambil nomor iterasi terbaik dengan format laporan
    yang dimulai dari 1.
    """

    if model_name == "XGBoost":

        best_index = getattr(
            model,
            "best_iteration",
            None
        )


        if best_index is None:

            booster = model.get_booster()

            best_index = getattr(
                booster,
                "best_iteration",
                None
            )


        if best_index is None:

            return np.nan


        return int(
            best_index
        ) + 1


    if model_name == "CatBoost":

        best_index = model.get_best_iteration()


        if (
            best_index is None
            or int(best_index) < 0
        ):

            return np.nan


        return int(
            best_index
        ) + 1


    return np.nan


# ============================================================
# FUNGSI MENGAMBIL JUMLAH ITERASI YANG TERSEDIA
# ============================================================

def get_model_final_iteration(
    model_name,
    model
):
    """
    Mengambil jumlah iterasi yang benar-benar dijalankan.
    """

    if model_name == "XGBoost":

        evaluation_results = model.evals_result()


        if not evaluation_results:

            return np.nan


        first_dataset = next(
            iter(
                evaluation_results.values()
            )
        )


        if "logloss" not in first_dataset:

            return np.nan


        return int(
            len(
                first_dataset[
                    "logloss"
                ]
            )
        )


    if model_name == "CatBoost":

        evaluation_results = model.get_evals_result()


        if not evaluation_results:

            return np.nan


        validation_result = (
            evaluation_results.get(
                "validation",
                {}
            )
        )


        if "Logloss" not in validation_result:

            # Kompatibilitas jika key ditulis logloss
            logloss_values = validation_result.get(
                "logloss",
                []
            )

        else:

            logloss_values = validation_result[
                "Logloss"
            ]


        if len(logloss_values) == 0:

            return np.nan


        return int(
            len(
                logloss_values
            )
        )


    return np.nan


# ============================================================
# FUNGSI EVALUASI SATU MODEL DAN SATU PATIENCE
# ============================================================

def evaluate_one_patience_cv(
    model_name,
    patience
):
    """
    Mengevaluasi satu kandidat patience menggunakan
    Stratified K-Fold CV pada data train.
    """

    selected_features = list(
        selected_features_by_model[
            model_name
        ]
    )

    params = copy.deepcopy(
        best_params_by_model[
            model_name
        ]
    )

    fold_rows = []


    for fold_no, (
        train_pos,
        valid_pos
    ) in enumerate(
        cv_splits,
        start=1
    ):

        print(
            f"{model_name} | patience={patience} | "
            f"Fold {fold_no}/{N_SPLITS}"
        )


        X_fold_train_raw = (
            X_train_raw
            .iloc[
                train_pos
            ]
            .reset_index(
                drop=True
            )
        )


        X_fold_valid_raw = (
            X_train_raw
            .iloc[
                valid_pos
            ]
            .reset_index(
                drop=True
            )
        )


        y_fold_train = (
            y_train
            .iloc[
                train_pos
            ]
            .reset_index(
                drop=True
            )
            .astype(int)
        )


        y_fold_valid = (
            y_train
            .iloc[
                valid_pos
            ]
            .reset_index(
                drop=True
            )
            .astype(int)
        )


        (
            X_fold_train,
            X_fold_valid,
            selected_cat
        ) = prepare_fold_data(
            model_name=model_name,
            X_fold_train_raw=X_fold_train_raw,
            y_fold_train=y_fold_train,
            X_fold_valid_raw=X_fold_valid_raw,
            selected_features=selected_features
        )


        model = build_patience_test_model(
            model_name=model_name,
            params=params,
            patience=patience
        )


        fold_start_time = time.perf_counter()


        if model_name == "XGBoost":

            model.fit(
                X_fold_train,
                y_fold_train,
                eval_set=[
                    (
                        X_fold_valid,
                        y_fold_valid
                    )
                ],
                verbose=False
            )


        elif model_name == "CatBoost":

            model.fit(
                X_fold_train,
                y_fold_train,
                eval_set=(
                    X_fold_valid,
                    y_fold_valid
                ),
                cat_features=selected_cat,
                verbose=False
            )


        fold_elapsed_time = (
            time.perf_counter()
            - fold_start_time
        )


        y_fold_proba = (
            model
            .predict_proba(
                X_fold_valid
            )[:, 1]
        )


        fold_logloss = log_loss(
            y_fold_valid,
            y_fold_proba,
            labels=[
                0,
                1
            ]
        )


        best_iteration = get_model_best_iteration(
            model_name=model_name,
            model=model
        )


        final_iteration = get_model_final_iteration(
            model_name=model_name,
            model=model
        )


        if (
            np.isfinite(best_iteration)
            and np.isfinite(final_iteration)
        ):

            stopping_gap = int(
                final_iteration
                - best_iteration
            )

        else:

            stopping_gap = np.nan


        fold_rows.append({
            "Skenario Pelabelan":
                LABEL_DISPLAY_NAME,

            "Model":
                model_name,

            "Patience":
                int(
                    patience
                ),

            "Fold":
                int(
                    fold_no
                ),

            "Log Loss":
                float(
                    fold_logloss
                ),

            "Best Iteration":
                best_iteration,

            "Final Iteration":
                final_iteration,

            "Jeda Setelah Best Iteration":
                stopping_gap,

            "Waktu Komputasi (s)":
                float(
                    fold_elapsed_time
                )
        })


        del (
            model,
            X_fold_train_raw,
            X_fold_valid_raw,
            X_fold_train,
            X_fold_valid,
            y_fold_train,
            y_fold_valid,
            y_fold_proba
        )


        gc.collect()


    fold_table = pd.DataFrame(
        fold_rows
    )


    if len(
        fold_table
    ) != N_SPLITS:

        raise RuntimeError(
            f"Jumlah fold {model_name}, patience={patience} "
            "tidak sesuai."
        )


    numeric_validation_cols = [
        "Log Loss",
        "Waktu Komputasi (s)"
    ]


    if not np.isfinite(
        fold_table[
            numeric_validation_cols
        ].to_numpy(
            dtype=float
        )
    ).all():

        raise ValueError(
            f"Hasil {model_name}, patience={patience} "
            "mengandung NaN atau infinity."
        )


    return fold_table


# ============================================================
# MENJALANKAN UJI PATIENCE
# ============================================================

patience_fold_tables = {}

patience_summary_rows = []


for model_name in [
    "XGBoost",
    "CatBoost"
]:

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"UJI PATIENCE — {model_name}"
    )

    print(
        "=" * 70
    )


    for patience in PATIENCE_CANDIDATES:

        safe_model = safe_file_name(
            model_name
        )


        checkpoint_path = os.path.join(
            PATIENCE_CHECKPOINT_DIR,
            (
                f"patience_cv_"
                f"{safe_model}_"
                f"{int(patience)}.pkl"
            )
        )


        if os.path.exists(
            checkpoint_path
        ):

            print(
                f"Memuat checkpoint {model_name}, "
                f"patience={patience}"
            )


            fold_table = pd.read_pickle(
                checkpoint_path
            )


        else:

            fold_table = evaluate_one_patience_cv(
                model_name=model_name,
                patience=patience
            )


            fold_table.to_pickle(
                checkpoint_path
            )


        patience_fold_tables[
            (
                model_name,
                int(patience)
            )
        ] = fold_table


        patience_summary_rows.append({
            "Skenario Pelabelan":
                LABEL_DISPLAY_NAME,

            "Model":
                model_name,

            "Patience":
                int(
                    patience
                ),

            "Mean CV Log Loss":
                float(
                    fold_table[
                        "Log Loss"
                    ].mean()
                ),

            "Mean Best Iteration":
                float(
                    fold_table[
                        "Best Iteration"
                    ].mean()
                ),

            "Mean Final Iteration":
                float(
                    fold_table[
                        "Final Iteration"
                    ].mean()
                ),

            "Mean Jeda Setelah Best":
                float(
                    fold_table[
                        "Jeda Setelah Best Iteration"
                    ].mean()
                ),

            "Total Waktu Komputasi (s)":
                float(
                    fold_table[
                        "Waktu Komputasi (s)"
                    ].sum()
                )
        })


# ============================================================
# MEMBENTUK TABEL RINGKAS UJI PATIENCE
# ============================================================

patience_summary = pd.DataFrame(
    patience_summary_rows
)


patience_summary = (
    patience_summary
    .sort_values(
        [
            "Model",
            "Mean CV Log Loss",
            "Total Waktu Komputasi (s)",
            "Patience"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


patience_summary.insert(
    0,
    "No",
    range(
        1,
        len(
            patience_summary
        ) + 1
    )
)


# ============================================================
# MEMILIH PATIENCE TERBAIK SETIAP MODEL
# ============================================================

selected_patience_by_model = {}


for model_name in [
    "XGBoost",
    "CatBoost"
]:

    model_results = (
        patience_summary
        .loc[
            patience_summary[
                "Model"
            ] == model_name
        ]
        .sort_values(
            [
                "Mean CV Log Loss",
                "Total Waktu Komputasi (s)",
                "Patience"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


    selected_patience_by_model[
        model_name
    ] = int(
        model_results
        .iloc[0][
            "Patience"
        ]
    )


# Random Forest tidak menggunakan early stopping
selected_patience_by_model[
    "Random Forest"
] = None


# Variabel fallback untuk kompatibilitas dengan kode lama
EARLY_STOPPING_ROUNDS = int(
    selected_patience_by_model[
        "XGBoost"
    ]
)


# ============================================================
# TABEL PATIENCE TERPILIH
# ============================================================

selected_patience_rows = []


for model_name in [
    "XGBoost",
    "CatBoost"
]:

    selected_patience = (
        selected_patience_by_model[
            model_name
        ]
    )


    selected_row = (
        patience_summary
        .loc[
            (
                patience_summary[
                    "Model"
                ] == model_name
            )
            &
            (
                patience_summary[
                    "Patience"
                ] == selected_patience
            )
        ]
        .iloc[0]
    )


    selected_patience_rows.append({
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            model_name,

        "Patience Terpilih":
            int(
                selected_patience
            ),

        "Mean CV Log Loss":
            float(
                selected_row[
                    "Mean CV Log Loss"
                ]
            ),

        "Mean Best Iteration":
            float(
                selected_row[
                    "Mean Best Iteration"
                ]
            ),

        "Mean Final Iteration":
            float(
                selected_row[
                    "Mean Final Iteration"
                ]
            ),

        "Total Waktu Komputasi (s)":
            float(
                selected_row[
                    "Total Waktu Komputasi (s)"
                ]
            )
    })


selected_patience_table = pd.DataFrame(
    selected_patience_rows
)


selected_patience_table.insert(
    0,
    "No",
    range(
        1,
        len(
            selected_patience_table
        ) + 1
    )
)


# ============================================================
# MENDEFINISIKAN ULANG BUILD_MODEL
#
# Fungsi ini menggantikan fungsi lama pada notebook.
# Blok 8 dan Blok 9 tetap dapat memanggil fungsi yang sama.
# ============================================================

def build_model(
    model_name,
    params,
    use_early_stopping=True
):
    """
    Membentuk model menggunakan hyperparameter terpilih.

    XGBoost dan CatBoost menggunakan patience yang telah
    dipilih melalui cross-validation.
    """

    model_params = copy.deepcopy(
        params
    )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            **model_params,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )


    if model_name == "XGBoost":

        model_params[
            "n_estimators"
        ] = max(
            int(
                model_params.get(
                    "n_estimators",
                    MAX_BOOSTING_ITERATIONS
                )
            ),
            MAX_BOOSTING_ITERATIONS
        )


        xgb_extra_params = {}


        if use_early_stopping:

            xgb_extra_params[
                "early_stopping_rounds"
            ] = int(
                selected_patience_by_model[
                    "XGBoost"
                ]
            )


        return XGBClassifier(
            **model_params,
            **xgb_extra_params,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            enable_categorical=True,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        )


    if model_name == "CatBoost":

        # Mencegah benturan antara n_estimators dan iterations
        model_params.pop(
            "n_estimators",
            None
        )


        model_params[
            "iterations"
        ] = max(
            int(
                model_params.get(
                    "iterations",
                    MAX_BOOSTING_ITERATIONS
                )
            ),
            MAX_BOOSTING_ITERATIONS
        )


        catboost_extra_params = {}


        if use_early_stopping:

            catboost_extra_params.update({
                "od_type":
                    "Iter",

                "od_wait":
                    int(
                        selected_patience_by_model[
                            "CatBoost"
                        ]
                    ),

                "use_best_model":
                    True
            })


        return CatBoostClassifier(
            **model_params,
            **catboost_extra_params,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=RANDOM_STATE,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )


    raise ValueError(
        f"Model tidak dikenali: {model_name}"
    )


# ============================================================
# MENDEFINISIKAN ULANG FIT_MODEL
#
# Signature dibuat sama dengan pemanggilan pada Blok 8.
# ============================================================

def fit_model(
    model_name,
    model,
    X_train_data,
    y_train_data,
    X_valid_data=None,
    y_valid_data=None,
    cat_features=None,
    include_train_in_eval=False,
    use_early_stopping=True
):
    """
    Melatih model dengan struktur pemanggilan yang kompatibel
    dengan Blok 8 dan Blok 9.
    """

    if model_name == "Random Forest":

        model.fit(
            X_train_data,
            y_train_data
        )


    elif model_name == "XGBoost":

        if (
            use_early_stopping
            and (
                X_valid_data is None
                or y_valid_data is None
            )
        ):

            raise ValueError(
                "XGBoost dengan early stopping "
                "membutuhkan data validation."
            )


        if X_valid_data is None:

            model.fit(
                X_train_data,
                y_train_data,
                verbose=False
            )


        else:

            if include_train_in_eval:

                eval_set = [
                    (
                        X_train_data,
                        y_train_data
                    ),
                    (
                        X_valid_data,
                        y_valid_data
                    )
                ]

            else:

                eval_set = [
                    (
                        X_valid_data,
                        y_valid_data
                    )
                ]


            model.fit(
                X_train_data,
                y_train_data,
                eval_set=eval_set,
                verbose=False
            )


    elif model_name == "CatBoost":

        if (
            use_early_stopping
            and (
                X_valid_data is None
                or y_valid_data is None
            )
        ):

            raise ValueError(
                "CatBoost dengan early stopping "
                "membutuhkan data validation."
            )


        fit_params = {}


        if cat_features is not None:

            fit_params[
                "cat_features"
            ] = cat_features


        if X_valid_data is not None:

            fit_params[
                "eval_set"
            ] = (
                X_valid_data,
                y_valid_data
            )


        model.fit(
            X_train_data,
            y_train_data,
            verbose=False,
            **fit_params
        )


    else:

        raise ValueError(
            f"Model tidak dikenali: {model_name}"
        )


    return model


# ============================================================
# MENYIMPAN HASIL UJI PATIENCE
# ============================================================

patience_summary.to_csv(
    os.path.join(
        PATIENCE_TABLE_DIR,
        "patience_cv_summary_logloss.csv"
    ),
    index=False
)


patience_summary.to_pickle(
    os.path.join(
        PATIENCE_TABLE_DIR,
        "patience_cv_summary_logloss.pkl"
    )
)


selected_patience_table.to_csv(
    os.path.join(
        PATIENCE_TABLE_DIR,
        "selected_patience_logloss.csv"
    ),
    index=False
)


selected_patience_table.to_pickle(
    os.path.join(
        PATIENCE_TABLE_DIR,
        "selected_patience_logloss.pkl"
    )
)


patience_results = {
    "Patience Candidates":
        PATIENCE_CANDIDATES,

    "Maximum Boosting Iterations":
        MAX_BOOSTING_ITERATIONS,

    "Best Parameters Original":
        best_params_by_model_original,

    "Best Parameters Operational":
        best_params_by_model,

    "Patience Fold Tables":
        patience_fold_tables,

    "Patience Summary":
        patience_summary,

    "Selected Patience by Model":
        selected_patience_by_model,

    "Selected Patience Table":
        selected_patience_table
}


joblib.dump(
    patience_results,
    os.path.join(
        PATIENCE_DIR,
        "patience_selection_results_logloss.joblib"
    )
)


patience_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Kandidat Patience":
        PATIENCE_CANDIDATES,

    "Jumlah Fold":
        int(
            N_SPLITS
        ),

    "Kriteria Utama":
        "Mean CV Log Loss terendah",

    "Tie-Break":
        [
            "Total waktu komputasi lebih singkat",
            "Nilai patience lebih kecil"
        ],

    "Batas Maksimum Boosting":
        int(
            MAX_BOOSTING_ITERATIONS
        ),

    "Menggunakan Data Validation Held-Out":
        False,

    "Menggunakan Data Test":
        False,

    "Patience Terpilih":
        {
            "XGBoost":
                int(
                    selected_patience_by_model[
                        "XGBoost"
                    ]
                ),

            "CatBoost":
                int(
                    selected_patience_by_model[
                        "CatBoost"
                    ]
                )
        }
}


with open(
    os.path.join(
        PATIENCE_DIR,
        "patience_selection_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        patience_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MENGHAPUS CHECKPOINT LAMA BLOK 8 SAMPAI BLOK 13
#
# Diperlukan agar kode setelahnya tidak memuat model yang
# masih menggunakan patience atau batas boosting lama.
# ============================================================

if RESET_DOWNSTREAM_CHECKPOINTS:

    downstream_directories = [
        os.path.join(
            "./artifacts",
            (
                f"final_cv_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        ),

        os.path.join(
            "./artifacts",
            (
                f"validation_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        ),

        os.path.join(
            "./artifacts",
            (
                f"logloss_analysis_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        ),

        os.path.join(
            "./artifacts",
            (
                f"test_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        ),

        os.path.join(
            "./artifacts",
            (
                f"final_results_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        ),

        os.path.join(
            "./artifacts",
            (
                f"error_analysis_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )
    ]


    print(
        "\nMenghapus checkpoint lama Blok 8–13:"
    )


    for directory in downstream_directories:

        if os.path.exists(
            directory
        ):

            shutil.rmtree(
                directory
            )

            print(
                "- Dihapus:",
                directory
            )

        else:

            print(
                "- Tidak ditemukan:",
                directory
            )


# ============================================================
# OUTPUT HASIL
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "HASIL UJI SELURUH KANDIDAT PATIENCE"
)

print(
    "=" * 70
)


display_without_index(
    patience_summary,
    formats={
        "Mean CV Log Loss":
            "{:.6f}",

        "Mean Best Iteration":
            "{:.2f}",

        "Mean Final Iteration":
            "{:.2f}",

        "Mean Jeda Setelah Best":
            "{:.2f}",

        "Total Waktu Komputasi (s)":
            "{:.2f}"
    }
)


print(
    "\n"
    + "=" * 70
)

print(
    "PATIENCE TERPILIH"
)

print(
    "=" * 70
)


display_without_index(
    selected_patience_table,
    formats={
        "Mean CV Log Loss":
            "{:.6f}",

        "Mean Best Iteration":
            "{:.2f}",

        "Mean Final Iteration":
            "{:.2f}",

        "Total Waktu Komputasi (s)":
            "{:.2f}"
    }
)


print(
    "\nPatience XGBoost:",
    selected_patience_by_model[
        "XGBoost"
    ]
)

print(
    "Patience CatBoost:",
    selected_patience_by_model[
        "CatBoost"
    ]
)

print(
    "\nHyperparameter hasil tuning asli tetap tersimpan pada:"
)

print(
    "best_params_by_model_original"
)

print(
    "\nHyperparameter operasional yang dipakai Blok 8:"
)

print(
    "best_params_by_model"
)

print(
    "\nFungsi build_model() dan fit_model() "
    "telah diperbarui."
)

print(
    "Blok 8 sampai akhir dapat dijalankan tanpa perubahan."
)

print(
    "\nSeluruh hasil uji patience tersimpan di:"
)

print(
    PATIENCE_DIR
)

## HASIL PATIENCE

In [ ]:
import os
import shutil

LABEL_NAME_SAFE = safe_file_name(LABEL_DISPLAY_NAME)

folders_to_delete = [
    os.path.join(
        "./artifacts",
        f"final_cv_logloss_{LABEL_NAME_SAFE}"
    ),
    os.path.join(
        "./artifacts",
        f"validation_evaluation_logloss_{LABEL_NAME_SAFE}"
    ),
    os.path.join(
        "./artifacts",
        f"logloss_analysis_logloss_{LABEL_NAME_SAFE}"
    ),
    os.path.join(
        "./artifacts",
        f"test_evaluation_logloss_{LABEL_NAME_SAFE}"
    ),
    os.path.join(
        "./artifacts",
        f"final_results_logloss_{LABEL_NAME_SAFE}"
    ),
    os.path.join(
        "./artifacts",
        f"error_analysis_logloss_{LABEL_NAME_SAFE}"
    )
]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print("Dihapus:", folder)
    else:
        print("Tidak ditemukan:", folder)

In [ ]:
# ============================================================
# BLOK 8 — CROSS-VALIDATION DETAIL KONFIGURASI TERPILIH
# HASIL OPTIMASI BERBASIS MEAN CV LOG LOSS
# DENGAN PENYIMPANAN DAN RESUME
#
# Tidak melakukan tuning ulang.
# Hanya mengevaluasi Top-K dan hyperparameter terpilih
# menggunakan Stratified 5-Fold Cross-Validation
# pada data train.
#
# Metrik evaluasi:
# 1. Accuracy
# 2. Precision
# 3. Recall
# 4. F1-Score
# 5. ROC-AUC
#
# Log Loss tetap dihitung sebagai metrik tambahan.
# ============================================================

import os
import json
import joblib


# ============================================================
# FOLDER PENYIMPANAN BLOK 8
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC
# agar checkpoint lama tidak termuat kembali.
# ============================================================

BLOCK8_DIR = os.path.join(
    "./artifacts",
    (
        f"final_cv_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

CV_TABLE_DIR = os.path.join(
    BLOCK8_DIR,
    "fold_tables"
)

os.makedirs(
    BLOCK8_DIR,
    exist_ok=True
)

os.makedirs(
    CV_TABLE_DIR,
    exist_ok=True
)


print(
    "Folder hasil Blok 8:",
    BLOCK8_DIR
)

print(
    "Dasar konfigurasi:",
    "Mean CV Log Loss terendah"
)

print(
    "Evaluasi:",
    "Stratified 5-Fold Cross-Validation"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block8_objects = [
    "X_train_raw",
    "y_train",
    "MODEL_ORDER",
    "LABEL_DISPLAY_NAME",
    "N_SPLITS",
    "cv_splits",
    "THRESHOLD"
]


missing_block8_objects = [
    object_name
    for object_name in required_block8_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block8_objects:

    raise NameError(
        "Objek berikut belum tersedia untuk Blok 8: "
        f"{missing_block8_objects}"
    )


# ============================================================
# MEMUAT HASIL BLOK 7 JIKA VARIABEL BELUM TERSEDIA
# ============================================================

required_block7_objects = [
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model",
    "best_params_by_model"
]


missing_block7_objects = [
    object_name
    for object_name in required_block7_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block7_objects:

    print(
        "Objek hasil Blok 7 belum tersedia. "
        "Memuat hasil Blok 7 berbasis Log Loss."
    )


    if "BLOCK7_DIR" not in globals():

        BLOCK7_DIR = os.path.join(
            "./artifacts",
            (
                f"final_combination_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block7_result_path = os.path.join(
        BLOCK7_DIR,
        "block7_results_logloss.joblib"
    )


    if not os.path.exists(
        block7_result_path
    ):

        raise FileNotFoundError(
            "Hasil Blok 7 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block7_result_path}\n"
            "Jalankan Blok 7 terlebih dahulu."
        )


    block7_results = joblib.load(
        block7_result_path
    )


    required_block7_keys = [
        "selected_aggregation_method",
        "selected_features_by_model",
        "selected_k_by_model",
        "best_params_by_model"
    ]


    missing_block7_keys = [
        key
        for key in required_block7_keys
        if key not in block7_results
    ]


    if missing_block7_keys:

        raise KeyError(
            "Isi hasil Blok 7 belum lengkap. "
            f"Key yang tidak ditemukan: {missing_block7_keys}"
        )


    selected_aggregation_method = (
        block7_results[
            "selected_aggregation_method"
        ]
    )


    selected_features_by_model = (
        block7_results[
            "selected_features_by_model"
        ]
    )


    selected_k_by_model = (
        block7_results[
            "selected_k_by_model"
        ]
    )


    best_params_by_model = (
        block7_results[
            "best_params_by_model"
        ]
    )


    selected_logloss_by_model = (
        block7_results.get(
            "selected_logloss_by_model",
            {}
        )
    )


    selected_auc_by_model = (
        block7_results.get(
            "selected_auc_by_model",
            {}
        )
    )


    print(
        "Hasil Blok 7 berhasil dimuat."
    )


# ============================================================
# VALIDASI KONFIGURASI TERPILIH
# ============================================================

for model_name in MODEL_ORDER:

    if model_name not in selected_features_by_model:

        raise ValueError(
            f"Fitur terpilih {model_name} belum tersedia."
        )


    if model_name not in selected_k_by_model:

        raise ValueError(
            f"Top-K terpilih {model_name} belum tersedia."
        )


    if model_name not in best_params_by_model:

        raise ValueError(
            f"Hyperparameter terpilih {model_name} "
            "belum tersedia."
        )


    selected_features = (
        selected_features_by_model[
            model_name
        ]
    )


    selected_k = int(
        selected_k_by_model[
            model_name
        ]
    )


    if len(
        selected_features
    ) != selected_k:

        raise ValueError(
            f"Jumlah fitur terpilih {model_name} "
            "tidak sesuai dengan nilai Top-K."
        )


    unknown_features = [
        feature
        for feature in selected_features
        if feature not in features_before_fs
    ]


    if unknown_features:

        raise ValueError(
            f"Fitur {model_name} tidak dikenali: "
            f"{unknown_features}"
        )


    if not isinstance(
        best_params_by_model[
            model_name
        ],
        dict
    ):

        raise TypeError(
            f"Hyperparameter {model_name} "
            "harus berupa dictionary."
        )


# ============================================================
# FUNGSI CROSS-VALIDATION KONFIGURASI TERPILIH
# ============================================================

def evaluate_final_combination_cv(
    model_name,
    selected_features,
    params
):
    """
    Mengevaluasi satu konfigurasi model menggunakan
    Stratified K-Fold Cross-Validation.

    Hyperparameter tidak dituning ulang.
    Setiap fold menghasilkan lima metrik klasifikasi
    dan Log Loss sebagai metrik tambahan.
    """

    rows = []


    for fold_no, (
        train_pos,
        valid_pos
    ) in enumerate(
        cv_splits,
        start=1
    ):

        print(
            f"{model_name} — Fold "
            f"{fold_no}/{N_SPLITS}"
        )


        # ----------------------------------------------------
        # Membentuk training dan validation fold
        # ----------------------------------------------------

        X_fold_train_raw = (
            X_train_raw
            .iloc[
                train_pos
            ]
            .reset_index(
                drop=True
            )
        )


        X_fold_valid_raw = (
            X_train_raw
            .iloc[
                valid_pos
            ]
            .reset_index(
                drop=True
            )
        )


        y_fold_train = (
            y_train
            .iloc[
                train_pos
            ]
            .reset_index(
                drop=True
            )
            .astype(int)
        )


        y_fold_valid = (
            y_train
            .iloc[
                valid_pos
            ]
            .reset_index(
                drop=True
            )
            .astype(int)
        )


        # ----------------------------------------------------
        # Preprocessing dilakukan secara terpisah per fold
        # ----------------------------------------------------

        (
            X_fold_train,
            X_fold_valid,
            selected_cat
        ) = prepare_fold_data(
            model_name=model_name,
            X_fold_train_raw=X_fold_train_raw,
            y_fold_train=y_fold_train,
            X_fold_valid_raw=X_fold_valid_raw,
            selected_features=selected_features
        )


        # ----------------------------------------------------
        # Membentuk model menggunakan hyperparameter terpilih
        # ----------------------------------------------------

        model = build_model(
            model_name=model_name,
            params=params.copy(),
            use_early_stopping=True
        )


        # ----------------------------------------------------
        # Melatih model pada training fold
        # ----------------------------------------------------

        model = fit_model(
            model_name=model_name,
            model=model,
            X_train_data=X_fold_train,
            y_train_data=y_fold_train,
            X_valid_data=X_fold_valid,
            y_valid_data=y_fold_valid,
            cat_features=selected_cat,
            include_train_in_eval=False,
            use_early_stopping=True
        )


        # ----------------------------------------------------
        # Menghasilkan probabilitas validation fold
        # ----------------------------------------------------

        y_proba = (
            model
            .predict_proba(
                X_fold_valid
            )[:, 1]
        )


        # ----------------------------------------------------
        # Menghitung metrik
        # ----------------------------------------------------

        metrics, _ = calculate_metrics(
            y_true=y_fold_valid,
            y_proba=y_proba,
            threshold=THRESHOLD
        )


        rows.append({
            "Fold":
                int(fold_no),

            "Accuracy":
                float(
                    metrics[
                        "Accuracy"
                    ]
                ),

            "Precision":
                float(
                    metrics[
                        "Precision"
                    ]
                ),

            "Recall":
                float(
                    metrics[
                        "Recall"
                    ]
                ),

            "F1-Score":
                float(
                    metrics[
                        "F1-Score"
                    ]
                ),

            "ROC-AUC":
                float(
                    metrics[
                        "ROC-AUC"
                    ]
                ),

            "Log Loss":
                float(
                    metrics[
                        "Log Loss"
                    ]
                )
        })


        del (
            model,
            X_fold_train,
            X_fold_valid,
            X_fold_train_raw,
            X_fold_valid_raw,
            y_fold_train,
            y_fold_valid,
            y_proba
        )


        gc.collect()


    # --------------------------------------------------------
    # Membentuk tabel hasil setiap fold
    # --------------------------------------------------------

    fold_result = pd.DataFrame(
        rows
    )


    if len(
        fold_result
    ) != N_SPLITS:

        raise RuntimeError(
            f"Jumlah fold {model_name} tidak sesuai. "
            f"Ditemukan={len(fold_result)}, "
            f"seharusnya={N_SPLITS}."
        )


    metric_cols = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "Log Loss"
    ]


    if fold_result[
        metric_cols
    ].isna().any().any():

        raise ValueError(
            f"Hasil cross-validation {model_name} "
            "mengandung nilai kosong."
        )


    if not np.isfinite(
        fold_result[
            metric_cols
        ].to_numpy(
            dtype=float
        )
    ).all():

        raise ValueError(
            f"Hasil cross-validation {model_name} "
            "mengandung NaN atau infinity."
        )


    # --------------------------------------------------------
    # Menambahkan rata-rata lima fold
    # --------------------------------------------------------

    mean_row = {
        "Fold":
            "Mean"
    }


    for metric in metric_cols:

        mean_row[
            metric
        ] = float(
            fold_result[
                metric
            ].mean()
        )


    fold_result[
        "Fold"
    ] = fold_result[
        "Fold"
    ].astype(str)


    fold_result = pd.concat(
        [
            fold_result,
            pd.DataFrame([
                mean_row
            ])
        ],
        ignore_index=True
    )


    return fold_result


# ============================================================
# CROSS-VALIDATION SETIAP MODEL
# ============================================================

cv_fold_metric_tables = {}

cv_summary_rows = []


for model_name in MODEL_ORDER:

    print(
        "\n"
        + "=" * 70
    )


    print(
        "CROSS-VALIDATION KONFIGURASI TERPILIH "
        f"— {model_name}"
    )


    print(
        "=" * 70
    )


    safe_model = safe_file_name(
        model_name
    )


    fold_pickle_path = os.path.join(
        CV_TABLE_DIR,
        (
            f"cv_fold_logloss_"
            f"{safe_model}.pkl"
        )
    )


    fold_csv_path = os.path.join(
        CV_TABLE_DIR,
        (
            f"cv_fold_logloss_"
            f"{safe_model}.csv"
        )
    )


    # --------------------------------------------------------
    # Memuat hasil cross-validation jika tersedia
    # --------------------------------------------------------

    if os.path.exists(
        fold_pickle_path
    ):

        print(
            "Hasil cross-validation sudah tersedia. "
            "Memuat dari checkpoint."
        )


        fold_table = pd.read_pickle(
            fold_pickle_path
        )


    # --------------------------------------------------------
    # Menghitung cross-validation jika belum tersedia
    # --------------------------------------------------------

    else:

        fold_table = evaluate_final_combination_cv(
            model_name=model_name,
            selected_features=(
                selected_features_by_model[
                    model_name
                ]
            ),
            params=(
                best_params_by_model[
                    model_name
                ]
            )
        )


        fold_table.to_pickle(
            fold_pickle_path
        )


        fold_table.to_csv(
            fold_csv_path,
            index=False
        )


        print(
            "Checkpoint cross-validation tersimpan:",
            fold_pickle_path
        )


    # --------------------------------------------------------
    # Validasi struktur tabel fold
    # --------------------------------------------------------

    required_fold_columns = [
        "Fold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "Log Loss"
    ]


    missing_fold_columns = [
        column
        for column in required_fold_columns
        if column not in fold_table.columns
    ]


    if missing_fold_columns:

        raise ValueError(
            f"Kolom cross-validation {model_name} "
            f"belum lengkap: {missing_fold_columns}"
        )


    expected_fold_labels = (
        [
            str(fold_no)
            for fold_no in range(
                1,
                N_SPLITS + 1
            )
        ]
        + [
            "Mean"
        ]
    )


    actual_fold_labels = (
        fold_table[
            "Fold"
        ]
        .astype(str)
        .tolist()
    )


    if actual_fold_labels != expected_fold_labels:

        raise ValueError(
            f"Isi tabel cross-validation "
            f"{model_name} tidak sesuai.\n"
            f"Ditemukan: {actual_fold_labels}\n"
            f"Seharusnya: {expected_fold_labels}"
        )


    cv_fold_metric_tables[
        model_name
    ] = fold_table


    # --------------------------------------------------------
    # Menampilkan tabel hasil setiap model
    # --------------------------------------------------------

    print(
        "\nTABEL CROSS-VALIDATION "
        f"— {model_name}"
    )


    display_without_index(
        fold_table,
        formats={
            "Accuracy":
                "{:.4f}",

            "Precision":
                "{:.4f}",

            "Recall":
                "{:.4f}",

            "F1-Score":
                "{:.4f}",

            "ROC-AUC":
                "{:.4f}",

            "Log Loss":
                "{:.6f}"
        }
    )


    # --------------------------------------------------------
    # Mengambil nilai rata-rata cross-validation
    # --------------------------------------------------------

    mean_values = (
        fold_table
        .loc[
            fold_table[
                "Fold"
            ].astype(str) == "Mean"
        ]
        .iloc[0]
    )


    cv_summary_rows.append({
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            model_name,

        "Metode Agregasi":
            selected_aggregation_method,

        "Top-K":
            int(
                selected_k_by_model[
                    model_name
                ]
            ),

        "Fitur Terpilih":
            ", ".join(
                selected_features_by_model[
                    model_name
                ]
            ),

        "Mean Accuracy":
            float(
                mean_values[
                    "Accuracy"
                ]
            ),

        "Mean Precision":
            float(
                mean_values[
                    "Precision"
                ]
            ),

        "Mean Recall":
            float(
                mean_values[
                    "Recall"
                ]
            ),

        "Mean F1-Score":
            float(
                mean_values[
                    "F1-Score"
                ]
            ),

        "Mean ROC-AUC":
            float(
                mean_values[
                    "ROC-AUC"
                ]
            ),

        "Mean Log Loss":
            float(
                mean_values[
                    "Log Loss"
                ]
            )
    })


# ============================================================
# TABEL RINGKAS CROSS-VALIDATION SELURUH MODEL
# ============================================================

cv_fold_summary_table = pd.DataFrame(
    cv_summary_rows
)


# ------------------------------------------------------------
# Mengatur urutan model
# ------------------------------------------------------------

cv_fold_summary_table[
    "Model"
] = pd.Categorical(
    cv_fold_summary_table[
        "Model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


cv_fold_summary_table = (
    cv_fold_summary_table
    .sort_values(
        "Model"
    )
    .reset_index(
        drop=True
    )
)


cv_fold_summary_table[
    "Model"
] = cv_fold_summary_table[
    "Model"
].astype(str)


cv_fold_summary_table.insert(
    0,
    "No",
    range(
        1,
        len(
            cv_fold_summary_table
        ) + 1
    )
)


# ------------------------------------------------------------
# Versi ringkas untuk Bab 4
#
# Lima metrik klasifikasi dan Log Loss tambahan.
# ------------------------------------------------------------

cv_fold_summary_report = (
    cv_fold_summary_table[
        [
            "No",
            "Model",
            "Top-K",
            "Mean Accuracy",
            "Mean Precision",
            "Mean Recall",
            "Mean F1-Score",
            "Mean ROC-AUC",
            "Mean Log Loss"
        ]
    ]
    .copy()
)


# ============================================================
# MENYIMPAN HASIL BLOK 8
# ============================================================

cv_fold_summary_table.to_csv(
    os.path.join(
        BLOCK8_DIR,
        "cv_logloss_summary_full.csv"
    ),
    index=False
)


cv_fold_summary_table.to_pickle(
    os.path.join(
        BLOCK8_DIR,
        "cv_logloss_summary_full.pkl"
    )
)


cv_fold_summary_report.to_csv(
    os.path.join(
        BLOCK8_DIR,
        "cv_logloss_summary_report.csv"
    ),
    index=False
)


cv_fold_summary_report.to_pickle(
    os.path.join(
        BLOCK8_DIR,
        "cv_logloss_summary_report.pkl"
    )
)


block8_results = {
    "selection_criterion":
        "Mean CV Log Loss terendah",

    "selected_aggregation_method":
        selected_aggregation_method,

    "selected_features_by_model":
        selected_features_by_model,

    "selected_k_by_model":
        selected_k_by_model,

    "best_params_by_model":
        best_params_by_model,

    "cv_fold_metric_tables":
        cv_fold_metric_tables,

    "cv_fold_summary_table":
        cv_fold_summary_table,

    "cv_fold_summary_report":
        cv_fold_summary_report
}


joblib.dump(
    block8_results,
    os.path.join(
        BLOCK8_DIR,
        "block8_results_logloss.joblib"
    )
)


# ============================================================
# METADATA BLOK 8
# ============================================================

block8_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Dasar Pemilihan Konfigurasi":
        "Mean CV Log Loss terendah",

    "Metode Agregasi":
        selected_aggregation_method,

    "Jumlah Fold":
        int(
            N_SPLITS
        ),

    "Threshold":
        float(
            THRESHOLD
        ),

    "Metrik Klasifikasi":
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ],

    "Metrik Tambahan":
        "Log Loss",

    "Model":
        list(
            MODEL_ORDER
        ),

    "Top-K per Model":
        {
            model_name:
                int(
                    selected_k_by_model[
                        model_name
                    ]
                )
            for model_name in MODEL_ORDER
        },

    "Fitur per Model":
        {
            model_name:
                list(
                    selected_features_by_model[
                        model_name
                    ]
                )
            for model_name in MODEL_ORDER
        }
}


with open(
    os.path.join(
        BLOCK8_DIR,
        "block8_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block8_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT AKHIR BLOK 8
# ============================================================

print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKASAN CROSS-VALIDATION "
    "KONFIGURASI TERPILIH"
)


print(
    "=" * 70
)


display_without_index(
    cv_fold_summary_table,
    formats={
        "Mean Accuracy":
            "{:.4f}",

        "Mean Precision":
            "{:.4f}",

        "Mean Recall":
            "{:.4f}",

        "Mean F1-Score":
            "{:.4f}",

        "Mean ROC-AUC":
            "{:.4f}",

        "Mean Log Loss":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL CROSS-VALIDATION "
    "UNTUK BAB 4"
)


print(
    "=" * 70
)


display_without_index(
    cv_fold_summary_report,
    formats={
        "Mean Accuracy":
            "{:.4f}",

        "Mean Precision":
            "{:.4f}",

        "Mean Recall":
            "{:.4f}",

        "Mean F1-Score":
            "{:.4f}",

        "Mean ROC-AUC":
            "{:.4f}",

        "Mean Log Loss":
            "{:.6f}"
    }
)


print(
    "\nCross-validation tidak melakukan tuning ulang."
)


print(
    "Konfigurasi yang dievaluasi merupakan hasil "
    "pemilihan berdasarkan Mean CV Log Loss terendah."
)


print(
    "\nSeluruh hasil cross-validation "
    "berhasil disimpan di:"
)


print(
    BLOCK8_DIR
)

## HASIL CROSVAL

In [ ]:
# ============================================================
# BLOK 9 — MEMBENTUK DATA FINAL DAN EVALUASI VALIDATION
# HASIL OPTIMASI BERBASIS MEAN CV LOG LOSS
# DENGAN PENYIMPANAN DAN RESUME
#
# Data validation digunakan untuk:
# 1. early stopping XGBoost dan CatBoost;
# 2. evaluasi lima metrik utama;
# 3. pemilihan model terbaik pada setiap label.
#
# Log Loss tetap dihitung sebagai metrik tambahan.
# ============================================================

import os
import json
import joblib


# ============================================================
# FOLDER PENYIMPANAN BLOK 9
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC
# agar checkpoint lama tidak termuat kembali.
# ============================================================

BLOCK9_DIR = os.path.join(
    "./artifacts",
    (
        f"validation_evaluation_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

MODEL_DIR = os.path.join(
    BLOCK9_DIR,
    "models"
)

PREDICTION_DIR = os.path.join(
    BLOCK9_DIR,
    "predictions"
)

LOGLOSS_DIR = os.path.join(
    BLOCK9_DIR,
    "logloss"
)

TABLE_DIR = os.path.join(
    BLOCK9_DIR,
    "tables"
)

for directory in [
    BLOCK9_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    LOGLOSS_DIR,
    TABLE_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 9:",
    BLOCK9_DIR
)

print(
    "Dasar konfigurasi:",
    "Mean CV Log Loss terendah"
)

print(
    "Dasar pemilihan model validation:",
    "Jumlah kemenangan pada lima metrik utama"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block9_objects = [
    "MODEL_ORDER",
    "SELECTION_METRICS",
    "ALL_METRICS",
    "LABEL_DISPLAY_NAME",
    "THRESHOLD",
    "cat_cols",
    "y_train",
    "y_val",
    "y_test",
    "X_train_rf",
    "X_val_rf",
    "X_test_rf",
    "X_train_xgb",
    "X_val_xgb",
    "X_test_xgb",
    "X_train_cb",
    "X_val_cb",
    "X_test_cb"
]


missing_block9_objects = [
    object_name
    for object_name in required_block9_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block9_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 9: "
        f"{missing_block9_objects}"
    )


# ============================================================
# MEMUAT HASIL BLOK 7 JIKA OBJEK BELUM TERSEDIA
# ============================================================

required_block7_objects = [
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model",
    "best_params_by_model"
]


missing_block7_objects = [
    object_name
    for object_name in required_block7_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block7_objects:

    print(
        "Objek hasil Blok 7 belum tersedia. "
        "Memuat hasil Blok 7 berbasis Log Loss."
    )


    if "BLOCK7_DIR" not in globals():

        BLOCK7_DIR = os.path.join(
            "./artifacts",
            (
                f"final_combination_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block7_result_path = os.path.join(
        BLOCK7_DIR,
        "block7_results_logloss.joblib"
    )


    if not os.path.exists(
        block7_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 7 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block7_result_path}\n"
            "Jalankan Blok 7 terlebih dahulu."
        )


    block7_results = joblib.load(
        block7_result_path
    )


    required_block7_keys = [
        "selected_aggregation_method",
        "selected_features_by_model",
        "selected_k_by_model",
        "best_params_by_model"
    ]


    missing_block7_keys = [
        key
        for key in required_block7_keys
        if key not in block7_results
    ]


    if missing_block7_keys:
        raise KeyError(
            "Isi hasil Blok 7 belum lengkap. "
            f"Key yang tidak ditemukan: {missing_block7_keys}"
        )


    selected_aggregation_method = (
        block7_results[
            "selected_aggregation_method"
        ]
    )


    selected_features_by_model = (
        block7_results[
            "selected_features_by_model"
        ]
    )


    selected_k_by_model = (
        block7_results[
            "selected_k_by_model"
        ]
    )


    best_params_by_model = (
        block7_results[
            "best_params_by_model"
        ]
    )


    selected_logloss_by_model = (
        block7_results.get(
            "selected_logloss_by_model",
            {}
        )
    )


    selected_auc_by_model = (
        block7_results.get(
            "selected_auc_by_model",
            {}
        )
    )


    print(
        "Hasil Blok 7 berhasil dimuat."
    )


# ============================================================
# VALIDASI HASIL BLOK 7
# ============================================================

for model_name in MODEL_ORDER:

    if model_name not in selected_features_by_model:
        raise KeyError(
            f"Fitur terpilih {model_name} belum tersedia."
        )


    if model_name not in selected_k_by_model:
        raise KeyError(
            f"Top-K terpilih {model_name} belum tersedia."
        )


    if model_name not in best_params_by_model:
        raise KeyError(
            f"Hyperparameter terpilih {model_name} "
            "belum tersedia."
        )


    if not isinstance(
        best_params_by_model[
            model_name
        ],
        dict
    ):
        raise TypeError(
            f"Hyperparameter {model_name} "
            "harus berupa dictionary."
        )


    if (
        len(
            selected_features_by_model[
                model_name
            ]
        )
        != int(
            selected_k_by_model[
                model_name
            ]
        )
    ):
        raise ValueError(
            f"Jumlah fitur terpilih {model_name} "
            "tidak sesuai dengan Top-K."
        )


# ============================================================
# DATA HASIL PREPROCESSING PENUH
# ============================================================

base_prep_data = {
    "Random Forest": {
        "train":
            X_train_rf,

        "val":
            X_val_rf,

        "test":
            X_test_rf
    },

    "XGBoost": {
        "train":
            X_train_xgb,

        "val":
            X_val_xgb,

        "test":
            X_test_xgb
    },

    "CatBoost": {
        "train":
            X_train_cb,

        "val":
            X_val_cb,

        "test":
            X_test_cb
    }
}


# ============================================================
# MEMBENTUK DATA FINAL SETIAP MODEL
# ============================================================

final_model_data = {}


for model_name in MODEL_ORDER:

    selected_features = list(
        selected_features_by_model[
            model_name
        ]
    )


    missing_features = {
        subset_name: [
            feature
            for feature in selected_features
            if feature not in dataset.columns
        ]
        for subset_name, dataset in (
            base_prep_data[
                model_name
            ].items()
        )
    }


    missing_features = {
        subset_name: features
        for subset_name, features in (
            missing_features.items()
        )
        if features
    }


    if missing_features:
        raise ValueError(
            f"Fitur {model_name} tidak ditemukan: "
            f"{missing_features}"
        )


    final_model_data[
        model_name
    ] = {
        "features":
            selected_features,

        "train": (
            base_prep_data[
                model_name
            ]["train"]
            .loc[
                :,
                selected_features
            ]
            .copy()
            .reset_index(drop=True)
        ),

        "val": (
            base_prep_data[
                model_name
            ]["val"]
            .loc[
                :,
                selected_features
            ]
            .copy()
            .reset_index(drop=True)
        ),

        "test": (
            base_prep_data[
                model_name
            ]["test"]
            .loc[
                :,
                selected_features
            ]
            .copy()
            .reset_index(drop=True)
        )
    }


# Menyamakan indeks target
y_train_final = (
    pd.Series(y_train)
    .reset_index(drop=True)
    .astype(int)
)

y_val_final = (
    pd.Series(y_val)
    .reset_index(drop=True)
    .astype(int)
)

y_test_final = (
    pd.Series(y_test)
    .reset_index(drop=True)
    .astype(int)
)


# ============================================================
# VALIDASI DATA FINAL
# ============================================================

expected_lengths = {
    "train":
        len(y_train_final),

    "val":
        len(y_val_final),

    "test":
        len(y_test_final)
}


for model_name in MODEL_ORDER:

    data = final_model_data[
        model_name
    ]


    for subset_name in [
        "train",
        "val",
        "test"
    ]:

        X_subset = data[
            subset_name
        ]


        if (
            list(
                X_subset.columns
            )
            != list(
                data["features"]
            )
        ):
            raise ValueError(
                f"Urutan fitur {model_name} "
                f"pada {subset_name} tidak sesuai."
            )


        if len(
            X_subset
        ) != expected_lengths[
            subset_name
        ]:
            raise ValueError(
                f"Jumlah data {subset_name} "
                f"{model_name} tidak sesuai."
            )


        if X_subset.isna().any().any():
            raise ValueError(
                f"Data {model_name} pada "
                f"{subset_name} memiliki missing value."
            )


        selected_num = [
            col
            for col in data["features"]
            if col in num_cols
        ]


        if selected_num:

            numeric_values = (
                X_subset[
                    selected_num
                ]
                .to_numpy(
                    dtype=float
                )
            )


            if not np.isfinite(
                numeric_values
            ).all():
                raise ValueError(
                    f"Fitur numerik {model_name} pada "
                    f"{subset_name} memiliki infinity."
                )


print(
    "Seluruh data final model valid."
)


# ============================================================
# FUNGSI PATH PER MODEL
# ============================================================

def get_block9_paths(
    model_name
):
    safe_model = safe_file_name(
        model_name
    )


    return {
        "model":
            os.path.join(
                MODEL_DIR,
                (
                    f"final_model_logloss_"
                    f"{safe_model}.joblib"
                )
            ),

        "prediction":
            os.path.join(
                PREDICTION_DIR,
                (
                    f"validation_prediction_logloss_"
                    f"{safe_model}.npz"
                )
            ),

        "metrics":
            os.path.join(
                TABLE_DIR,
                (
                    f"validation_metrics_logloss_"
                    f"{safe_model}.pkl"
                )
            ),

        "metrics_csv":
            os.path.join(
                TABLE_DIR,
                (
                    f"validation_metrics_logloss_"
                    f"{safe_model}.csv"
                )
            ),

        "logloss":
            os.path.join(
                LOGLOSS_DIR,
                (
                    f"logloss_history_"
                    f"{safe_model}.joblib"
                )
            )
    }


# ============================================================
# TRAINING DAN EVALUASI VALIDATION PER MODEL
# ============================================================

candidate_models = {}

validation_predictions = {}

validation_rows = []

xgb_logloss_results = {}

catboost_logloss_results = {}


for model_name in MODEL_ORDER:

    print(
        "\n"
        + "=" * 70
    )


    print(
        "EVALUASI VALIDATION "
        f"— {model_name}"
    )


    print(
        "=" * 70
    )


    paths = get_block9_paths(
        model_name
    )


    data = final_model_data[
        model_name
    ]


    selected_cat = [
        col
        for col in data["features"]
        if col in cat_cols
    ]


    model_checkpoint_complete = all([
        os.path.exists(
            paths["model"]
        ),
        os.path.exists(
            paths["prediction"]
        ),
        os.path.exists(
            paths["metrics"]
        )
    ])


    # ========================================================
    # MEMUAT HASIL MODEL JIKA SUDAH TERSEDIA
    # ========================================================

    if model_checkpoint_complete:

        print(
            "Hasil model sudah tersedia. "
            "Memuat dari checkpoint."
        )


        model = joblib.load(
            paths["model"]
        )


        prediction_data = np.load(
            paths["prediction"]
        )


        y_val_pred = np.asarray(
            prediction_data[
                "y_pred"
            ],
            dtype=int
        )


        y_val_proba = np.asarray(
            prediction_data[
                "y_proba"
            ],
            dtype=float
        )


        model_metric_table = pd.read_pickle(
            paths["metrics"]
        )


        missing_metric_columns = [
            metric
            for metric in ALL_METRICS
            if metric not in (
                model_metric_table.columns
            )
        ]


        if missing_metric_columns:
            raise ValueError(
                f"Checkpoint metrik {model_name} "
                f"belum lengkap: {missing_metric_columns}"
            )


        validation_metrics = {
            metric:
                float(
                    model_metric_table
                    .iloc[0][metric]
                )
            for metric in ALL_METRICS
        }


        if (
            model_name == "XGBoost"
            and os.path.exists(
                paths["logloss"]
            )
        ):
            xgb_logloss_results[
                model_name
            ] = joblib.load(
                paths["logloss"]
            )


        if (
            model_name == "CatBoost"
            and os.path.exists(
                paths["logloss"]
            )
        ):
            catboost_logloss_results[
                model_name
            ] = joblib.load(
                paths["logloss"]
            )


    # ========================================================
    # MELATIH MODEL JIKA CHECKPOINT BELUM TERSEDIA
    # ========================================================

    else:

        print(
            "Melatih konfigurasi terpilih "
            "pada seluruh data train."
        )


        model = build_model(
            model_name=model_name,
            params=(
                best_params_by_model[
                    model_name
                ].copy()
            ),
            use_early_stopping=True
        )


        model = fit_model(
            model_name=model_name,
            model=model,
            X_train_data=(
                data["train"]
            ),
            y_train_data=(
                y_train_final
            ),
            X_valid_data=(
                data["val"]
            ),
            y_valid_data=(
                y_val_final
            ),
            cat_features=(
                selected_cat
            ),
            include_train_in_eval=True,
            use_early_stopping=True
        )


        y_val_proba = (
            model
            .predict_proba(
                data["val"]
            )[:, 1]
        )


        (
            validation_metrics,
            y_val_pred
        ) = calculate_metrics(
            y_true=y_val_final,
            y_proba=y_val_proba,
            threshold=THRESHOLD
        )


        # ----------------------------------------------------
        # Menyimpan riwayat Log Loss XGBoost
        # ----------------------------------------------------

        if model_name == "XGBoost":

            eval_result = (
                model.evals_result()
            )


            required_eval_keys = [
                "validation_0",
                "validation_1"
            ]


            missing_eval_keys = [
                key
                for key in required_eval_keys
                if key not in eval_result
            ]


            if missing_eval_keys:
                raise ValueError(
                    "Riwayat evaluasi XGBoost "
                    f"belum lengkap: {missing_eval_keys}"
                )


            logloss_result = {
                "Train":
                    [
                        float(value)
                        for value in (
                            eval_result[
                                "validation_0"
                            ]["logloss"]
                        )
                    ],

                "Validation":
                    [
                        float(value)
                        for value in (
                            eval_result[
                                "validation_1"
                            ]["logloss"]
                        )
                    ],

                "Best Iteration":
                    (
                        None
                        if (
                            model.best_iteration
                            is None
                        )
                        else int(
                            model.best_iteration
                        )
                    )
            }


            xgb_logloss_results[
                model_name
            ] = logloss_result


            joblib.dump(
                logloss_result,
                paths["logloss"]
            )


        # ----------------------------------------------------
        # Menyimpan riwayat Log Loss CatBoost
        # ----------------------------------------------------

        if model_name == "CatBoost":

            eval_result = (
                model.get_evals_result()
            )


            validation_keys = [
                key
                for key in eval_result
                if key != "learn"
            ]


            if not validation_keys:
                raise ValueError(
                    "Riwayat validation CatBoost "
                    "tidak ditemukan."
                )


            if (
                "learn" not in eval_result
                or "Logloss" not in (
                    eval_result["learn"]
                )
            ):
                raise ValueError(
                    "Riwayat training Log Loss "
                    "CatBoost tidak ditemukan."
                )


            validation_key = (
                validation_keys[0]
            )


            if (
                "Logloss"
                not in eval_result[
                    validation_key
                ]
            ):
                raise ValueError(
                    "Riwayat validation Log Loss "
                    "CatBoost tidak ditemukan."
                )


            best_iteration_cat = (
                model.get_best_iteration()
            )


            logloss_result = {
                "Train":
                    [
                        float(value)
                        for value in (
                            eval_result[
                                "learn"
                            ]["Logloss"]
                        )
                    ],

                "Validation":
                    [
                        float(value)
                        for value in (
                            eval_result[
                                validation_key
                            ]["Logloss"]
                        )
                    ],

                "Best Iteration":
                    (
                        None
                        if (
                            best_iteration_cat
                            is None
                            or best_iteration_cat < 0
                        )
                        else int(
                            best_iteration_cat
                        )
                    )
            }


            catboost_logloss_results[
                model_name
            ] = logloss_result


            joblib.dump(
                logloss_result,
                paths["logloss"]
            )


        # ----------------------------------------------------
        # Membentuk tabel metrik per model
        # ----------------------------------------------------

        model_metric_table = pd.DataFrame([
            {
                "Skenario Pelabelan":
                    LABEL_DISPLAY_NAME,

                "Model":
                    model_name,

                "Metode Agregasi":
                    selected_aggregation_method,

                "Top-K":
                    int(
                        selected_k_by_model[
                            model_name
                        ]
                    ),

                "Fitur":
                    ", ".join(
                        data["features"]
                    ),

                **{
                    metric:
                        float(value)
                    for metric, value in (
                        validation_metrics.items()
                    )
                }
            }
        ])


        # ----------------------------------------------------
        # Menyimpan model, prediksi, dan metrik
        # ----------------------------------------------------

        joblib.dump(
            model,
            paths["model"]
        )


        np.savez_compressed(
            paths["prediction"],
            y_pred=(
                np.asarray(
                    y_val_pred,
                    dtype=int
                )
            ),
            y_proba=(
                np.asarray(
                    y_val_proba,
                    dtype=float
                )
            )
        )


        model_metric_table.to_pickle(
            paths["metrics"]
        )


        model_metric_table.to_csv(
            paths["metrics_csv"],
            index=False
        )


        print(
            "Checkpoint model tersimpan:",
            paths["model"]
        )


    # ========================================================
    # VALIDASI HASIL PREDIKSI DAN METRIK
    # ========================================================

    if len(
        y_val_pred
    ) != len(
        y_val_final
    ):
        raise ValueError(
            f"Jumlah prediksi validation {model_name} "
            "tidak sesuai."
        )


    if len(
        y_val_proba
    ) != len(
        y_val_final
    ):
        raise ValueError(
            f"Jumlah probabilitas validation {model_name} "
            "tidak sesuai."
        )


    if not np.isfinite(
        y_val_proba
    ).all():
        raise ValueError(
            f"Probabilitas validation {model_name} "
            "mengandung NaN atau infinity."
        )


    if (
        (y_val_proba < 0).any()
        or (y_val_proba > 1).any()
    ):
        raise ValueError(
            f"Probabilitas validation {model_name} "
            "berada di luar rentang 0 sampai 1."
        )


    if any(
        not np.isfinite(
            float(
                validation_metrics[
                    metric
                ]
            )
        )
        for metric in ALL_METRICS
    ):
        raise ValueError(
            f"Metrik validation {model_name} "
            "mengandung nilai tidak valid."
        )


    # ========================================================
    # MENYIMPAN HASIL PADA VARIABEL NOTEBOOK
    # ========================================================

    candidate_models[
        model_name
    ] = model


    validation_predictions[
        model_name
    ] = {
        "y_pred":
            np.asarray(
                y_val_pred,
                dtype=int
            ),

        "y_proba":
            np.asarray(
                y_val_proba,
                dtype=float
            )
    }


    validation_rows.append({
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            model_name,

        "Metode Agregasi":
            selected_aggregation_method,

        "Top-K":
            int(
                selected_k_by_model[
                    model_name
                ]
            ),

        "Fitur":
            ", ".join(
                data["features"]
            ),

        **{
            metric:
                float(value)
            for metric, value in (
                validation_metrics.items()
            )
        }
    })


    print(
        "Accuracy :",
        f"{validation_metrics['Accuracy']:.4f}"
    )

    print(
        "Precision:",
        f"{validation_metrics['Precision']:.4f}"
    )

    print(
        "Recall   :",
        f"{validation_metrics['Recall']:.4f}"
    )

    print(
        "F1-Score :",
        f"{validation_metrics['F1-Score']:.4f}"
    )

    print(
        "ROC-AUC  :",
        f"{validation_metrics['ROC-AUC']:.4f}"
    )

    print(
        "Log Loss :",
        f"{validation_metrics['Log Loss']:.6f}"
    )


    gc.collect()


# ============================================================
# TABEL HASIL VALIDATION SELURUH MODEL
# ============================================================

validation_table = pd.DataFrame(
    validation_rows
)


validation_table[
    "Model"
] = pd.Categorical(
    validation_table[
        "Model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


validation_table = (
    validation_table
    .sort_values(
        "Model"
    )
    .reset_index(drop=True)
)


validation_table[
    "Model"
] = validation_table[
    "Model"
].astype(str)


validation_table.insert(
    0,
    "No",
    range(
        1,
        len(
            validation_table
        ) + 1
    )
)


# ------------------------------------------------------------
# Tabel ringkas lima metrik utama
# ------------------------------------------------------------

validation_summary = (
    validation_table[
        [
            "No",
            "Model",
            "Top-K",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC",
            "Log Loss"
        ]
    ]
    .copy()
)


# ============================================================
# MENGHITUNG PEMENANG SETIAP METRIK UTAMA
#
# Log Loss tidak dihitung sebagai jumlah kemenangan karena
# pemilihan model menggunakan lima metrik utama.
# ============================================================

validation_win_count = {
    model_name: 0
    for model_name in MODEL_ORDER
}


metric_winner_rows = []


for metric in SELECTION_METRICS:

    best_value = float(
        validation_table[
            metric
        ].max()
    )


    winner_mask = np.isclose(
        validation_table[
            metric
        ].astype(float),
        best_value,
        rtol=0.0,
        atol=1e-12
    )


    winner_names = (
        validation_table
        .loc[
            winner_mask,
            "Model"
        ]
        .tolist()
    )


    if len(
        winner_names
    ) == 1:

        winner_text = (
            winner_names[0]
        )


        validation_win_count[
            winner_text
        ] += 1


    else:

        winner_text = (
            "Tie: "
            + ", ".join(
                winner_names
            )
        )


    metric_winner_rows.append({
        "Metrik":
            metric,

        "Pemenang":
            winner_text,

        "Nilai Terbaik":
            best_value
    })


validation_metric_winner_table = pd.DataFrame(
    metric_winner_rows
)


validation_metric_winner_table.insert(
    0,
    "No",
    range(
        1,
        len(
            validation_metric_winner_table
        ) + 1
    )
)


# ============================================================
# MENENTUKAN MODEL TERBAIK
#
# Urutan:
# 1. jumlah metrik utama yang dimenangkan;
# 2. ROC-AUC validation tertinggi;
# 3. Log Loss validation terendah jika masih sama;
# 4. urutan MODEL_ORDER jika seluruh nilai sama.
# ============================================================

model_order_mapping = {
    model_name: index
    for index, model_name in enumerate(
        MODEL_ORDER
    )
}


validation_comparison = (
    validation_table
    .copy()
)


validation_comparison[
    "Jumlah Menang"
] = (
    validation_comparison[
        "Model"
    ]
    .map(
        validation_win_count
    )
)


validation_comparison[
    "Urutan Model"
] = (
    validation_comparison[
        "Model"
    ]
    .map(
        model_order_mapping
    )
)


validation_comparison = (
    validation_comparison
    .sort_values(
        by=[
            "Jumlah Menang",
            "ROC-AUC",
            "Log Loss",
            "Urutan Model"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


validation_comparison[
    "Peringkat"
] = range(
    1,
    len(
        validation_comparison
    ) + 1
)


validation_comparison = (
    validation_comparison[
        [
            "Peringkat",
            "Skenario Pelabelan",
            "Model",
            "Metode Agregasi",
            "Top-K",
            "Fitur",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC",
            "PR-AUC",
            "Log Loss",
            "Brier Score",
            "Jumlah Menang"
        ]
    ]
)


best_model_name = str(
    validation_comparison
    .loc[
        0,
        "Model"
    ]
)


best_model = (
    candidate_models[
        best_model_name
    ]
)


# ------------------------------------------------------------
# Tabel ringkas penentuan model terbaik
# ------------------------------------------------------------

validation_selection_summary = (
    validation_comparison[
        [
            "Peringkat",
            "Model",
            "Top-K",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC",
            "Log Loss",
            "Jumlah Menang"
        ]
    ]
    .copy()
)


# ============================================================
# MENYIMPAN HASIL BLOK 9
# ============================================================

tables_to_save = {
    "validation_table_logloss":
        validation_table,

    "validation_summary_logloss":
        validation_summary,

    "validation_metric_winner_table_logloss":
        validation_metric_winner_table,

    "validation_comparison_logloss":
        validation_comparison,

    "validation_selection_summary_logloss":
        validation_selection_summary
}


for file_name, dataframe in (
    tables_to_save.items()
):

    dataframe.to_csv(
        os.path.join(
            TABLE_DIR,
            f"{file_name}.csv"
        ),
        index=False
    )


    dataframe.to_pickle(
        os.path.join(
            TABLE_DIR,
            f"{file_name}.pkl"
        )
    )


# ============================================================
# MENYIMPAN HASIL UTAMA BLOK 9
# ============================================================

block9_results = {
    "configuration_selection_criterion":
        "Mean CV Log Loss terendah",

    "model_selection_criterion":
        (
            "Jumlah kemenangan lima metrik utama, "
            "kemudian ROC-AUC validation tertinggi"
        ),

    "selected_aggregation_method":
        selected_aggregation_method,

    "selected_features_by_model":
        selected_features_by_model,

    "selected_k_by_model":
        selected_k_by_model,

    "best_params_by_model":
        best_params_by_model,

    "final_model_data":
        final_model_data,

    "candidate_models":
        candidate_models,

    "validation_predictions":
        validation_predictions,

    "validation_table":
        validation_table,

    "validation_summary":
        validation_summary,

    "validation_metric_winner_table":
        validation_metric_winner_table,

    "validation_comparison":
        validation_comparison,

    "validation_selection_summary":
        validation_selection_summary,

    "validation_win_count":
        validation_win_count,

    "best_model_name":
        best_model_name,

    "best_model":
        best_model,

    "xgb_logloss_results":
        xgb_logloss_results,

    "catboost_logloss_results":
        catboost_logloss_results
}


joblib.dump(
    block9_results,
    os.path.join(
        BLOCK9_DIR,
        "block9_results_logloss.joblib"
    )
)


# ============================================================
# METADATA BLOK 9
# ============================================================

block9_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Dasar Pemilihan Konfigurasi":
        "Mean CV Log Loss terendah",

    "Metode Agregasi":
        selected_aggregation_method,

    "Threshold":
        float(
            THRESHOLD
        ),

    "Metrik Pemilihan Model":
        list(
            SELECTION_METRICS
        ),

    "Metrik Tambahan":
        [
            "PR-AUC",
            "Log Loss",
            "Brier Score"
        ],

    "Aturan Pemilihan Model":
        (
            "Jumlah kemenangan lima metrik utama; "
            "ROC-AUC validation tertinggi sebagai tie-break; "
            "Log Loss validation terendah sebagai tie-break tambahan"
        ),

    "Model Terbaik":
        best_model_name,

    "Top-K Model Terbaik":
        int(
            selected_k_by_model[
                best_model_name
            ]
        ),

    "Fitur Model Terbaik":
        list(
            selected_features_by_model[
                best_model_name
            ]
        ),

    "Jumlah Menang":
        int(
            validation_comparison
            .loc[
                0,
                "Jumlah Menang"
            ]
        ),

    "Validation ROC-AUC Model Terbaik":
        float(
            validation_comparison
            .loc[
                0,
                "ROC-AUC"
            ]
        ),

    "Validation Log Loss Model Terbaik":
        float(
            validation_comparison
            .loc[
                0,
                "Log Loss"
            ]
        )
}


with open(
    os.path.join(
        BLOCK9_DIR,
        "block9_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block9_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT BLOK 9
# ============================================================

print(
    "\n"
    + "=" * 70
)


print(
    "TABEL LENGKAP HASIL VALIDATION"
)


print(
    "=" * 70
)


display_without_index(
    validation_table,
    formats={
        metric:
            "{:.4f}"
        for metric in ALL_METRICS
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS HASIL VALIDATION"
)


print(
    "=" * 70
)


display_without_index(
    validation_summary,
    formats={
        "Accuracy":
            "{:.4f}",

        "Precision":
            "{:.4f}",

        "Recall":
            "{:.4f}",

        "F1-Score":
            "{:.4f}",

        "ROC-AUC":
            "{:.4f}",

        "Log Loss":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL PEMENANG SETIAP "
    "METRIK VALIDATION"
)


print(
    "=" * 70
)


display_without_index(
    validation_metric_winner_table,
    formats={
        "Nilai Terbaik":
            "{:.4f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS PEMILIHAN MODEL"
)


print(
    "=" * 70
)


display_without_index(
    validation_selection_summary,
    formats={
        "Accuracy":
            "{:.4f}",

        "Precision":
            "{:.4f}",

        "Recall":
            "{:.4f}",

        "F1-Score":
            "{:.4f}",

        "ROC-AUC":
            "{:.4f}",

        "Log Loss":
            "{:.6f}"
    }
)


print(
    "\nModel terbaik pada validation:",
    best_model_name
)


print(
    "Top-K model terbaik:",
    selected_k_by_model[
        best_model_name
    ]
)


print(
    "Validation ROC-AUC:",
    (
        f"{validation_comparison.loc[0, 'ROC-AUC']:.4f}"
    )
)


print(
    "Validation Log Loss:",
    (
        f"{validation_comparison.loc[0, 'Log Loss']:.6f}"
    )
)


print(
    "\nSeluruh hasil Blok 9 "
    "berhasil disimpan di:"
)


print(
    BLOCK9_DIR
)

## MODEL TERBAIK

In [ ]:
# ============================================================
# BLOK 10 — KURVA LOG LOSS XGBOOST DAN CATBOOST
# HASIL KONFIGURASI BERBASIS MEAN CV LOG LOSS
# DENGAN PENYIMPANAN DAN RESUME
#
# Blok ini tidak melakukan training ulang.
# Riwayat Train Log Loss dan Validation Log Loss
# diambil dari hasil Blok 9.
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import (
    MultipleLocator,
    FormatStrFormatter
)


# ============================================================
# FOLDER PENYIMPANAN BLOK 10
# ============================================================

BLOCK10_DIR = os.path.join(
    "./artifacts",
    (
        f"logloss_analysis_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

FIGURE_DIR = os.path.join(
    BLOCK10_DIR,
    "figures"
)

TABLE_DIR_BLOCK10 = os.path.join(
    BLOCK10_DIR,
    "tables"
)

CURVE_DIR = os.path.join(
    BLOCK10_DIR,
    "curve_data"
)

for directory in [
    BLOCK10_DIR,
    FIGURE_DIR,
    TABLE_DIR_BLOCK10,
    CURVE_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 10:",
    BLOCK10_DIR
)

print(
    "Sumber riwayat Log Loss:",
    "Hasil training validation pada Blok 9"
)

print(
    "Blok 10 tidak melakukan training ulang."
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block10_objects = [
    "LABEL_DISPLAY_NAME",
    "MODEL_ORDER"
]

missing_block10_objects = [
    object_name
    for object_name in required_block10_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]

if missing_block10_objects:

    raise NameError(
        "Objek berikut belum tersedia untuk Blok 10: "
        f"{missing_block10_objects}"
    )


# ============================================================
# MEMUAT RIWAYAT LOG LOSS DARI BLOK 9
# ============================================================

xgb_missing = (
    "xgb_logloss_results" not in globals()
    or not xgb_logloss_results
)

catboost_missing = (
    "catboost_logloss_results" not in globals()
    or not catboost_logloss_results
)


if xgb_missing or catboost_missing:

    print(
        "Riwayat Log Loss belum tersedia pada notebook. "
        "Memuat hasil Blok 9 berbasis Log Loss."
    )


    if "BLOCK9_DIR" not in globals():

        BLOCK9_DIR = os.path.join(
            "./artifacts",
            (
                f"validation_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block9_result_path = os.path.join(
        BLOCK9_DIR,
        "block9_results_logloss.joblib"
    )


    if not os.path.exists(
        block9_result_path
    ):

        raise FileNotFoundError(
            "Hasil Blok 9 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block9_result_path}\n"
            "Jalankan Blok 9 terlebih dahulu."
        )


    block9_results = joblib.load(
        block9_result_path
    )


    xgb_logloss_results = (
        block9_results.get(
            "xgb_logloss_results",
            {}
        )
    )


    catboost_logloss_results = (
        block9_results.get(
            "catboost_logloss_results",
            {}
        )
    )


    print(
        "Riwayat Log Loss berhasil dimuat dari:"
    )

    print(
        block9_result_path
    )


# ============================================================
# VALIDASI KETERSEDIAAN RIWAYAT LOG LOSS
# ============================================================

if "XGBoost" not in xgb_logloss_results:

    raise ValueError(
        "Riwayat Log Loss XGBoost tidak tersedia. "
        "Pastikan Blok 9 menjalankan XGBoost dengan "
        "include_train_in_eval=True."
    )


if "CatBoost" not in catboost_logloss_results:

    raise ValueError(
        "Riwayat Log Loss CatBoost tidak tersedia. "
        "Pastikan Blok 9 menjalankan CatBoost dengan "
        "eval_set data validation."
    )


required_logloss_keys = [
    "Train",
    "Validation",
    "Best Iteration"
]


logloss_results_by_model = {
    "XGBoost":
        xgb_logloss_results[
            "XGBoost"
        ],

    "CatBoost":
        catboost_logloss_results[
            "CatBoost"
        ]
}


for model_name, result in (
    logloss_results_by_model.items()
):

    missing_keys = [
        key
        for key in required_logloss_keys
        if key not in result
    ]


    if missing_keys:

        raise ValueError(
            f"Riwayat Log Loss {model_name} "
            f"belum lengkap: {missing_keys}"
        )


    train_curve = np.asarray(
        result["Train"],
        dtype=float
    )


    validation_curve = np.asarray(
        result["Validation"],
        dtype=float
    )


    if train_curve.size == 0:

        raise ValueError(
            f"Kurva Train Log Loss {model_name} kosong."
        )


    if validation_curve.size == 0:

        raise ValueError(
            f"Kurva Validation Log Loss {model_name} kosong."
        )


    if not np.isfinite(
        train_curve
    ).all():

        raise ValueError(
            f"Kurva Train Log Loss {model_name} "
            "mengandung NaN atau infinity."
        )


    if not np.isfinite(
        validation_curve
    ).all():

        raise ValueError(
            f"Kurva Validation Log Loss {model_name} "
            "mengandung NaN atau infinity."
        )


# ============================================================
# KONFIGURASI VISUALISASI
# ============================================================

LOGLOSS_GAP_WARNING_LIMIT = 0.10

# Jarak antartick sumbu-y
Y_AXIS_INTERVAL = 0.025

# Ruang kecil di atas dan bawah kurva
Y_AXIS_TOP_PADDING = 0.005
Y_AXIS_BOTTOM_PADDING = 0.005

# Sumbu-x dimulai dari 0 dan berhenti tepat
# pada iterasi terakhir
X_AXIS_LEFT_LIMIT = -30
X_AXIS_RIGHT_PADDING = 0


# ============================================================
# FUNGSI INTERPRETASI GAP LOG LOSS
# ============================================================

def interpret_logloss_gap(
    gap,
    warning_limit=LOGLOSS_GAP_WARNING_LIMIT
):
    """
    Memberikan interpretasi deskriptif terhadap selisih
    Validation Log Loss dan Train Log Loss.
    """

    absolute_gap = abs(
        float(gap)
    )


    if gap < 0:

        if absolute_gap <= warning_limit:

            return (
                "Validation Log Loss lebih rendah daripada "
                "Train Log Loss dengan selisih di bawah 0,10."
            )


        return (
            "Validation Log Loss lebih rendah daripada "
            "Train Log Loss dengan selisih melebihi 0,10. "
            "Pola kurva perlu diperiksa lebih lanjut."
        )


    if gap <= warning_limit:

        return (
            "Validation Log Loss lebih tinggi daripada "
            "Train Log Loss, tetapi selisihnya masih "
            "berada di bawah atau sama dengan 0,10."
        )


    return (
        "Validation Log Loss lebih tinggi daripada "
        "Train Log Loss dengan selisih melebihi 0,10. "
        "Kondisi ini perlu dicermati bersama pola kurva "
        "sebagai kemungkinan penurunan generalisasi."
    )


# ============================================================
# FUNGSI MENENTUKAN ITERASI TERBAIK
# ============================================================

def resolve_best_iteration_index(
    result,
    validation_curve,
    maximum_index
):
    """
    Menghasilkan indeks iterasi terbaik berbasis indeks Python.
    """

    raw_best_iteration = result.get(
        "Best Iteration"
    )


    if (
        raw_best_iteration is None
        or int(raw_best_iteration) < 0
    ):

        best_index = int(
            np.argmin(
                validation_curve[
                    :maximum_index + 1
                ]
            )
        )


    else:

        best_index = int(
            raw_best_iteration
        )


    best_index = max(
        0,
        min(
            best_index,
            maximum_index
        )
    )


    return best_index


# ============================================================
# FUNGSI MEMBENTUK RINGKASAN GAP LOG LOSS
# ============================================================

def create_logloss_gap_row(
    model_name,
    result
):
    """
    Menghitung Train Log Loss, Validation Log Loss, dan gap
    pada iterasi terbaik serta iterasi terakhir.
    """

    train_curve = np.asarray(
        result["Train"],
        dtype=float
    )


    validation_curve = np.asarray(
        result["Validation"],
        dtype=float
    )


    common_length = min(
        len(train_curve),
        len(validation_curve)
    )


    if common_length == 0:

        raise ValueError(
            f"Kurva Log Loss {model_name} tidak memiliki "
            "iterasi yang dapat dibandingkan."
        )


    train_curve = train_curve[
        :common_length
    ]


    validation_curve = validation_curve[
        :common_length
    ]


    maximum_index = (
        common_length - 1
    )


    best_index = resolve_best_iteration_index(
        result=result,
        validation_curve=validation_curve,
        maximum_index=maximum_index
    )


    best_iteration_display = (
        best_index + 1
    )


    train_logloss_best = float(
        train_curve[
            best_index
        ]
    )


    validation_logloss_best = float(
        validation_curve[
            best_index
        ]
    )


    gap_best = float(
        validation_logloss_best
        - train_logloss_best
    )


    final_index = maximum_index


    final_iteration_display = (
        final_index + 1
    )


    train_logloss_final = float(
        train_curve[
            final_index
        ]
    )


    validation_logloss_final = float(
        validation_curve[
            final_index
        ]
    )


    gap_final = float(
        validation_logloss_final
        - train_logloss_final
    )


    minimum_validation_index = int(
        np.argmin(
            validation_curve
        )
    )


    minimum_validation_iteration = (
        minimum_validation_index + 1
    )


    minimum_validation_logloss = float(
        validation_curve[
            minimum_validation_index
        ]
    )


    return {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            model_name,

        "Jumlah Iterasi Tersedia":
            int(
                common_length
            ),

        "Best Iteration":
            int(
                best_iteration_display
            ),

        "Best Iteration Index":
            int(
                best_index
            ),

        "Train Log Loss Best":
            train_logloss_best,

        "Validation Log Loss Best":
            validation_logloss_best,

        "Gap Best":
            gap_best,

        "Absolute Gap Best":
            abs(
                gap_best
            ),

        "Status Batas 0,10":
            (
                "Di bawah atau sama dengan 0,10"
                if abs(
                    gap_best
                ) <= LOGLOSS_GAP_WARNING_LIMIT
                else "Melebihi 0,10"
            ),

        "Interpretasi Gap Best":
            interpret_logloss_gap(
                gap_best
            ),

        "Iterasi Validation Log Loss Minimum":
            int(
                minimum_validation_iteration
            ),

        "Validation Log Loss Minimum":
            minimum_validation_logloss,

        "Final Iteration":
            int(
                final_iteration_display
            ),

        "Train Log Loss Final":
            train_logloss_final,

        "Validation Log Loss Final":
            validation_logloss_final,

        "Gap Final":
            gap_final,

        "Absolute Gap Final":
            abs(
                gap_final
            ),

        "Interpretasi Gap Final":
            interpret_logloss_gap(
                gap_final
            )
    }


# ============================================================
# MEMBENTUK RINGKASAN DAN VISUALISASI
# ============================================================

logloss_gap_rows = []

logloss_curve_tables = {}


for model_name in [
    "XGBoost",
    "CatBoost"
]:

    print(
        "\n"
        + "=" * 70
    )


    print(
        f"ANALISIS LOG LOSS — {model_name}"
    )


    print(
        "=" * 70
    )


    result = logloss_results_by_model[
        model_name
    ]


    gap_row = create_logloss_gap_row(
        model_name=model_name,
        result=result
    )


    logloss_gap_rows.append(
        gap_row
    )


    train_curve = np.asarray(
        result["Train"],
        dtype=float
    )


    validation_curve = np.asarray(
        result["Validation"],
        dtype=float
    )


    common_length = min(
        len(train_curve),
        len(validation_curve)
    )


    train_curve = train_curve[
        :common_length
    ]


    validation_curve = validation_curve[
        :common_length
    ]


    iteration_numbers = np.arange(
        1,
        common_length + 1
    )


    # --------------------------------------------------------
    # Membentuk tabel data kurva
    # --------------------------------------------------------

    curve_table = pd.DataFrame({
        "Iteration":
            iteration_numbers,

        "Train Log Loss":
            train_curve,

        "Validation Log Loss":
            validation_curve,

        "Gap":
            (
                validation_curve
                - train_curve
            )
    })


    curve_table[
        "Best Iteration"
    ] = (
        curve_table[
            "Iteration"
        ]
        == gap_row[
            "Best Iteration"
        ]
    )


    logloss_curve_tables[
        model_name
    ] = curve_table


    safe_model = safe_file_name(
        model_name
    )


    curve_csv_path = os.path.join(
        CURVE_DIR,
        (
            f"logloss_curve_data_logloss_"
            f"{safe_model}.csv"
        )
    )


    curve_pickle_path = os.path.join(
        CURVE_DIR,
        (
            f"logloss_curve_data_logloss_"
            f"{safe_model}.pkl"
        )
    )


    curve_table.to_csv(
        curve_csv_path,
        index=False
    )


    curve_table.to_pickle(
        curve_pickle_path
    )


    # --------------------------------------------------------
    # Menentukan batas visualisasi
    # --------------------------------------------------------

    best_iteration = gap_row[
        "Best Iteration"
    ]


    final_iteration = gap_row[
        "Final Iteration"
    ]


    y_all = np.concatenate(
        [
            train_curve,
            validation_curve
        ]
    )


    y_min = float(
        np.min(
            y_all
        )
    )


    y_max = float(
        np.max(
            y_all
        )
    )


    # --------------------------------------------------------
    # Batas sumbu-y mengikuti nilai aktual kurva.
    #
    # Padding dibuat kecil agar bagian atas dan bawah grafik
    # tidak terlalu jauh dari kurva.
    #
    # Batas tidak dibulatkan ke kelipatan 0,025 karena
    # pembulatan dapat menciptakan ruang kosong berlebihan.
    # --------------------------------------------------------

    y_lower_limit = (
        y_min
        - Y_AXIS_BOTTOM_PADDING
    )


    y_upper_limit = (
        y_max
        + Y_AXIS_TOP_PADDING
    )


    # ========================================================
    # VISUALISASI KURVA
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )


    ax.plot(
        iteration_numbers,
        train_curve,
        label="Train Log Loss",
        linewidth=1.8,
        solid_capstyle="round"
    )


    ax.plot(
        iteration_numbers,
        validation_curve,
        label="Validation Log Loss",
        color="darkorange",
        linewidth=1.8,
        solid_capstyle="round"
    )


    ax.axvline(
        x=best_iteration,
        linestyle="--",
        linewidth=1.5,
        label=(
            "Best Iteration "
            f"({best_iteration})"
        )
    )


    ax.scatter(
        [
            best_iteration
        ],
        [
            gap_row[
                "Validation Log Loss Best"
            ]
        ],
        color="darkorange",
        zorder=3
    )


    # --------------------------------------------------------
    # Batas sumbu-x
    #
    # Dimulai dari 0.
    # Berhenti tepat pada iterasi terakhir.
    # --------------------------------------------------------

    ax.set_xlim(
        X_AXIS_LEFT_LIMIT,
        final_iteration
        + X_AXIS_RIGHT_PADDING
    )


    # --------------------------------------------------------
    # Batas dan interval sumbu-y
    # --------------------------------------------------------

    ax.set_ylim(
        y_lower_limit,
        y_upper_limit
    )


    ax.yaxis.set_major_locator(
        MultipleLocator(
            Y_AXIS_INTERVAL
        )
    )


    # Tiga desimal diperlukan agar interval 0,025
    # ditampilkan secara tepat.
    ax.yaxis.set_major_formatter(
        FormatStrFormatter(
            "%.3f"
        )
    )


    # --------------------------------------------------------
    # Judul dan label
    # --------------------------------------------------------

    ax.set_title(
        "Log Loss Curve — "
        f"{model_name}\n"
        f"{LABEL_DISPLAY_NAME}"
    )


    ax.set_xlabel(
        "Iteration"
    )


    ax.set_ylabel(
        "Log Loss"
    )


    # --------------------------------------------------------
    # Posisi legenda konsisten di kiri bawah
    # --------------------------------------------------------

    ax.legend(
        loc="lower left"
    )


    fig.tight_layout()


    figure_path = os.path.join(
        FIGURE_DIR,
        (
            f"logloss_curve_logloss_"
            f"{safe_model}.png"
        )
    )


    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


    plt.close(
        fig
    )


    # --------------------------------------------------------
    # Output analisis
    # --------------------------------------------------------

    print(
        "Best iteration:",
        gap_row[
            "Best Iteration"
        ]
    )


    print(
        "Final iteration:",
        gap_row[
            "Final Iteration"
        ]
    )


    print(
        "Jeda setelah best iteration:",
        (
            gap_row[
                "Final Iteration"
            ]
            - gap_row[
                "Best Iteration"
            ]
        )
    )


    print(
        "Nilai Log Loss terendah aktual:",
        f"{y_min:.6f}"
    )


    print(
        "Nilai Log Loss tertinggi aktual:",
        f"{y_max:.6f}"
    )


    print(
        "Batas bawah sumbu-y:",
        f"{y_lower_limit:.6f}"
    )


    print(
        "Batas atas sumbu-y:",
        f"{y_upper_limit:.6f}"
    )


    print(
        "Interval sumbu-y:",
        f"{Y_AXIS_INTERVAL:.3f}"
    )


    print(
        "Train Log Loss Best:",
        f"{gap_row['Train Log Loss Best']:.6f}"
    )


    print(
        "Validation Log Loss Best:",
        f"{gap_row['Validation Log Loss Best']:.6f}"
    )


    print(
        "Gap Best:",
        f"{gap_row['Gap Best']:.6f}"
    )


    print(
        "Status batas 0,10:",
        gap_row[
            "Status Batas 0,10"
        ]
    )


    print(
        "Interpretasi:",
        gap_row[
            "Interpretasi Gap Best"
        ]
    )


    print(
        "Data kurva tersimpan:",
        curve_csv_path
    )


    print(
        "Gambar tersimpan:",
        figure_path
    )


# ============================================================
# TABEL RINGKAS GAP LOG LOSS
# ============================================================

logloss_gap_summary = pd.DataFrame(
    logloss_gap_rows
)


logloss_gap_summary[
    "Model"
] = pd.Categorical(
    logloss_gap_summary[
        "Model"
    ],
    categories=[
        "XGBoost",
        "CatBoost"
    ],
    ordered=True
)


logloss_gap_summary = (
    logloss_gap_summary
    .sort_values(
        "Model"
    )
    .reset_index(
        drop=True
    )
)


logloss_gap_summary[
    "Model"
] = (
    logloss_gap_summary[
        "Model"
    ]
    .astype(str)
)


logloss_gap_summary.insert(
    0,
    "No",
    range(
        1,
        len(
            logloss_gap_summary
        ) + 1
    )
)


# ------------------------------------------------------------
# Versi ringkas untuk Bab 4
# ------------------------------------------------------------

logloss_gap_report = (
    logloss_gap_summary[
        [
            "No",
            "Model",
            "Jumlah Iterasi Tersedia",
            "Best Iteration",
            "Final Iteration",
            "Train Log Loss Best",
            "Validation Log Loss Best",
            "Gap Best",
            "Status Batas 0,10",
            "Interpretasi Gap Best"
        ]
    ]
    .copy()
)


# ============================================================
# MENYIMPAN HASIL BLOK 10
# ============================================================

logloss_gap_summary.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK10,
        "logloss_gap_summary_full_logloss.csv"
    ),
    index=False
)


logloss_gap_summary.to_pickle(
    os.path.join(
        TABLE_DIR_BLOCK10,
        "logloss_gap_summary_full_logloss.pkl"
    )
)


logloss_gap_report.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK10,
        "logloss_gap_report_logloss.csv"
    ),
    index=False
)


logloss_gap_report.to_pickle(
    os.path.join(
        TABLE_DIR_BLOCK10,
        "logloss_gap_report_logloss.pkl"
    )
)


# ============================================================
# MENYIMPAN OBJEK UTAMA BLOK 10
# ============================================================

block10_results = {
    "source":
        "Riwayat training dan validation dari Blok 9",

    "xgb_logloss_results":
        xgb_logloss_results,

    "catboost_logloss_results":
        catboost_logloss_results,

    "logloss_curve_tables":
        logloss_curve_tables,

    "logloss_gap_summary":
        logloss_gap_summary,

    "logloss_gap_report":
        logloss_gap_report,

    "visualization_configuration": {
        "Y Axis Interval":
            Y_AXIS_INTERVAL,

        "Y Axis Top Padding":
            Y_AXIS_TOP_PADDING,

        "Y Axis Bottom Padding":
            Y_AXIS_BOTTOM_PADDING,

        "Legend Location":
            "lower left",

        "X Axis Left Limit":
            X_AXIS_LEFT_LIMIT,

        "X Axis Right Padding":
            X_AXIS_RIGHT_PADDING
    }
}


joblib.dump(
    block10_results,
    os.path.join(
        BLOCK10_DIR,
        "block10_results_logloss.joblib"
    )
)


print(
    "\nBlok 10 selesai dijalankan."
)


print(
    "Ringkasan hasil tersimpan di:",
    BLOCK10_DIR
)

## LOG LOSS

In [ ]:
# ============================================================
# BLOK 11 — EVALUASI AKHIR DATA TEST
# HANYA UNTUK MODEL TERBAIK PADA VALIDATION
# HASIL KONFIGURASI BERBASIS MEAN CV LOG LOSS
# DENGAN PENYIMPANAN DAN RESUME
#
# Data test hanya digunakan satu kali untuk evaluasi akhir.
# Tidak dilakukan tuning, pemilihan fitur, pemilihan threshold,
# atau pemilihan model menggunakan data test.
# ============================================================

import os
import json
import joblib

# ============================================================
# FOLDER PENYIMPANAN BLOK 11
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC
# agar checkpoint lama tidak termuat kembali.
# ============================================================

BLOCK11_DIR = os.path.join(
    "./artifacts",
    (
        f"test_evaluation_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

TABLE_DIR_BLOCK11 = os.path.join(
    BLOCK11_DIR,
    "tables"
)

PREDICTION_DIR_BLOCK11 = os.path.join(
    BLOCK11_DIR,
    "predictions"
)

FIGURE_DIR_BLOCK11 = os.path.join(
    BLOCK11_DIR,
    "figures"
)

for directory in [
    BLOCK11_DIR,
    TABLE_DIR_BLOCK11,
    PREDICTION_DIR_BLOCK11,
    FIGURE_DIR_BLOCK11
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 11:",
    BLOCK11_DIR
)

print(
    "Data test hanya digunakan untuk evaluasi akhir."
)


# ============================================================
# PATH HASIL BLOK 11
# ============================================================

FINAL_TEST_TABLE_PATH = os.path.join(
    TABLE_DIR_BLOCK11,
    "final_test_table_logloss.pkl"
)

FINAL_TEST_SUMMARY_PATH = os.path.join(
    TABLE_DIR_BLOCK11,
    "final_test_summary_logloss.pkl"
)

CONFUSION_SUMMARY_PATH = os.path.join(
    TABLE_DIR_BLOCK11,
    "confusion_matrix_summary_logloss.pkl"
)

TEST_PREDICTION_PATH = os.path.join(
    PREDICTION_DIR_BLOCK11,
    "test_predictions_logloss.npz"
)

TEST_PREDICTION_TABLE_PATH = os.path.join(
    PREDICTION_DIR_BLOCK11,
    "test_prediction_table_logloss.pkl"
)

CONFUSION_MATRIX_PATH = os.path.join(
    PREDICTION_DIR_BLOCK11,
    "confusion_matrix_logloss.npy"
)

CONFUSION_FIGURE_PATH = os.path.join(
    FIGURE_DIR_BLOCK11,
    "confusion_matrix_logloss.png"
)

CHECKPOINT_METADATA_PATH = os.path.join(
    BLOCK11_DIR,
    "block11_checkpoint_metadata_logloss.json"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block11_objects = [
    "MODEL_ORDER",
    "ALL_METRICS",
    "SELECTION_METRICS",
    "LABEL_DISPLAY_NAME",
    "THRESHOLD",
    "y_test"
]


missing_block11_objects = [
    object_name
    for object_name in required_block11_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block11_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 11: "
        f"{missing_block11_objects}"
    )


# Menyamakan indeks dan tipe target test
y_test_final = (
    pd.Series(y_test)
    .reset_index(drop=True)
    .astype(int)
)


# ============================================================
# MEMUAT HASIL BLOK 9 JIKA VARIABEL BELUM TERSEDIA
# ============================================================

required_block9_objects = [
    "final_model_data",
    "best_model_name",
    "best_model"
]


missing_block9_objects = [
    object_name
    for object_name in required_block9_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block9_objects:

    print(
        "Objek hasil Blok 9 belum tersedia. "
        "Memuat hasil Blok 9 berbasis Log Loss."
    )


    if "BLOCK9_DIR" not in globals():

        BLOCK9_DIR = os.path.join(
            "./artifacts",
            (
                f"validation_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block9_result_path = os.path.join(
        BLOCK9_DIR,
        "block9_results_logloss.joblib"
    )


    if not os.path.exists(
        block9_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 9 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block9_result_path}\n"
            "Jalankan Blok 9 terlebih dahulu."
        )


    block9_results = joblib.load(
        block9_result_path
    )


    required_block9_keys = [
        "final_model_data",
        "best_model_name",
        "best_model"
    ]


    missing_block9_keys = [
        key
        for key in required_block9_keys
        if key not in block9_results
    ]


    if missing_block9_keys:
        raise KeyError(
            "Isi hasil Blok 9 belum lengkap. "
            f"Key yang tidak ditemukan: {missing_block9_keys}"
        )


    final_model_data = (
        block9_results[
            "final_model_data"
        ]
    )


    best_model_name = (
        block9_results[
            "best_model_name"
        ]
    )


    best_model = (
        block9_results[
            "best_model"
        ]
    )


    candidate_models = (
        block9_results.get(
            "candidate_models",
            {}
        )
    )


    validation_comparison = (
        block9_results.get(
            "validation_comparison"
        )
    )


    print(
        "Hasil Blok 9 berhasil dimuat."
    )


# ============================================================
# MEMUAT HASIL BLOK 7 JIKA OBJEK PENDUKUNG BELUM TERSEDIA
# ============================================================

required_block7_objects = [
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model"
]


missing_block7_objects = [
    object_name
    for object_name in required_block7_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block7_objects:

    print(
        "Objek pendukung Blok 7 belum tersedia. "
        "Memuat hasil Blok 7 berbasis Log Loss."
    )


    if "BLOCK7_DIR" not in globals():

        BLOCK7_DIR = os.path.join(
            "./artifacts",
            (
                f"final_combination_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block7_result_path = os.path.join(
        BLOCK7_DIR,
        "block7_results_logloss.joblib"
    )


    if not os.path.exists(
        block7_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 7 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block7_result_path}"
        )


    block7_results = joblib.load(
        block7_result_path
    )


    required_block7_keys = [
        "selected_aggregation_method",
        "selected_features_by_model",
        "selected_k_by_model"
    ]


    missing_block7_keys = [
        key
        for key in required_block7_keys
        if key not in block7_results
    ]


    if missing_block7_keys:
        raise KeyError(
            "Isi hasil Blok 7 belum lengkap. "
            f"Key yang tidak ditemukan: {missing_block7_keys}"
        )


    selected_aggregation_method = (
        block7_results[
            "selected_aggregation_method"
        ]
    )


    selected_features_by_model = (
        block7_results[
            "selected_features_by_model"
        ]
    )


    selected_k_by_model = (
        block7_results[
            "selected_k_by_model"
        ]
    )


    print(
        "Hasil Blok 7 berhasil dimuat."
    )


# ============================================================
# VALIDASI MODEL TERBAIK DAN DATA TEST
# ============================================================

best_model_name = str(
    best_model_name
)


if best_model_name not in MODEL_ORDER:
    raise ValueError(
        "Model terbaik tidak dikenali: "
        f"{best_model_name}"
    )


if best_model_name not in final_model_data:
    raise KeyError(
        "Data final model terbaik tidak tersedia."
    )


if best_model_name not in selected_features_by_model:
    raise KeyError(
        "Fitur model terbaik tidak tersedia."
    )


if best_model_name not in selected_k_by_model:
    raise KeyError(
        "Top-K model terbaik tidak tersedia."
    )


best_data = (
    final_model_data[
        best_model_name
    ]
)


if "test" not in best_data:
    raise KeyError(
        "Data test model terbaik tidak tersedia."
    )


selected_best_features = list(
    selected_features_by_model[
        best_model_name
    ]
)


selected_best_k = int(
    selected_k_by_model[
        best_model_name
    ]
)


X_test_best = (
    best_data[
        "test"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(
    X_test_best
) != len(
    y_test_final
):
    raise ValueError(
        "Jumlah baris data test tidak sesuai "
        "dengan jumlah target test."
    )


if list(
    X_test_best.columns
) != selected_best_features:
    raise ValueError(
        "Urutan fitur data test tidak sesuai "
        "dengan fitur model terbaik."
    )


if len(
    selected_best_features
) != selected_best_k:
    raise ValueError(
        "Jumlah fitur model terbaik "
        "tidak sesuai dengan Top-K."
    )


if X_test_best.isna().any().any():
    raise ValueError(
        "Data test model terbaik memiliki missing value."
    )


# Pemeriksaan nilai numerik tanpa memaksa kategori menjadi float
numeric_test_columns = (
    X_test_best
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


if numeric_test_columns:

    numeric_test_values = (
        X_test_best[
            numeric_test_columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    if not np.isfinite(
        numeric_test_values
    ).all():
        raise ValueError(
            "Data test model terbaik memiliki "
            "nilai numerik NaN atau infinity."
        )


print(
    "Model terbaik:",
    best_model_name
)

print(
    "Metode agregasi:",
    selected_aggregation_method
)

print(
    "Top-K:",
    selected_best_k
)

print(
    "Jumlah data test:",
    len(
        y_test_final
    )
)


# ============================================================
# METADATA KONFIGURASI CHECKPOINT
#
# Digunakan untuk memastikan checkpoint berasal dari
# model, fitur, threshold, dan label yang sama.
# ============================================================

current_checkpoint_metadata = {
    "Skenario Pelabelan":
        str(
            LABEL_DISPLAY_NAME
        ),

    "Model":
        best_model_name,

    "Metode Agregasi":
        str(
            selected_aggregation_method
        ),

    "Top-K":
        selected_best_k,

    "Fitur":
        selected_best_features,

    "Threshold":
        float(
            THRESHOLD
        ),

    "Jumlah Data Test":
        int(
            len(
                y_test_final
            )
        )
}


# ============================================================
# MEMERIKSA CHECKPOINT BLOK 11
# ============================================================

block11_files_complete = all([
    os.path.exists(
        FINAL_TEST_TABLE_PATH
    ),
    os.path.exists(
        FINAL_TEST_SUMMARY_PATH
    ),
    os.path.exists(
        CONFUSION_SUMMARY_PATH
    ),
    os.path.exists(
        TEST_PREDICTION_PATH
    ),
    os.path.exists(
        TEST_PREDICTION_TABLE_PATH
    ),
    os.path.exists(
        CONFUSION_MATRIX_PATH
    ),
    os.path.exists(
        CHECKPOINT_METADATA_PATH
    )
])


checkpoint_metadata_matches = False


if block11_files_complete:

    with open(
        CHECKPOINT_METADATA_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        saved_checkpoint_metadata = json.load(
            file
        )


    checkpoint_metadata_matches = (
        saved_checkpoint_metadata
        == current_checkpoint_metadata
    )


    if not checkpoint_metadata_matches:

        print(
            "\nCheckpoint Blok 11 ditemukan, tetapi "
            "konfigurasinya berbeda."
        )

        print(
            "Evaluasi test akan dihitung ulang."
        )


block11_complete = (
    block11_files_complete
    and checkpoint_metadata_matches
)


# ============================================================
# MEMUAT HASIL TEST JIKA SUDAH TERSEDIA
# ============================================================

if block11_complete:

    print(
        "\nHasil evaluasi test sudah tersedia. "
        "Memuat dari checkpoint."
    )


    final_test_table = pd.read_pickle(
        FINAL_TEST_TABLE_PATH
    )


    final_test_summary = pd.read_pickle(
        FINAL_TEST_SUMMARY_PATH
    )


    confusion_matrix_summary = pd.read_pickle(
        CONFUSION_SUMMARY_PATH
    )


    test_prediction_table = pd.read_pickle(
        TEST_PREDICTION_TABLE_PATH
    )


    with np.load(
        TEST_PREDICTION_PATH
    ) as test_prediction_data:

        y_test_true_saved = np.asarray(
            test_prediction_data[
                "y_true"
            ],
            dtype=int
        )


        y_test_proba = np.asarray(
            test_prediction_data[
                "y_proba"
            ],
            dtype=float
        )


        y_test_pred = np.asarray(
            test_prediction_data[
                "y_pred"
            ],
            dtype=int
        )


    cm = np.asarray(
        np.load(
            CONFUSION_MATRIX_PATH
        ),
        dtype=int
    )


    if not np.array_equal(
        y_test_true_saved,
        y_test_final.to_numpy()
    ):
        raise ValueError(
            "Target test pada checkpoint berbeda "
            "dengan target test saat ini."
        )


    missing_metric_columns = [
        metric
        for metric in ALL_METRICS
        if metric not in final_test_table.columns
    ]


    if missing_metric_columns:
        raise ValueError(
            "Kolom metrik checkpoint test belum lengkap: "
            f"{missing_metric_columns}"
        )


    test_metrics = {
        metric:
            float(
                final_test_table
                .iloc[0][metric]
            )
        for metric in ALL_METRICS
    }


# ============================================================
# EVALUASI TEST JIKA CHECKPOINT BELUM TERSEDIA
# ============================================================

else:

    print(
        "\nMelakukan evaluasi akhir "
        "pada data test."
    )


    y_test_proba = (
        best_model
        .predict_proba(
            X_test_best
        )[:, 1]
    )


    y_test_proba = np.asarray(
        y_test_proba,
        dtype=float
    )


    (
        test_metrics,
        y_test_pred
    ) = calculate_metrics(
        y_true=y_test_final,
        y_proba=y_test_proba,
        threshold=THRESHOLD
    )


    y_test_pred = np.asarray(
        y_test_pred,
        dtype=int
    )


    # --------------------------------------------------------
    # Validasi prediksi
    # --------------------------------------------------------

    if len(
        y_test_proba
    ) != len(
        y_test_final
    ):
        raise ValueError(
            "Jumlah probabilitas test tidak sesuai."
        )


    if len(
        y_test_pred
    ) != len(
        y_test_final
    ):
        raise ValueError(
            "Jumlah prediksi test tidak sesuai."
        )


    if not np.isfinite(
        y_test_proba
    ).all():
        raise ValueError(
            "Probabilitas test mengandung "
            "NaN atau infinity."
        )


    if (
        (y_test_proba < 0).any()
        or (y_test_proba > 1).any()
    ):
        raise ValueError(
            "Probabilitas test berada di luar "
            "rentang 0 sampai 1."
        )


    if not set(
        np.unique(
            y_test_pred
        )
    ).issubset({
        0,
        1
    }):
        raise ValueError(
            "Prediksi test harus bernilai 0 atau 1."
        )


    # --------------------------------------------------------
    # Tabel lengkap hasil test
    # --------------------------------------------------------

    final_test_table = pd.DataFrame([
        {
            "Skenario Pelabelan":
                LABEL_DISPLAY_NAME,

            "Model":
                best_model_name,

            "Metode Agregasi":
                selected_aggregation_method,

            "Top-K":
                selected_best_k,

            "Fitur":
                ", ".join(
                    selected_best_features
                ),

            "Threshold":
                float(
                    THRESHOLD
                ),

            **{
                metric:
                    float(value)
                for metric, value in (
                    test_metrics.items()
                )
            }
        }
    ])


    # --------------------------------------------------------
    # Tabel ringkas hasil test
    #
    # Lima metrik utama dan Log Loss sebagai metrik tambahan.
    # --------------------------------------------------------

    final_test_summary = (
        final_test_table[
            [
                "Model",
                "Top-K",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-Score",
                "ROC-AUC",
                "Log Loss"
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_test_final,
        y_test_pred,
        labels=[
            0,
            1
        ]
    )


    if cm.shape != (
        2,
        2
    ):
        raise ValueError(
            "Ukuran confusion matrix tidak sesuai."
        )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    confusion_matrix_summary = pd.DataFrame([
        {
            "Skenario Pelabelan":
                LABEL_DISPLAY_NAME,

            "Model":
                best_model_name,

            "Threshold":
                float(
                    THRESHOLD
                ),

            "TN":
                int(tn),

            "FP":
                int(fp),

            "FN":
                int(fn),

            "TP":
                int(tp),

            "Total":
                int(
                    tn + fp + fn + tp
                )
        }
    ])


    # --------------------------------------------------------
    # Membentuk data prediksi test lengkap
    # --------------------------------------------------------

    test_prediction_table = (
        X_test_best
        .reset_index(drop=True)
        .copy()
    )


    test_prediction_table[
        "y_true"
    ] = y_test_final.to_numpy(
        dtype=int
    )


    test_prediction_table[
        "y_proba"
    ] = y_test_proba


    test_prediction_table[
        "y_pred"
    ] = y_test_pred


    test_prediction_table[
        "Jenis Prediksi"
    ] = np.select(
        [
            (
                test_prediction_table[
                    "y_true"
                ] == 0
            )
            &
            (
                test_prediction_table[
                    "y_pred"
                ] == 0
            ),

            (
                test_prediction_table[
                    "y_true"
                ] == 0
            )
            &
            (
                test_prediction_table[
                    "y_pred"
                ] == 1
            ),

            (
                test_prediction_table[
                    "y_true"
                ] == 1
            )
            &
            (
                test_prediction_table[
                    "y_pred"
                ] == 0
            ),

            (
                test_prediction_table[
                    "y_true"
                ] == 1
            )
            &
            (
                test_prediction_table[
                    "y_pred"
                ] == 1
            )
        ],

        [
            "TN",
            "FP",
            "FN",
            "TP"
        ],

        default="Tidak Diketahui"
    )


    # --------------------------------------------------------
    # Menyimpan tabel
    # --------------------------------------------------------

    final_test_table.to_pickle(
        FINAL_TEST_TABLE_PATH
    )


    final_test_table.to_csv(
        os.path.join(
            TABLE_DIR_BLOCK11,
            "final_test_table_logloss.csv"
        ),
        index=False
    )


    final_test_summary.to_pickle(
        FINAL_TEST_SUMMARY_PATH
    )


    final_test_summary.to_csv(
        os.path.join(
            TABLE_DIR_BLOCK11,
            "final_test_summary_logloss.csv"
        ),
        index=False
    )


    confusion_matrix_summary.to_pickle(
        CONFUSION_SUMMARY_PATH
    )


    confusion_matrix_summary.to_csv(
        os.path.join(
            TABLE_DIR_BLOCK11,
            "confusion_matrix_summary_logloss.csv"
        ),
        index=False
    )


    test_prediction_table.to_pickle(
        TEST_PREDICTION_TABLE_PATH
    )


    test_prediction_table.to_csv(
        os.path.join(
            PREDICTION_DIR_BLOCK11,
            "test_prediction_table_logloss.csv"
        ),
        index=False
    )


    # --------------------------------------------------------
    # Menyimpan prediksi dan confusion matrix
    # --------------------------------------------------------

    np.savez_compressed(
        TEST_PREDICTION_PATH,

        y_true=(
            y_test_final.to_numpy(
                dtype=int
            )
        ),

        y_pred=(
            y_test_pred
        ),

        y_proba=(
            y_test_proba
        )
    )


    np.save(
        CONFUSION_MATRIX_PATH,
        cm
    )


    # --------------------------------------------------------
    # Menyimpan metadata checkpoint
    # --------------------------------------------------------

    with open(
        CHECKPOINT_METADATA_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            current_checkpoint_metadata,
            file,
            ensure_ascii=False,
            indent=4
        )


    print(
        "Hasil evaluasi test berhasil disimpan."
    )


# ============================================================
# VALIDASI HASIL AKHIR YANG DIMUAT ATAU DIHITUNG
# ============================================================

if len(
    y_test_proba
) != len(
    y_test_final
):
    raise ValueError(
        "Jumlah probabilitas test akhir tidak sesuai."
    )


if len(
    y_test_pred
) != len(
    y_test_final
):
    raise ValueError(
        "Jumlah prediksi test akhir tidak sesuai."
    )


if cm.shape != (
    2,
    2
):
    raise ValueError(
        "Ukuran confusion matrix akhir tidak sesuai."
    )


if not np.isfinite(
    y_test_proba
).all():
    raise ValueError(
        "Probabilitas test akhir mengandung "
        "NaN atau infinity."
    )


# ============================================================
# MENGAMBIL NILAI CONFUSION MATRIX
# ============================================================

tn, fp, fn, tp = (
    cm.ravel()
)


if (
    tn + fp + fn + tp
) != len(
    y_test_final
):
    raise ValueError(
        "Total confusion matrix tidak sama "
        "dengan jumlah data test."
    )


# ============================================================
# MEMBENTUK ULANG TABEL PREDIKSI JIKA CHECKPOINT DIMUAT
# ============================================================

if block11_complete:

    required_prediction_columns = (
        selected_best_features
        + [
            "y_true",
            "y_proba",
            "y_pred",
            "Jenis Prediksi"
        ]
    )


    missing_prediction_columns = [
        column
        for column in required_prediction_columns
        if column not in test_prediction_table.columns
    ]


    if missing_prediction_columns:
        raise ValueError(
            "Kolom tabel prediksi test belum lengkap: "
            f"{missing_prediction_columns}"
        )


    if len(
        test_prediction_table
    ) != len(
        y_test_final
    ):
        raise ValueError(
            "Jumlah baris tabel prediksi test "
            "tidak sesuai."
        )


# ============================================================
# VISUALISASI CONFUSION MATRIX
# ============================================================

cm_display = ConfusionMatrixDisplay(
    confusion_matrix=cm,

    display_labels=[
        "Not Retained",
        "Retained"
    ]
)


cm_display.plot(
    values_format="d"
)


plt.title(
    "Confusion Matrix — "
    f"{best_model_name}\n"
    f"{LABEL_DISPLAY_NAME}"
)


plt.tight_layout()


plt.savefig(
    CONFUSION_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


plt.close()


# ============================================================
# MENYIMPAN HASIL UTAMA BLOK 11
# ============================================================

block11_results = {
    "configuration_selection_criterion":
        "Mean CV Log Loss terendah",

    "model_selection_source":
        "Data validation pada Blok 9",

    "test_usage":
        "Evaluasi akhir satu kali",

    "best_model_name":
        best_model_name,

    "best_model":
        best_model,

    "selected_aggregation_method":
        selected_aggregation_method,

    "selected_features":
        selected_best_features,

    "selected_k":
        selected_best_k,

    "best_data":
        best_data,

    "test_metrics":
        test_metrics,

    "y_test_true":
        y_test_final.to_numpy(
            dtype=int
        ),

    "y_test_proba":
        np.asarray(
            y_test_proba,
            dtype=float
        ),

    "y_test_pred":
        np.asarray(
            y_test_pred,
            dtype=int
        ),

    "final_test_table":
        final_test_table,

    "final_test_summary":
        final_test_summary,

    "confusion_matrix":
        cm,

    "confusion_matrix_summary":
        confusion_matrix_summary,

    "test_prediction_table":
        test_prediction_table
}


joblib.dump(
    block11_results,
    os.path.join(
        BLOCK11_DIR,
        "block11_results_logloss.joblib"
    )
)


# ============================================================
# METADATA BLOK 11
# ============================================================

block11_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Dasar Pemilihan Konfigurasi":
        "Mean CV Log Loss terendah",

    "Sumber Pemilihan Model":
        "Data validation pada Blok 9",

    "Penggunaan Data Test":
        (
            "Evaluasi akhir tanpa tuning, "
            "pemilihan fitur, threshold, atau model"
        ),

    "Model Terbaik":
        best_model_name,

    "Metode Agregasi":
        selected_aggregation_method,

    "Top-K":
        selected_best_k,

    "Fitur":
        selected_best_features,

    "Threshold":
        float(
            THRESHOLD
        ),

    "Jumlah Data Test":
        int(
            len(
                y_test_final
            )
        ),

    "Metrik Test":
        {
            metric:
                float(
                    test_metrics[
                        metric
                    ]
                )
            for metric in ALL_METRICS
        },

    "TN":
        int(tn),

    "FP":
        int(fp),

    "FN":
        int(fn),

    "TP":
        int(tp)
}


with open(
    os.path.join(
        BLOCK11_DIR,
        "block11_metadata_logloss.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block11_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT BLOK 11
# ============================================================

print(
    "\n"
    + "=" * 70
)


print(
    "TABEL LENGKAP EVALUASI "
    "AKHIR DATA TEST"
)


print(
    "=" * 70
)


display_without_index(
    final_test_table,
    formats={
        metric:
            "{:.4f}"
        for metric in ALL_METRICS
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS EVALUASI "
    "AKHIR DATA TEST"
)


print(
    "=" * 70
)


display_without_index(
    final_test_summary,
    formats={
        "Accuracy":
            "{:.4f}",

        "Precision":
            "{:.4f}",

        "Recall":
            "{:.4f}",

        "F1-Score":
            "{:.4f}",

        "ROC-AUC":
            "{:.4f}",

        "Log Loss":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS CONFUSION MATRIX"
)


print(
    "=" * 70
)


display_without_index(
    confusion_matrix_summary
)


print(
    f"\nTN={tn}, FP={fp}, "
    f"FN={fn}, TP={tp}"
)


print(
    "\nAccuracy :",
    f"{test_metrics['Accuracy']:.4f}"
)

print(
    "Precision:",
    f"{test_metrics['Precision']:.4f}"
)

print(
    "Recall   :",
    f"{test_metrics['Recall']:.4f}"
)

print(
    "F1-Score :",
    f"{test_metrics['F1-Score']:.4f}"
)

print(
    "ROC-AUC  :",
    f"{test_metrics['ROC-AUC']:.4f}"
)

print(
    "Log Loss :",
    f"{test_metrics['Log Loss']:.6f}"
)


print(
    "\nGambar confusion matrix tersimpan di:"
)

print(
    CONFUSION_FIGURE_PATH
)


print(
    "\nSeluruh hasil Blok 11 "
    "berhasil disimpan di:"
)

print(
    BLOCK11_DIR
)

## TEST

In [ ]:
# ============================================================
# BLOK 12 — KONSOLIDASI DAN SIMPAN SELURUH HASIL
# VERSI OPTIMASI MEAN CV LOG LOSS
#
# Blok ini tidak melakukan training, tuning, atau evaluasi ulang.
# Hasil dari Blok 3–11 dikonsolidasikan menjadi satu paket.
# ============================================================

import os
import json
import joblib
from datetime import datetime


# ============================================================
# FOLDER PENYIMPANAN BLOK 12
#
# Folder dibedakan dari eksperimen lama berbasis ROC-AUC.
# ============================================================

BLOCK12_DIR = os.path.join(
    "./artifacts",
    (
        f"final_results_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

TABLE_DIR_BLOCK12 = os.path.join(
    BLOCK12_DIR,
    "tables"
)

OBJECT_DIR_BLOCK12 = os.path.join(
    BLOCK12_DIR,
    "objects"
)

for directory in [
    BLOCK12_DIR,
    TABLE_DIR_BLOCK12,
    OBJECT_DIR_BLOCK12
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 12:",
    BLOCK12_DIR
)

print(
    "Versi hasil:",
    "Optimasi Mean CV Log Loss"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block12_objects = [
    "LABEL_DISPLAY_NAME",
    "RANDOM_STATE",
    "N_SPLITS",
    "THRESHOLD",
    "N_TRIALS_INITIAL",
    "N_TRIALS_TOPK",
    "MIN_K",
    "MAX_K",
    "MODEL_ORDER",
    "AGGREGATION_METHODS",
    "features_before_fs",
    "y_train",
    "y_val",
    "y_test"
]


missing_block12_objects = [
    name
    for name in required_block12_objects
    if (
        name not in globals()
        or globals()[name] is None
    )
]


if missing_block12_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 12: "
        f"{missing_block12_objects}"
    )


# ============================================================
# MEMASTIKAN PATH SETIAP BLOK TERSEDIA
# ============================================================

if "INITIAL_TUNING_DIR" not in globals():

    INITIAL_TUNING_DIR = os.path.join(
        "./artifacts",
        (
            f"initial_tuning_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK4_DIR" not in globals():

    BLOCK4_DIR = os.path.join(
        "./artifacts",
        (
            f"shap_ranking_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK5_DIR" not in globals():

    BLOCK5_DIR = os.path.join(
        "./artifacts",
        (
            f"aggregation_ranking_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK6_DIR" not in globals():

    BLOCK6_DIR = os.path.join(
        "./artifacts",
        (
            f"topk_tuning_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK7_DIR" not in globals():

    BLOCK7_DIR = os.path.join(
        "./artifacts",
        (
            f"final_combination_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK8_DIR" not in globals():

    BLOCK8_DIR = os.path.join(
        "./artifacts",
        (
            f"final_cv_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK9_DIR" not in globals():

    BLOCK9_DIR = os.path.join(
        "./artifacts",
        (
            f"validation_evaluation_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK10_DIR" not in globals():

    BLOCK10_DIR = os.path.join(
        "./artifacts",
        (
            f"logloss_analysis_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


if "BLOCK11_DIR" not in globals():

    BLOCK11_DIR = os.path.join(
        "./artifacts",
        (
            f"test_evaluation_logloss_"
            f"{safe_file_name(LABEL_DISPLAY_NAME)}"
        )
    )


# ============================================================
# VALIDASI FOLDER SETIAP BLOK
# ============================================================

required_stage_directories = {
    "Blok 3":
        INITIAL_TUNING_DIR,

    "Blok 4":
        BLOCK4_DIR,

    "Blok 5":
        BLOCK5_DIR,

    "Blok 6":
        BLOCK6_DIR,

    "Blok 7":
        BLOCK7_DIR,

    "Blok 8":
        BLOCK8_DIR,

    "Blok 9":
        BLOCK9_DIR,

    "Blok 10":
        BLOCK10_DIR,

    "Blok 11":
        BLOCK11_DIR
}


missing_stage_directories = {
    stage_name: stage_path
    for stage_name, stage_path in (
        required_stage_directories.items()
    )
    if not os.path.exists(
        stage_path
    )
}


if missing_stage_directories:

    missing_text = "\n".join([
        f"{stage_name}: {stage_path}"
        for stage_name, stage_path in (
            missing_stage_directories.items()
        )
    ])

    raise FileNotFoundError(
        "Folder hasil berikut belum tersedia:\n"
        f"{missing_text}"
    )


# ============================================================
# FUNGSI MEMUAT FILE WAJIB
# ============================================================

def load_joblib_required(
    path,
    description
):
    if not os.path.exists(
        path
    ):
        raise FileNotFoundError(
            f"{description} tidak ditemukan:\n{path}"
        )

    return joblib.load(
        path
    )


def read_pickle_required(
    path,
    description
):
    if not os.path.exists(
        path
    ):
        raise FileNotFoundError(
            f"{description} tidak ditemukan:\n{path}"
        )

    return pd.read_pickle(
        path
    )


def read_json_required(
    path,
    description
):
    if not os.path.exists(
        path
    ):
        raise FileNotFoundError(
            f"{description} tidak ditemukan:\n{path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(
            file
        )


# ============================================================
# MEMUAT HASIL BLOK 7
# ============================================================

required_block7_variables = [
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model",
    "best_params_by_model",
    "selected_combinations",
    "final_hyperparameter_table"
]


block7_needs_loading = any(
    name not in globals()
    or globals()[name] is None
    for name in required_block7_variables
)


if block7_needs_loading:

    print(
        "Memuat hasil Blok 7 berbasis Log Loss."
    )


    block7_results = load_joblib_required(
        os.path.join(
            BLOCK7_DIR,
            "block7_results_logloss.joblib"
        ),
        "Hasil Blok 7"
    )


    required_block7_keys = [
        "selected_aggregation_method",
        "selected_features_by_model",
        "selected_k_by_model",
        "best_params_by_model",
        "selected_combinations",
        "final_hyperparameter_table"
    ]


    missing_block7_keys = [
        key
        for key in required_block7_keys
        if key not in block7_results
    ]


    if missing_block7_keys:
        raise KeyError(
            "Hasil Blok 7 belum lengkap: "
            f"{missing_block7_keys}"
        )


    selected_aggregation_method = (
        block7_results[
            "selected_aggregation_method"
        ]
    )


    selected_features_by_model = (
        block7_results[
            "selected_features_by_model"
        ]
    )


    selected_k_by_model = (
        block7_results[
            "selected_k_by_model"
        ]
    )


    best_params_by_model = (
        block7_results[
            "best_params_by_model"
        ]
    )


    selected_combinations = (
        block7_results[
            "selected_combinations"
        ]
    )


    final_hyperparameter_table = (
        block7_results[
            "final_hyperparameter_table"
        ]
    )


    selected_logloss_by_model = (
        block7_results.get(
            "selected_logloss_by_model",
            {}
        )
    )


    selected_auc_by_model = (
        block7_results.get(
            "selected_auc_by_model",
            {}
        )
    )


# ============================================================
# MEMUAT HASIL BLOK 8
# ============================================================

if (
    "cv_fold_summary_table" not in globals()
    or cv_fold_summary_table is None
):

    print(
        "Memuat hasil Blok 8 berbasis Log Loss."
    )


    block8_results = load_joblib_required(
        os.path.join(
            BLOCK8_DIR,
            "block8_results_logloss.joblib"
        ),
        "Hasil Blok 8"
    )


    required_block8_keys = [
        "cv_fold_metric_tables",
        "cv_fold_summary_table",
        "cv_fold_summary_report"
    ]


    missing_block8_keys = [
        key
        for key in required_block8_keys
        if key not in block8_results
    ]


    if missing_block8_keys:
        raise KeyError(
            "Hasil Blok 8 belum lengkap: "
            f"{missing_block8_keys}"
        )


    cv_fold_metric_tables = (
        block8_results[
            "cv_fold_metric_tables"
        ]
    )


    cv_fold_summary_table = (
        block8_results[
            "cv_fold_summary_table"
        ]
    )


    cv_fold_summary_report = (
        block8_results[
            "cv_fold_summary_report"
        ]
    )


# ============================================================
# MEMUAT HASIL BLOK 9
# ============================================================

required_block9_variables = [
    "validation_table",
    "validation_summary",
    "validation_comparison",
    "validation_selection_summary",
    "validation_metric_winner_table",
    "best_model_name",
    "best_model",
    "final_model_data",
    "validation_predictions"
]


block9_needs_loading = any(
    name not in globals()
    or globals()[name] is None
    for name in required_block9_variables
)


if block9_needs_loading:

    print(
        "Memuat hasil Blok 9 berbasis Log Loss."
    )


    block9_results = load_joblib_required(
        os.path.join(
            BLOCK9_DIR,
            "block9_results_logloss.joblib"
        ),
        "Hasil Blok 9"
    )


    required_block9_keys = [
        "final_model_data",
        "candidate_models",
        "validation_predictions",
        "validation_table",
        "validation_summary",
        "validation_metric_winner_table",
        "validation_comparison",
        "validation_selection_summary",
        "best_model_name",
        "best_model"
    ]


    missing_block9_keys = [
        key
        for key in required_block9_keys
        if key not in block9_results
    ]


    if missing_block9_keys:
        raise KeyError(
            "Hasil Blok 9 belum lengkap: "
            f"{missing_block9_keys}"
        )


    final_model_data = (
        block9_results[
            "final_model_data"
        ]
    )


    candidate_models = (
        block9_results[
            "candidate_models"
        ]
    )


    validation_predictions = (
        block9_results[
            "validation_predictions"
        ]
    )


    validation_table = (
        block9_results[
            "validation_table"
        ]
    )


    validation_summary = (
        block9_results[
            "validation_summary"
        ]
    )


    validation_metric_winner_table = (
        block9_results[
            "validation_metric_winner_table"
        ]
    )


    validation_comparison = (
        block9_results[
            "validation_comparison"
        ]
    )


    validation_selection_summary = (
        block9_results[
            "validation_selection_summary"
        ]
    )


    best_model_name = (
        block9_results[
            "best_model_name"
        ]
    )


    best_model = (
        block9_results[
            "best_model"
        ]
    )


    xgb_logloss_results = (
        block9_results.get(
            "xgb_logloss_results",
            {}
        )
    )


    catboost_logloss_results = (
        block9_results.get(
            "catboost_logloss_results",
            {}
        )
    )


# ============================================================
# MEMUAT HASIL BLOK 10
# ============================================================

if (
    "logloss_gap_summary" not in globals()
    or logloss_gap_summary is None
):

    print(
        "Memuat hasil Blok 10 berbasis Log Loss."
    )


    block10_results = load_joblib_required(
        os.path.join(
            BLOCK10_DIR,
            "block10_results_logloss.joblib"
        ),
        "Hasil Blok 10"
    )


    required_block10_keys = [
        "logloss_gap_summary",
        "logloss_gap_report"
    ]


    missing_block10_keys = [
        key
        for key in required_block10_keys
        if key not in block10_results
    ]


    if missing_block10_keys:
        raise KeyError(
            "Hasil Blok 10 belum lengkap: "
            f"{missing_block10_keys}"
        )


    logloss_gap_summary = (
        block10_results[
            "logloss_gap_summary"
        ]
    )


    logloss_gap_report = (
        block10_results[
            "logloss_gap_report"
        ]
    )


    logloss_curve_tables = (
        block10_results.get(
            "logloss_curve_tables",
            {}
        )
    )


# ============================================================
# MEMUAT HASIL BLOK 11
# ============================================================

required_block11_variables = [
    "final_test_table",
    "final_test_summary",
    "confusion_matrix_summary",
    "y_test_pred",
    "y_test_proba",
    "test_prediction_table"
]


block11_needs_loading = any(
    name not in globals()
    or globals()[name] is None
    for name in required_block11_variables
)


if block11_needs_loading:

    print(
        "Memuat hasil Blok 11 berbasis Log Loss."
    )


    block11_results = load_joblib_required(
        os.path.join(
            BLOCK11_DIR,
            "block11_results_logloss.joblib"
        ),
        "Hasil Blok 11"
    )


    required_block11_keys = [
        "final_test_table",
        "final_test_summary",
        "confusion_matrix_summary",
        "y_test_pred",
        "y_test_proba",
        "test_prediction_table",
        "best_model_name",
        "best_model"
    ]


    missing_block11_keys = [
        key
        for key in required_block11_keys
        if key not in block11_results
    ]


    if missing_block11_keys:
        raise KeyError(
            "Hasil Blok 11 belum lengkap: "
            f"{missing_block11_keys}"
        )


    final_test_table = (
        block11_results[
            "final_test_table"
        ]
    )


    final_test_summary = (
        block11_results[
            "final_test_summary"
        ]
    )


    confusion_matrix_summary = (
        block11_results[
            "confusion_matrix_summary"
        ]
    )


    y_test_pred = np.asarray(
        block11_results[
            "y_test_pred"
        ],
        dtype=int
    )


    y_test_proba = np.asarray(
        block11_results[
            "y_test_proba"
        ],
        dtype=float
    )


    test_prediction_table = (
        block11_results[
            "test_prediction_table"
        ]
    )


    test_metrics = (
        block11_results.get(
            "test_metrics",
            {}
        )
    )


    y_test_true_saved = np.asarray(
        block11_results.get(
            "y_test_true",
            y_test
        ),
        dtype=int
    )


    # Mengutamakan model yang tersimpan pada hasil test
    best_model_name = str(
        block11_results[
            "best_model_name"
        ]
    )


    best_model = (
        block11_results[
            "best_model"
        ]
    )


else:

    y_test_true_saved = np.asarray(
        y_test,
        dtype=int
    )


# ============================================================
# MEMUAT TABEL BLOK 3–6
# ============================================================

# ------------------------------------------------------------
# BLOK 3 — TUNING AWAL
# ------------------------------------------------------------

if (
    "initial_tuning_summary" not in globals()
    or initial_tuning_summary is None
):

    initial_tuning_summary = read_pickle_required(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_tuning_summary.pkl"
        ),
        "Ringkasan tuning awal"
    )


if (
    "initial_tuning_top5" not in globals()
    or initial_tuning_top5 is None
):

    initial_tuning_top5 = read_pickle_required(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_tuning_top5_semua_model.pkl"
        ),
        "Top-5 tuning awal"
    )


if (
    "initial_best_params" not in globals()
    or initial_best_params is None
):

    initial_best_params = read_json_required(
        os.path.join(
            INITIAL_TUNING_DIR,
            "initial_best_params.json"
        ),
        "Hyperparameter terbaik tuning awal"
    )


# ------------------------------------------------------------
# BLOK 4 — SHAP
# ------------------------------------------------------------

if (
    "importance_all_models" not in globals()
    or importance_all_models is None
):

    importance_all_models = read_pickle_required(
        os.path.join(
            BLOCK4_DIR,
            "tables",
            "importance_all_models_logloss.pkl"
        ),
        "Tabel SHAP seluruh model"
    )


if (
    "importance_top5_all_models" not in globals()
    or importance_top5_all_models is None
):

    importance_top5_all_models = read_pickle_required(
        os.path.join(
            BLOCK4_DIR,
            "tables",
            "importance_top5_all_models_logloss.pkl"
        ),
        "Top-5 SHAP seluruh model"
    )


# ------------------------------------------------------------
# BLOK 5 — AGREGASI RANKING
# ------------------------------------------------------------

if (
    "aggregation_display" not in globals()
    or aggregation_display is None
):

    aggregation_display = read_pickle_required(
        os.path.join(
            BLOCK5_DIR,
            "aggregation_display_logloss.pkl"
        ),
        "Tabel agregasi ranking"
    )


if (
    "aggregation_order_table" not in globals()
    or aggregation_order_table is None
):

    aggregation_order_table = read_pickle_required(
        os.path.join(
            BLOCK5_DIR,
            "aggregation_order_table_logloss.pkl"
        ),
        "Urutan fitur agregasi"
    )


# ------------------------------------------------------------
# BLOK 6 — TUNING TOP-K
# ------------------------------------------------------------

if (
    "topk_best_full" not in globals()
    or topk_best_full is None
):

    topk_best_full = read_pickle_required(
        os.path.join(
            BLOCK6_DIR,
            "topk_logloss_best_full.pkl"
        ),
        "Hasil terbaik seluruh Top-K"
    )


if (
    "topk_best_summary" not in globals()
    or topk_best_summary is None
):

    topk_best_summary = read_pickle_required(
        os.path.join(
            BLOCK6_DIR,
            "topk_logloss_best_summary.pkl"
        ),
        "Ringkasan hasil Top-K"
    )


# ============================================================
# VALIDASI KONSISTENSI HASIL
# ============================================================

best_model_name = str(
    best_model_name
)


if best_model_name not in MODEL_ORDER:
    raise ValueError(
        "Model terbaik tidak dikenali: "
        f"{best_model_name}"
    )


for model_name in MODEL_ORDER:

    if model_name not in selected_features_by_model:
        raise KeyError(
            f"Fitur final {model_name} tidak tersedia."
        )


    if model_name not in selected_k_by_model:
        raise KeyError(
            f"Top-K final {model_name} tidak tersedia."
        )


    if model_name not in best_params_by_model:
        raise KeyError(
            f"Hyperparameter final {model_name} tidak tersedia."
        )


    if (
        len(
            selected_features_by_model[
                model_name
            ]
        )
        != int(
            selected_k_by_model[
                model_name
            ]
        )
    ):
        raise ValueError(
            f"Jumlah fitur {model_name} tidak sesuai "
            "dengan Top-K."
        )


if len(
    y_test_true_saved
) != len(
    y_test_pred
):
    raise ValueError(
        "Jumlah y_true dan y_pred test tidak sama."
    )


if len(
    y_test_true_saved
) != len(
    y_test_proba
):
    raise ValueError(
        "Jumlah y_true dan y_proba test tidak sama."
    )


if not np.isfinite(
    y_test_proba
).all():
    raise ValueError(
        "Probabilitas test mengandung NaN atau infinity."
    )


if (
    (y_test_proba < 0).any()
    or (y_test_proba > 1).any()
):
    raise ValueError(
        "Probabilitas test berada di luar rentang 0–1."
    )


# ============================================================
# PREDIKSI FINAL TEST
# ============================================================

final_test_predictions = {
    "y_true":
        np.asarray(
            y_test_true_saved,
            dtype=int
        ),

    "y_pred":
        np.asarray(
            y_test_pred,
            dtype=int
        ),

    "y_proba":
        np.asarray(
            y_test_proba,
            dtype=float
        )
}


# ============================================================
# LOKASI FILE LENGKAP SETIAP TAHAP
# ============================================================

result_paths = {
    "Blok 3 - Tuning Awal Log Loss":
        INITIAL_TUNING_DIR,

    "Blok 4 - SHAP":
        BLOCK4_DIR,

    "Blok 5 - Agregasi Ranking":
        BLOCK5_DIR,

    "Blok 6 - Tuning Top-K Log Loss":
        BLOCK6_DIR,

    "Blok 7 - Kombinasi Terpilih":
        BLOCK7_DIR,

    "Blok 8 - Cross-Validation":
        BLOCK8_DIR,

    "Blok 9 - Validation":
        BLOCK9_DIR,

    "Blok 10 - Analisis Log Loss":
        BLOCK10_DIR,

    "Blok 11 - Evaluasi Test":
        BLOCK11_DIR
}


# ============================================================
# OBJEK MODELING RESULTS
#
# Objek besar seperti:
# - seluruh Optuna study;
# - seluruh tabel trial;
# - seluruh nilai SHAP;
#
# tidak diduplikasi ke paket ini.
# Lokasi lengkapnya dicatat pada Result Paths.
# ============================================================

created_at = datetime.now().isoformat()


modeling_results = {
    "Label":
        LABEL_DISPLAY_NAME,

    "Created At":
        created_at,

    "Optimization Objective":
        "Mean CV Log Loss terendah",

    # --------------------------------------------------------
    # Konfigurasi umum
    # --------------------------------------------------------

    "Configuration": {
        "Random State":
            int(
                RANDOM_STATE
            ),

        "Number of CV Folds":
            int(
                N_SPLITS
            ),

        "Threshold":
            float(
                THRESHOLD
            ),

        "Initial Trials":
            int(
                N_TRIALS_INITIAL
            ),

        "Top-K Trials":
            int(
                N_TRIALS_TOPK
            ),

        "Minimum K":
            int(
                MIN_K
            ),

        "Maximum K":
            int(
                MAX_K
            ),

        "Models":
            list(
                MODEL_ORDER
            ),

        "Aggregation Methods":
            list(
                AGGREGATION_METHODS
            ),

        "Features Before Selection":
            list(
                features_before_fs
            )
    },

    # --------------------------------------------------------
    # Tuning awal
    # --------------------------------------------------------

    "Initial Tuning Top 5":
        initial_tuning_top5,

    "Initial Tuning Summary":
        initial_tuning_summary,

    "Initial Best Parameters":
        initial_best_params,

    # --------------------------------------------------------
    # SHAP dan agregasi ranking
    # --------------------------------------------------------

    "SHAP Importance All Models":
        importance_all_models,

    "SHAP Importance Top 5":
        importance_top5_all_models,

    "Aggregation Display":
        aggregation_display,

    "Aggregation Order Table":
        aggregation_order_table,

    # --------------------------------------------------------
    # Tuning Top-K
    # --------------------------------------------------------

    "Top-K Best Full":
        topk_best_full,

    "Top-K Best Summary":
        topk_best_summary,

    # --------------------------------------------------------
    # Kombinasi terpilih
    # --------------------------------------------------------

    "Selected Aggregation Method":
        selected_aggregation_method,

    "Selected Combinations":
        selected_combinations,

    "Selected Features by Model":
        selected_features_by_model,

    "Selected K by Model":
        selected_k_by_model,

    "Best Parameters by Model":
        best_params_by_model,

    "Final Hyperparameter Table":
        final_hyperparameter_table,

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------

    "CV Fold Metric Tables":
        cv_fold_metric_tables,

    "CV Fold Summary":
        cv_fold_summary_table,

    "CV Fold Report":
        cv_fold_summary_report,

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    "Validation Predictions":
        validation_predictions,

    "Validation Table":
        validation_table,

    "Validation Summary":
        validation_summary,

    "Validation Comparison":
        validation_comparison,

    "Validation Selection Summary":
        validation_selection_summary,

    "Validation Metric Winners":
        validation_metric_winner_table,

    "Best Model Name":
        best_model_name,

    # --------------------------------------------------------
    # Analisis Log Loss
    # --------------------------------------------------------

    "Log Loss Gap Summary":
        logloss_gap_summary,

    "Log Loss Gap Report":
        logloss_gap_report,

    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    "Final Test Table":
        final_test_table,

    "Final Test Summary":
        final_test_summary,

    "Final Test Predictions":
        final_test_predictions,

    "Confusion Matrix Summary":
        confusion_matrix_summary,

    "Test Prediction Table":
        test_prediction_table,

    # --------------------------------------------------------
    # Model terbaik
    # --------------------------------------------------------

    "Best Model Object":
        best_model,

    # --------------------------------------------------------
    # Lokasi file lengkap
    # --------------------------------------------------------

    "Result Paths":
        result_paths
}


# ============================================================
# MENYIMPAN MODELING RESULTS
# ============================================================

MODELING_RESULTS_PATH = os.path.join(
    OBJECT_DIR_BLOCK12,
    "modeling_results_logloss.joblib"
)


joblib.dump(
    modeling_results,
    MODELING_RESULTS_PATH,
    compress=3
)


# ============================================================
# MENYIMPAN MODEL TERBAIK SECARA TERPISAH
# ============================================================

BEST_MODEL_PATH = os.path.join(
    OBJECT_DIR_BLOCK12,
    (
        f"best_model_logloss_"
        f"{safe_file_name(best_model_name)}.joblib"
    )
)


joblib.dump(
    best_model,
    BEST_MODEL_PATH,
    compress=3
)


# ============================================================
# MENYIMPAN PREDIKSI TEST SECARA TERPISAH
# ============================================================

FINAL_TEST_PREDICTION_PATH = os.path.join(
    OBJECT_DIR_BLOCK12,
    "final_test_predictions_logloss.npz"
)


np.savez_compressed(
    FINAL_TEST_PREDICTION_PATH,

    y_true=(
        final_test_predictions[
            "y_true"
        ]
    ),

    y_pred=(
        final_test_predictions[
            "y_pred"
        ]
    ),

    y_proba=(
        final_test_predictions[
            "y_proba"
        ]
    )
)


# ============================================================
# MENYIMPAN KONFIGURASI FINAL SEBAGAI JSON
# ============================================================

final_configuration = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Objective Optimasi":
        "Mean CV Log Loss terendah",

    "Metode Agregasi Terpilih":
        selected_aggregation_method,

    "Model Terbaik":
        best_model_name,

    "Top-K per Model":
        {
            model_name:
                int(
                    selected_k_by_model[
                        model_name
                    ]
                )
            for model_name in MODEL_ORDER
        },

    "Fitur per Model":
        {
            model_name:
                list(
                    selected_features_by_model[
                        model_name
                    ]
                )
            for model_name in MODEL_ORDER
        },

    "Hyperparameter per Model":
        best_params_by_model,

    "Threshold":
        float(
            THRESHOLD
        )
}


FINAL_CONFIGURATION_PATH = os.path.join(
    OBJECT_DIR_BLOCK12,
    "final_configuration_logloss.json"
)


with open(
    FINAL_CONFIGURATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_configuration,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MENYIMPAN TABEL UTAMA UNTUK LAPORAN
# ============================================================

report_tables = {
    "01_initial_tuning_summary":
        initial_tuning_summary,

    "02_initial_tuning_top5":
        initial_tuning_top5,

    "03_shap_importance_all_models":
        importance_all_models,

    "04_shap_importance_top5":
        importance_top5_all_models,

    "05_aggregation_display":
        aggregation_display,

    "06_aggregation_order":
        aggregation_order_table,

    "07_topk_best_full":
        topk_best_full,

    "08_topk_best_summary":
        topk_best_summary,

    "09_selected_combinations":
        selected_combinations,

    "10_final_hyperparameters":
        final_hyperparameter_table,

    "11_cv_summary":
        cv_fold_summary_table,

    "12_cv_report":
        cv_fold_summary_report,

    "13_validation_table":
        validation_table,

    "14_validation_summary":
        validation_summary,

    "15_validation_metric_winners":
        validation_metric_winner_table,

    "16_validation_selection":
        validation_selection_summary,

    "17_logloss_gap_full":
        logloss_gap_summary,

    "18_logloss_gap_report":
        logloss_gap_report,

    "19_final_test":
        final_test_table,

    "20_final_test_summary":
        final_test_summary,

    "21_confusion_matrix":
        confusion_matrix_summary
}


for file_name, dataframe in (
    report_tables.items()
):

    if not isinstance(
        dataframe,
        pd.DataFrame
    ):
        raise TypeError(
            f"{file_name} bukan DataFrame."
        )


    dataframe.to_csv(
        os.path.join(
            TABLE_DIR_BLOCK12,
            f"{file_name}.csv"
        ),
        index=False
    )


    dataframe.to_pickle(
        os.path.join(
            TABLE_DIR_BLOCK12,
            f"{file_name}.pkl"
        )
    )


# ============================================================
# MEMBUAT MANIFEST FILE
# ============================================================

manifest_rows = []


for stage_name, stage_path in (
    result_paths.items()
):

    manifest_rows.append({
        "Tahap":
            stage_name,

        "Folder":
            stage_path,

        "Tersedia":
            bool(
                os.path.exists(
                    stage_path
                )
            )
    })


result_manifest = pd.DataFrame(
    manifest_rows
)


RESULT_MANIFEST_PATH = os.path.join(
    BLOCK12_DIR,
    "result_manifest_logloss.csv"
)


result_manifest.to_csv(
    RESULT_MANIFEST_PATH,
    index=False
)


# ============================================================
# METADATA FINAL
# ============================================================

final_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Waktu Penyimpanan":
        created_at,

    "Objective Optimasi":
        "Mean CV Log Loss terendah",

    "Dasar Pemilihan Model":
        (
            "Jumlah kemenangan lima metrik validation, "
            "kemudian ROC-AUC tertinggi dan "
            "Log Loss terendah sebagai tie-break"
        ),

    "Metode Agregasi Terpilih":
        selected_aggregation_method,

    "Model Terbaik":
        best_model_name,

    "Top-K Model Terbaik":
        int(
            selected_k_by_model[
                best_model_name
            ]
        ),

    "Fitur Model Terbaik":
        list(
            selected_features_by_model[
                best_model_name
            ]
        ),

    "Threshold":
        float(
            THRESHOLD
        ),

    "Jumlah Data Train":
        int(
            len(
                y_train
            )
        ),

    "Jumlah Data Validation":
        int(
            len(
                y_val
            )
        ),

    "Jumlah Data Test":
        int(
            len(
                y_test_true_saved
            )
        ),

    "Modeling Results File":
        MODELING_RESULTS_PATH,

    "Best Model File":
        BEST_MODEL_PATH,

    "Final Test Prediction File":
        FINAL_TEST_PREDICTION_PATH,

    "Final Configuration File":
        FINAL_CONFIGURATION_PATH,

    "Result Manifest File":
        RESULT_MANIFEST_PATH,

    "Folder Hasil Setiap Blok":
        result_paths
}


FINAL_METADATA_PATH = os.path.join(
    BLOCK12_DIR,
    "final_metadata_logloss.json"
)


with open(
    FINAL_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# VALIDASI FILE HASIL FINAL
# ============================================================

required_final_files = [
    MODELING_RESULTS_PATH,
    BEST_MODEL_PATH,
    FINAL_TEST_PREDICTION_PATH,
    FINAL_CONFIGURATION_PATH,
    FINAL_METADATA_PATH,
    RESULT_MANIFEST_PATH
]


missing_final_files = [
    path
    for path in required_final_files
    if not os.path.exists(
        path
    )
]


if missing_final_files:
    raise FileNotFoundError(
        "Terdapat hasil final yang belum tersimpan:\n"
        + "\n".join(
            missing_final_files
        )
    )


# ============================================================
# MEMUAT ULANG FILE UTAMA SEBAGAI VALIDASI
# ============================================================

modeling_results_check = joblib.load(
    MODELING_RESULTS_PATH
)


if (
    modeling_results_check[
        "Best Model Name"
    ]
    != best_model_name
):
    raise ValueError(
        "Model terbaik pada file hasil tidak sesuai."
    )


with np.load(
    FINAL_TEST_PREDICTION_PATH
) as prediction_check:

    if not np.array_equal(
        prediction_check[
            "y_true"
        ],
        final_test_predictions[
            "y_true"
        ]
    ):
        raise ValueError(
            "y_true prediksi final tidak sesuai."
        )


    if not np.array_equal(
        prediction_check[
            "y_pred"
        ],
        final_test_predictions[
            "y_pred"
        ]
    ):
        raise ValueError(
            "y_pred prediksi final tidak sesuai."
        )


    if not np.allclose(
        prediction_check[
            "y_proba"
        ],
        final_test_predictions[
            "y_proba"
        ]
    ):
        raise ValueError(
            "y_proba prediksi final tidak sesuai."
        )


del modeling_results_check


# ============================================================
# OUTPUT BLOK 12
# ============================================================

print(
    "\n"
    + "=" * 70
)


print(
    "SELURUH HASIL BERHASIL DIKONSOLIDASIKAN"
)


print(
    "=" * 70
)


print(
    "\nSkenario pelabelan:",
    LABEL_DISPLAY_NAME
)


print(
    "Objective optimasi:",
    "Mean CV Log Loss terendah"
)


print(
    "Metode agregasi terpilih:",
    selected_aggregation_method
)


print(
    "Model terbaik:",
    best_model_name
)


print(
    "Top-K model terbaik:",
    selected_k_by_model[
        best_model_name
    ]
)


print(
    "\nFile modeling_results:"
)

print(
    MODELING_RESULTS_PATH
)


print(
    "\nFile model terbaik:"
)

print(
    BEST_MODEL_PATH
)


print(
    "\nFile prediksi test:"
)

print(
    FINAL_TEST_PREDICTION_PATH
)


print(
    "\nFile konfigurasi final:"
)

print(
    FINAL_CONFIGURATION_PATH
)


print(
    "\nFolder tabel laporan:"
)

print(
    TABLE_DIR_BLOCK12
)


print(
    "\nSeluruh hasil final tersimpan di:"
)

print(
    BLOCK12_DIR
)


print(
    "\nObjek utama untuk tahap berikutnya:"
)

print(
    "- modeling_results"
)

print(
    "- best_model"
)

print(
    "- best_model_name"
)

print(
    "- final_model_data[best_model_name]['test']"
)

print(
    "- y_test_true_saved"
)

print(
    "- y_test_proba"
)

print(
    "- y_test_pred"
)

print(
    "- test_prediction_table"
)

# Error Analysis

In [ ]:
# ============================================================
# BLOK 13 — ERROR ANALYSIS BERBASIS SHAP
# FALSE POSITIVE DAN FALSE NEGATIVE
#
# HASIL MODEL DENGAN KONFIGURASI BERBASIS
# MEAN CV LOG LOSS TERENDAH
#
# False Positive:
# nilai SHAP positif mendorong prediksi menuju kelas Retained.
#
# False Negative:
# nilai SHAP negatif mendorong prediksi menuju kelas
# Not Retained.
#
# Analisis ini bersifat post-hoc dan tidak digunakan kembali
# untuk mengubah model, fitur, threshold, atau hyperparameter.
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

from catboost import Pool


# ============================================================
# FOLDER PENYIMPANAN BLOK 13
# ============================================================

BLOCK13_DIR = os.path.join(
    "./artifacts",
    (
        f"error_analysis_logloss_"
        f"{safe_file_name(LABEL_DISPLAY_NAME)}"
    )
)

TABLE_DIR_BLOCK13 = os.path.join(
    BLOCK13_DIR,
    "tables"
)

FIGURE_DIR_BLOCK13 = os.path.join(
    BLOCK13_DIR,
    "figures"
)

DATA_DIR_BLOCK13 = os.path.join(
    BLOCK13_DIR,
    "data"
)

for directory in [
    BLOCK13_DIR,
    TABLE_DIR_BLOCK13,
    FIGURE_DIR_BLOCK13,
    DATA_DIR_BLOCK13
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print(
    "Folder hasil Blok 13:",
    BLOCK13_DIR
)

print(
    "Analisis dilakukan setelah evaluasi akhir data test."
)


# ============================================================
# PATH FILE BLOK 13
# ============================================================

TEST_SHAP_PATH = os.path.join(
    DATA_DIR_BLOCK13,
    "test_shap_values_logloss.npy"
)

SHAP_METADATA_PATH = os.path.join(
    DATA_DIR_BLOCK13,
    "test_shap_metadata_logloss.json"
)

ERROR_DATA_PATH = os.path.join(
    DATA_DIR_BLOCK13,
    "error_data_logloss.pkl"
)

FP_DATA_PATH = os.path.join(
    DATA_DIR_BLOCK13,
    "false_positive_data_logloss.pkl"
)

FN_DATA_PATH = os.path.join(
    DATA_DIR_BLOCK13,
    "false_negative_data_logloss.pkl"
)

FP_SUMMARY_PATH = os.path.join(
    TABLE_DIR_BLOCK13,
    "false_positive_shap_summary_logloss.pkl"
)

FN_SUMMARY_PATH = os.path.join(
    TABLE_DIR_BLOCK13,
    "false_negative_shap_summary_logloss.pkl"
)

ERROR_COUNT_PATH = os.path.join(
    TABLE_DIR_BLOCK13,
    "error_count_summary_logloss.pkl"
)

FP_FIGURE_PATH = os.path.join(
    FIGURE_DIR_BLOCK13,
    "false_positive_shap_contribution_logloss.png"
)

FN_FIGURE_PATH = os.path.join(
    FIGURE_DIR_BLOCK13,
    "false_negative_shap_contribution_logloss.png"
)


# ============================================================
# PEMERIKSAAN OBJEK DASAR
# ============================================================

required_block13_objects = [
    "LABEL_DISPLAY_NAME",
    "MODEL_ORDER",
    "cat_cols"
]


missing_block13_objects = [
    object_name
    for object_name in required_block13_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block13_objects:
    raise NameError(
        "Objek berikut belum tersedia untuk Blok 13: "
        f"{missing_block13_objects}"
    )


# ============================================================
# MEMUAT HASIL BLOK 7 JIKA DIPERLUKAN
# ============================================================

required_block7_objects = [
    "selected_aggregation_method",
    "selected_features_by_model",
    "selected_k_by_model"
]


missing_block7_objects = [
    object_name
    for object_name in required_block7_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block7_objects:

    print(
        "Objek Blok 7 belum tersedia. "
        "Memuat hasil Blok 7 berbasis Log Loss."
    )


    if "BLOCK7_DIR" not in globals():

        BLOCK7_DIR = os.path.join(
            "./artifacts",
            (
                f"final_combination_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block7_result_path = os.path.join(
        BLOCK7_DIR,
        "block7_results_logloss.joblib"
    )


    if not os.path.exists(
        block7_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 7 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block7_result_path}"
        )


    block7_results = joblib.load(
        block7_result_path
    )


    required_block7_keys = [
        "selected_aggregation_method",
        "selected_features_by_model",
        "selected_k_by_model"
    ]


    missing_block7_keys = [
        key
        for key in required_block7_keys
        if key not in block7_results
    ]


    if missing_block7_keys:
        raise KeyError(
            "Isi hasil Blok 7 belum lengkap: "
            f"{missing_block7_keys}"
        )


    selected_aggregation_method = (
        block7_results[
            "selected_aggregation_method"
        ]
    )


    selected_features_by_model = (
        block7_results[
            "selected_features_by_model"
        ]
    )


    selected_k_by_model = (
        block7_results[
            "selected_k_by_model"
        ]
    )


# ============================================================
# MEMUAT HASIL BLOK 9 JIKA DIPERLUKAN
# ============================================================

required_block9_objects = [
    "best_model_name",
    "best_model",
    "final_model_data"
]


missing_block9_objects = [
    object_name
    for object_name in required_block9_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block9_objects:

    print(
        "Objek Blok 9 belum tersedia. "
        "Memuat hasil Blok 9 berbasis Log Loss."
    )


    if "BLOCK9_DIR" not in globals():

        BLOCK9_DIR = os.path.join(
            "./artifacts",
            (
                f"validation_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block9_result_path = os.path.join(
        BLOCK9_DIR,
        "block9_results_logloss.joblib"
    )


    if not os.path.exists(
        block9_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 9 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block9_result_path}"
        )


    block9_results = joblib.load(
        block9_result_path
    )


    required_block9_keys = [
        "best_model_name",
        "best_model",
        "final_model_data"
    ]


    missing_block9_keys = [
        key
        for key in required_block9_keys
        if key not in block9_results
    ]


    if missing_block9_keys:
        raise KeyError(
            "Isi hasil Blok 9 belum lengkap: "
            f"{missing_block9_keys}"
        )


    best_model_name = (
        block9_results[
            "best_model_name"
        ]
    )


    best_model = (
        block9_results[
            "best_model"
        ]
    )


    final_model_data = (
        block9_results[
            "final_model_data"
        ]
    )


# ============================================================
# MEMUAT HASIL BLOK 11 JIKA DIPERLUKAN
# ============================================================

required_block11_objects = [
    "y_test_proba",
    "y_test_pred",
    "test_prediction_table"
]


missing_block11_objects = [
    object_name
    for object_name in required_block11_objects
    if (
        object_name not in globals()
        or globals()[object_name] is None
    )
]


if missing_block11_objects:

    print(
        "Objek Blok 11 belum tersedia. "
        "Memuat hasil Blok 11 berbasis Log Loss."
    )


    if "BLOCK11_DIR" not in globals():

        BLOCK11_DIR = os.path.join(
            "./artifacts",
            (
                f"test_evaluation_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    block11_result_path = os.path.join(
        BLOCK11_DIR,
        "block11_results_logloss.joblib"
    )


    if not os.path.exists(
        block11_result_path
    ):
        raise FileNotFoundError(
            "Hasil Blok 11 berbasis Log Loss "
            "tidak ditemukan pada path:\n"
            f"{block11_result_path}"
        )


    block11_results = joblib.load(
        block11_result_path
    )


    required_block11_keys = [
        "y_test_proba",
        "y_test_pred",
        "test_prediction_table"
    ]


    missing_block11_keys = [
        key
        for key in required_block11_keys
        if key not in block11_results
    ]


    if missing_block11_keys:
        raise KeyError(
            "Isi hasil Blok 11 belum lengkap: "
            f"{missing_block11_keys}"
        )


    y_test_proba = np.asarray(
        block11_results[
            "y_test_proba"
        ],
        dtype=float
    )


    y_test_pred = np.asarray(
        block11_results[
            "y_test_pred"
        ],
        dtype=int
    )


    test_prediction_table = (
        block11_results[
            "test_prediction_table"
        ]
    )


    y_test_true = np.asarray(
        block11_results.get(
            "y_test_true",
            test_prediction_table[
                "y_true"
            ].to_numpy()
        ),
        dtype=int
    )


    if "best_model_name" in block11_results:

        best_model_name = str(
            block11_results[
                "best_model_name"
            ]
        )


    if "best_model" in block11_results:

        best_model = (
            block11_results[
                "best_model"
            ]
        )


else:

    if (
        isinstance(
            test_prediction_table,
            pd.DataFrame
        )
        and "y_true" in test_prediction_table.columns
    ):

        y_test_true = (
            test_prediction_table[
                "y_true"
            ]
            .to_numpy(
                dtype=int
            )
        )


    elif "y_test" in globals():

        y_test_true = np.asarray(
            y_test,
            dtype=int
        )


    else:

        raise NameError(
            "Target test tidak tersedia."
        )


# ============================================================
# VALIDASI MODEL TERBAIK DAN DATA TEST
# ============================================================

best_model_name = str(
    best_model_name
)


if best_model_name not in MODEL_ORDER:
    raise ValueError(
        "Model terbaik tidak dikenali: "
        f"{best_model_name}"
    )


if best_model_name not in final_model_data:
    raise KeyError(
        "Data model terbaik tidak tersedia."
    )


if best_model_name not in selected_features_by_model:
    raise KeyError(
        "Fitur model terbaik tidak tersedia."
    )


if best_model_name not in selected_k_by_model:
    raise KeyError(
        "Top-K model terbaik tidak tersedia."
    )


best_data = (
    final_model_data[
        best_model_name
    ]
)


if "test" not in best_data:
    raise KeyError(
        "Data test model terbaik tidak tersedia."
    )


selected_features_best = list(
    selected_features_by_model[
        best_model_name
    ]
)


selected_k_best = int(
    selected_k_by_model[
        best_model_name
    ]
)


X_test_best = (
    best_data[
        "test"
    ]
    .copy()
    .reset_index(drop=True)
)


y_test_true = np.asarray(
    y_test_true,
    dtype=int
)

y_test_proba = np.asarray(
    y_test_proba,
    dtype=float
)

y_test_pred = np.asarray(
    y_test_pred,
    dtype=int
)


if list(
    X_test_best.columns
) != selected_features_best:
    raise ValueError(
        "Urutan fitur data test tidak sesuai "
        "dengan fitur model terbaik."
    )


if len(
    selected_features_best
) != selected_k_best:
    raise ValueError(
        "Jumlah fitur model terbaik "
        "tidak sesuai dengan Top-K."
    )


expected_test_length = len(
    X_test_best
)


if len(
    y_test_true
) != expected_test_length:
    raise ValueError(
        "Jumlah data test dan target test tidak sesuai."
    )


if len(
    y_test_proba
) != expected_test_length:
    raise ValueError(
        "Jumlah data test dan probabilitas tidak sesuai."
    )


if len(
    y_test_pred
) != expected_test_length:
    raise ValueError(
        "Jumlah data test dan prediksi tidak sesuai."
    )


if X_test_best.isna().any().any():
    raise ValueError(
        "Data test model terbaik memiliki missing value."
    )


if not np.isfinite(
    y_test_proba
).all():
    raise ValueError(
        "Probabilitas test mengandung NaN atau infinity."
    )


if (
    (y_test_proba < 0).any()
    or (y_test_proba > 1).any()
):
    raise ValueError(
        "Probabilitas test berada di luar rentang 0–1."
    )


if not set(
    np.unique(
        y_test_true
    )
).issubset({
    0,
    1
}):
    raise ValueError(
        "Target test harus bernilai 0 atau 1."
    )


if not set(
    np.unique(
        y_test_pred
    )
).issubset({
    0,
    1
}):
    raise ValueError(
        "Prediksi test harus bernilai 0 atau 1."
    )


if (
    "features_before_fs" in globals()
    and selected_k_best
    == len(
        features_before_fs
    )
):

    feature_scenario_text = (
        "Seluruh Fitur"
    )


else:

    feature_scenario_text = (
        f"Top-{selected_k_best} Fitur"
    )


print(
    "Model terbaik:",
    best_model_name
)

print(
    "Metode agregasi:",
    selected_aggregation_method
)

print(
    "Top-K:",
    selected_k_best
)

print(
    "Jumlah data test:",
    expected_test_length
)


# ============================================================
# FUNGSI PERHITUNGAN SHAP RANDOM FOREST
# ============================================================

def calculate_rf_shap_test(
    model,
    X_data
):

    explainer = shap.TreeExplainer(
        model
    )


    try:

        shap_output = explainer(
            X_data,
            check_additivity=False
        )


        shap_values = (
            shap_output.values
        )


    except Exception:

        shap_values = (
            explainer.shap_values(
                X_data
            )
        )


    if isinstance(
        shap_values,
        list
    ):

        if len(
            shap_values
        ) == 2:

            shap_values = (
                shap_values[1]
            )


        else:

            shap_values = (
                shap_values[0]
            )


    shap_values = np.asarray(
        shap_values
    )


    if shap_values.ndim == 3:

        if (
            shap_values.shape[0]
            == len(X_data)
            and shap_values.shape[1]
            == X_data.shape[1]
            and shap_values.shape[2] >= 2
        ):

            shap_values = (
                shap_values[
                    :,
                    :,
                    1
                ]
            )


        elif (
            shap_values.shape[0] >= 2
            and shap_values.shape[1]
            == len(X_data)
            and shap_values.shape[2]
            == X_data.shape[1]
        ):

            shap_values = (
                shap_values[
                    1
                ]
            )


    return np.asarray(
        shap_values,
        dtype=float
    )


# ============================================================
# FUNGSI PERHITUNGAN SHAP XGBOOST
# ============================================================

def calculate_xgb_shap_test(
    model,
    X_data
):

    dmatrix = xgb.DMatrix(
        X_data,
        enable_categorical=True
    )


    shap_output = (
        model
        .get_booster()
        .predict(
            dmatrix,
            pred_contribs=True
        )
    )


    shap_output = np.asarray(
        shap_output,
        dtype=float
    )


    if shap_output.ndim != 2:
        raise ValueError(
            "Format SHAP XGBoost tidak sesuai."
        )


    return shap_output[
        :,
        :-1
    ]


# ============================================================
# FUNGSI PERHITUNGAN SHAP CATBOOST
# ============================================================

def calculate_catboost_shap_test(
    model,
    X_data
):

    selected_cat = [
        column
        for column in X_data.columns
        if column in cat_cols
    ]


    data_pool = Pool(
        X_data,
        cat_features=selected_cat
    )


    shap_output = (
        model
        .get_feature_importance(
            data_pool,
            type="ShapValues"
        )
    )


    shap_output = np.asarray(
        shap_output,
        dtype=float
    )


    if shap_output.ndim != 2:
        raise ValueError(
            "Format SHAP CatBoost tidak sesuai."
        )


    return shap_output[
        :,
        :-1
    ]


# ============================================================
# METADATA CHECKPOINT SHAP
# ============================================================

current_shap_metadata = {
    "Skenario Pelabelan":
        str(
            LABEL_DISPLAY_NAME
        ),

    "Model":
        best_model_name,

    "Metode Agregasi":
        str(
            selected_aggregation_method
        ),

    "Top-K":
        selected_k_best,

    "Fitur":
        selected_features_best,

    "Jumlah Data Test":
        int(
            expected_test_length
        )
}


# ============================================================
# MEMERIKSA CHECKPOINT SHAP
# ============================================================

shap_checkpoint_complete = all([
    os.path.exists(
        TEST_SHAP_PATH
    ),
    os.path.exists(
        SHAP_METADATA_PATH
    )
])


shap_checkpoint_matches = False


if shap_checkpoint_complete:

    with open(
        SHAP_METADATA_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        saved_shap_metadata = json.load(
            file
        )


    shap_checkpoint_matches = (
        saved_shap_metadata
        == current_shap_metadata
    )


    if not shap_checkpoint_matches:

        print(
            "\nCheckpoint SHAP ditemukan, tetapi "
            "konfigurasinya berbeda."
        )

        print(
            "Nilai SHAP akan dihitung ulang."
        )


# ============================================================
# MENGHITUNG ATAU MEMUAT SHAP DATA TEST
# ============================================================

if (
    shap_checkpoint_complete
    and shap_checkpoint_matches
):

    print(
        "\nNilai SHAP test sudah tersedia. "
        "Memuat dari checkpoint."
    )


    test_shap_values = np.asarray(
        np.load(
            TEST_SHAP_PATH
        ),
        dtype=float
    )


else:

    print(
        "\nMenghitung nilai SHAP "
        "pada data test."
    )


    if best_model_name == "Random Forest":

        test_shap_values = (
            calculate_rf_shap_test(
                model=best_model,
                X_data=X_test_best
            )
        )


    elif best_model_name == "XGBoost":

        test_shap_values = (
            calculate_xgb_shap_test(
                model=best_model,
                X_data=X_test_best
            )
        )


    elif best_model_name == "CatBoost":

        test_shap_values = (
            calculate_catboost_shap_test(
                model=best_model,
                X_data=X_test_best
            )
        )


    else:

        raise ValueError(
            "Model tidak dikenali: "
            f"{best_model_name}"
        )


    test_shap_values = np.asarray(
        test_shap_values,
        dtype=float
    )


    np.save(
        TEST_SHAP_PATH,
        test_shap_values
    )


    with open(
        SHAP_METADATA_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            current_shap_metadata,
            file,
            ensure_ascii=False,
            indent=4
        )


    print(
        "Nilai SHAP test tersimpan:",
        TEST_SHAP_PATH
    )


# ============================================================
# VALIDASI NILAI SHAP TEST
# ============================================================

if test_shap_values.shape != (
    expected_test_length,
    len(
        selected_features_best
    )
):
    raise ValueError(
        "Ukuran SHAP test tidak sesuai. "
        f"SHAP={test_shap_values.shape}, "
        f"data={X_test_best.shape}"
    )


if not np.isfinite(
    test_shap_values
).all():
    raise ValueError(
        "Nilai SHAP test mengandung NaN atau infinity."
    )


# ============================================================
# MEMBENTUK DATA HASIL PREDIKSI
# ============================================================

error_df = (
    X_test_best
    .reset_index(drop=True)
    .copy()
)


error_df[
    "y_true"
] = y_test_true


error_df[
    "y_proba"
] = y_test_proba


error_df[
    "y_pred"
] = y_test_pred


error_df[
    "Jenis Prediksi"
] = np.select(
    [
        (
            error_df[
                "y_true"
            ] == 0
        )
        &
        (
            error_df[
                "y_pred"
            ] == 0
        ),

        (
            error_df[
                "y_true"
            ] == 0
        )
        &
        (
            error_df[
                "y_pred"
            ] == 1
        ),

        (
            error_df[
                "y_true"
            ] == 1
        )
        &
        (
            error_df[
                "y_pred"
            ] == 0
        ),

        (
            error_df[
                "y_true"
            ] == 1
        )
        &
        (
            error_df[
                "y_pred"
            ] == 1
        )
    ],

    [
        "TN",
        "FP",
        "FN",
        "TP"
    ],

    default="Tidak Diketahui"
)


if (
    error_df[
        "Jenis Prediksi"
    ] == "Tidak Diketahui"
).any():
    raise ValueError(
        "Terdapat jenis prediksi yang tidak dikenali."
    )


# ============================================================
# MASK FALSE POSITIVE DAN FALSE NEGATIVE
# ============================================================

fp_mask = (
    error_df[
        "Jenis Prediksi"
    ]
    .eq(
        "FP"
    )
    .to_numpy()
)


fn_mask = (
    error_df[
        "Jenis Prediksi"
    ]
    .eq(
        "FN"
    )
    .to_numpy()
)


fp_error_data = (
    error_df
    .loc[
        fp_mask
    ]
    .reset_index(drop=True)
    .copy()
)


fn_error_data = (
    error_df
    .loc[
        fn_mask
    ]
    .reset_index(drop=True)
    .copy()
)


# ============================================================
# FUNGSI RINGKASAN KONTRIBUSI SHAP
# ============================================================

def build_error_shap_summary(
    shap_matrix,
    mask,
    error_type,
    feature_names
):

    selected_shap = (
        shap_matrix[
            mask
        ]
    )


    total_cases = int(
        mask.sum()
    )


    if total_cases == 0:

        return pd.DataFrame(
            columns=[
                "Rank",
                "Fitur",
                "Jumlah Kontribusi",
                "Persentase Kasus",
                "Rata-rata Besar SHAP",
                "Jenis Kesalahan"
            ]
        )


    if error_type == "False Positive":

        contribution_mask = (
            selected_shap > 0
        )


        contribution_size = np.where(
            contribution_mask,
            selected_shap,
            np.nan
        )


    elif error_type == "False Negative":

        contribution_mask = (
            selected_shap < 0
        )


        contribution_size = np.where(
            contribution_mask,
            np.abs(
                selected_shap
            ),
            np.nan
        )


    else:

        raise ValueError(
            "error_type harus False Positive "
            "atau False Negative."
        )


    counts = (
        contribution_mask
        .sum(
            axis=0
        )
        .astype(int)
    )


    percentages = (
        counts
        / total_cases
        * 100
    )


    mean_sizes = np.zeros(
        len(
            feature_names
        ),
        dtype=float
    )


    for feature_index in range(
        len(
            feature_names
        )
    ):

        if counts[
            feature_index
        ] > 0:

            mean_sizes[
                feature_index
            ] = float(
                np.nanmean(
                    contribution_size[
                        :,
                        feature_index
                    ]
                )
            )


    summary = pd.DataFrame({
        "Fitur":
            list(
                feature_names
            ),

        "Jumlah Kontribusi":
            counts,

        "Persentase Kasus":
            percentages,

        "Rata-rata Besar SHAP":
            mean_sizes,

        "Jenis Kesalahan":
            error_type
    })


    summary = (
        summary
        .sort_values(
            by=[
                "Jumlah Kontribusi",
                "Rata-rata Besar SHAP",
                "Fitur"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


    summary.insert(
        0,
        "Rank",
        range(
            1,
            len(
                summary
            ) + 1
        )
    )


    return summary


# ============================================================
# MEMBENTUK RINGKASAN FP DAN FN
# ============================================================

fp_shap_summary = build_error_shap_summary(
    shap_matrix=test_shap_values,
    mask=fp_mask,
    error_type="False Positive",
    feature_names=selected_features_best
)


fn_shap_summary = build_error_shap_summary(
    shap_matrix=test_shap_values,
    mask=fn_mask,
    error_type="False Negative",
    feature_names=selected_features_best
)


# ============================================================
# RINGKASAN JUMLAH KESALAHAN
# ============================================================

total_test_cases = len(
    error_df
)


total_error_cases = int(
    fp_mask.sum()
    + fn_mask.sum()
)


error_count_summary = pd.DataFrame([
    {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            best_model_name,

        "Jenis Kesalahan":
            "False Positive",

        "Jumlah Kasus":
            int(
                fp_mask.sum()
            ),

        "Persentase Data Test":
            float(
                fp_mask.sum()
                / total_test_cases
                * 100
            ),

        "Persentase Seluruh Kesalahan":
            (
                float(
                    fp_mask.sum()
                    / total_error_cases
                    * 100
                )
                if total_error_cases > 0
                else 0.0
            )
    },

    {
        "Skenario Pelabelan":
            LABEL_DISPLAY_NAME,

        "Model":
            best_model_name,

        "Jenis Kesalahan":
            "False Negative",

        "Jumlah Kasus":
            int(
                fn_mask.sum()
            ),

        "Persentase Data Test":
            float(
                fn_mask.sum()
                / total_test_cases
                * 100
            ),

        "Persentase Seluruh Kesalahan":
            (
                float(
                    fn_mask.sum()
                    / total_error_cases
                    * 100
                )
                if total_error_cases > 0
                else 0.0
            )
    }
])


# ============================================================
# TABEL TOP-5 UNTUK LAPORAN
# ============================================================

fp_shap_top5 = (
    fp_shap_summary
    .head(5)
    .copy()
)


fn_shap_top5 = (
    fn_shap_summary
    .head(5)
    .copy()
)


# ============================================================
# FUNGSI VISUALISASI VERTIKAL
# ============================================================

def plot_error_shap_vertical(
    summary,
    error_type,
    model_name,
    feature_scenario,
    label_name,
    output_path
):

    if summary.empty:

        print(
            f"Tidak terdapat data {error_type}."
        )

        return


    plot_data = (
        summary
        .sort_values(
            by=[
                "Jumlah Kontribusi",
                "Rata-rata Besar SHAP",
                "Fitur"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
        .copy()
    )


    feature_labels = (
        plot_data[
            "Fitur"
        ]
        .astype(str)
        .tolist()
    )


    contribution_counts = (
        plot_data[
            "Jumlah Kontribusi"
        ]
        .to_numpy(
            dtype=int
        )
    )


    mean_shap_values = (
        plot_data[
            "Rata-rata Besar SHAP"
        ]
        .to_numpy(
            dtype=float
        )
    )


    x_positions = np.arange(
        len(
            plot_data
        )
    )


    if error_type == "False Positive":

        chart_title = (
            "Kontribusi Fitur pada Kasus "
            "False Positive"
        )


        left_axis_label = (
            "Jumlah Kasus False Positive\n"
            "dengan Kontribusi SHAP Positif"
        )


        right_axis_label = (
            "Rata-rata Nilai SHAP Positif"
        )


        bar_legend = (
            "Jumlah Kontribusi Positif"
        )


        line_legend = (
            "Rata-rata SHAP Positif"
        )


    elif error_type == "False Negative":

        chart_title = (
            "Kontribusi Fitur pada Kasus "
            "False Negative"
        )


        left_axis_label = (
            "Jumlah Kasus False Negative\n"
            "dengan Kontribusi SHAP Negatif"
        )


        right_axis_label = (
            "Rata-rata Besar |SHAP| Negatif"
        )


        bar_legend = (
            "Jumlah Kontribusi Negatif"
        )


        line_legend = (
            "Rata-rata |SHAP| Negatif"
        )


    else:

        raise ValueError(
            "error_type harus False Positive "
            "atau False Negative."
        )


    fig, ax1 = plt.subplots(
        figsize=(
            12,
            6.5
        )
    )


    bars = ax1.bar(
        x_positions,
        contribution_counts,
        width=0.78,
        alpha=0.78,
        label=bar_legend
    )


    ax1.set_xlabel(
        "Fitur"
    )


    ax1.set_ylabel(
        left_axis_label
    )


    ax1.set_xticks(
        x_positions
    )


    ax1.set_xticklabels(
        feature_labels,
        rotation=40,
        ha="right"
    )


    max_count = (
        contribution_counts.max()
        if len(
            contribution_counts
        ) > 0
        else 1
    )


    upper_count_limit = (
        max_count * 1.16
        if max_count > 0
        else 1
    )


    ax1.set_ylim(
        0,
        upper_count_limit
    )


    for bar, count in zip(
        bars,
        contribution_counts
    ):

        ax1.text(
            (
                bar.get_x()
                + bar.get_width() / 2
            ),
            (
                bar.get_height()
                + upper_count_limit * 0.012
            ),
            f"{int(count)}",
            ha="center",
            va="bottom",
            fontsize=9
        )


    ax2 = ax1.twinx()


    # --------------------------------------------------------
    # GARIS SHAP MENGGUNAKAN WARNA DARK ORANGE
    # --------------------------------------------------------

    ax2.plot(
        x_positions,
        mean_shap_values,
        color="darkorange",
        marker="o",
        linewidth=2.5,
        markersize=6,
        label=line_legend
    )


    ax2.set_ylabel(
        right_axis_label
    )


    max_shap = (
        mean_shap_values.max()
        if len(
            mean_shap_values
        ) > 0
        else 0.0
    )


    upper_shap_limit = (
        max_shap * 1.16
        if max_shap > 0
        else 1.0
    )


    ax2.set_ylim(
        0,
        upper_shap_limit
    )


    ax1.set_title(
        (
            f"{chart_title}\n"
            f"{model_name} — "
            f"{feature_scenario} — "
            f"{label_name}"
        )
    )


    handles_1, labels_1 = (
        ax1.get_legend_handles_labels()
    )


    handles_2, labels_2 = (
        ax2.get_legend_handles_labels()
    )


    ax1.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="upper right",
        frameon=True
    )


    ax1.grid(
        axis="y",
        linestyle="--",
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


    plt.close()


    print(
        "Gambar tersimpan:",
        output_path
    )


# ============================================================
# VISUALISASI FALSE POSITIVE
# ============================================================

plot_error_shap_vertical(
    summary=fp_shap_summary,
    error_type="False Positive",
    model_name=best_model_name,
    feature_scenario=feature_scenario_text,
    label_name=LABEL_DISPLAY_NAME,
    output_path=FP_FIGURE_PATH
)


# ============================================================
# VISUALISASI FALSE NEGATIVE
# ============================================================

plot_error_shap_vertical(
    summary=fn_shap_summary,
    error_type="False Negative",
    model_name=best_model_name,
    feature_scenario=feature_scenario_text,
    label_name=LABEL_DISPLAY_NAME,
    output_path=FN_FIGURE_PATH
)


# ============================================================
# MENYIMPAN DATA DAN TABEL BLOK 13
# ============================================================

error_df.to_pickle(
    ERROR_DATA_PATH
)


error_df.to_csv(
    os.path.join(
        DATA_DIR_BLOCK13,
        "error_data_logloss.csv"
    ),
    index=False
)


fp_error_data.to_pickle(
    FP_DATA_PATH
)


fp_error_data.to_csv(
    os.path.join(
        DATA_DIR_BLOCK13,
        "false_positive_data_logloss.csv"
    ),
    index=False
)


fn_error_data.to_pickle(
    FN_DATA_PATH
)


fn_error_data.to_csv(
    os.path.join(
        DATA_DIR_BLOCK13,
        "false_negative_data_logloss.csv"
    ),
    index=False
)


fp_shap_summary.to_pickle(
    FP_SUMMARY_PATH
)


fp_shap_summary.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_positive_shap_summary_logloss.csv"
    ),
    index=False
)


fn_shap_summary.to_pickle(
    FN_SUMMARY_PATH
)


fn_shap_summary.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_negative_shap_summary_logloss.csv"
    ),
    index=False
)


fp_shap_top5.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_positive_shap_top5_logloss.csv"
    ),
    index=False
)


fp_shap_top5.to_pickle(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_positive_shap_top5_logloss.pkl"
    )
)


fn_shap_top5.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_negative_shap_top5_logloss.csv"
    ),
    index=False
)


fn_shap_top5.to_pickle(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "false_negative_shap_top5_logloss.pkl"
    )
)


error_count_summary.to_pickle(
    ERROR_COUNT_PATH
)


error_count_summary.to_csv(
    os.path.join(
        TABLE_DIR_BLOCK13,
        "error_count_summary_logloss.csv"
    ),
    index=False
)


# ============================================================
# MENYIMPAN HASIL UTAMA BLOK 13
# ============================================================

block13_results = {
    "analysis_type":
        "Post-hoc error analysis",

    "configuration_selection_criterion":
        "Mean CV Log Loss terendah",

    "best_model_name":
        best_model_name,

    "selected_aggregation_method":
        selected_aggregation_method,

    "selected_k_best":
        selected_k_best,

    "selected_features_best":
        selected_features_best,

    "error_data":
        error_df,

    "false_positive_data":
        fp_error_data,

    "false_negative_data":
        fn_error_data,

    "false_positive_shap_summary":
        fp_shap_summary,

    "false_negative_shap_summary":
        fn_shap_summary,

    "false_positive_shap_top5":
        fp_shap_top5,

    "false_negative_shap_top5":
        fn_shap_top5,

    "error_count_summary":
        error_count_summary,

    "test_shap_values_path":
        TEST_SHAP_PATH,

    "false_positive_figure_path":
        FP_FIGURE_PATH,

    "false_negative_figure_path":
        FN_FIGURE_PATH
}


BLOCK13_RESULTS_PATH = os.path.join(
    BLOCK13_DIR,
    "block13_results_logloss.joblib"
)


joblib.dump(
    block13_results,
    BLOCK13_RESULTS_PATH,
    compress=3
)


# ============================================================
# MEMPERBARUI MODELING_RESULTS DARI BLOK 12
# ============================================================

if (
    "modeling_results" not in globals()
    or modeling_results is None
):

    if "BLOCK12_DIR" not in globals():

        BLOCK12_DIR = os.path.join(
            "./artifacts",
            (
                f"final_results_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    modeling_results_path = os.path.join(
        BLOCK12_DIR,
        "objects",
        "modeling_results_logloss.joblib"
    )


    if os.path.exists(
        modeling_results_path
    ):

        modeling_results = joblib.load(
            modeling_results_path
        )


    else:

        modeling_results = {
            "Label":
                LABEL_DISPLAY_NAME,

            "Optimization Objective":
                "Mean CV Log Loss terendah"
        }


else:

    if "BLOCK12_DIR" not in globals():

        BLOCK12_DIR = os.path.join(
            "./artifacts",
            (
                f"final_results_logloss_"
                f"{safe_file_name(LABEL_DISPLAY_NAME)}"
            )
        )


    modeling_results_path = os.path.join(
        BLOCK12_DIR,
        "objects",
        "modeling_results_logloss.joblib"
    )


modeling_results[
    "Error Analysis"
] = {
    "Analysis Type":
        "Post-hoc analysis on final test errors",

    "Used for Model Modification":
        False,

    "Error Count Summary":
        error_count_summary,

    "False Positive SHAP Summary":
        fp_shap_summary,

    "False Negative SHAP Summary":
        fn_shap_summary,

    "False Positive SHAP Top 5":
        fp_shap_top5,

    "False Negative SHAP Top 5":
        fn_shap_top5,

    "Error Data Path":
        ERROR_DATA_PATH,

    "Test SHAP Values Path":
        TEST_SHAP_PATH,

    "False Positive Figure Path":
        FP_FIGURE_PATH,

    "False Negative Figure Path":
        FN_FIGURE_PATH,

    "Block 13 Results Path":
        BLOCK13_RESULTS_PATH,

    "Block 13 Folder":
        BLOCK13_DIR
}


os.makedirs(
    os.path.dirname(
        modeling_results_path
    ),
    exist_ok=True
)


joblib.dump(
    modeling_results,
    modeling_results_path,
    compress=3
)


# ============================================================
# METADATA BLOK 13
# ============================================================

block13_metadata = {
    "Skenario Pelabelan":
        LABEL_DISPLAY_NAME,

    "Dasar Pemilihan Konfigurasi":
        "Mean CV Log Loss terendah",

    "Jenis Analisis":
        "Post-hoc error analysis pada data test",

    "Digunakan untuk Mengubah Model":
        False,

    "Model Terbaik":
        best_model_name,

    "Metode Agregasi":
        selected_aggregation_method,

    "Top-K":
        selected_k_best,

    "Fitur":
        selected_features_best,

    "Jumlah Data Test":
        int(
            total_test_cases
        ),

    "Jumlah False Positive":
        int(
            fp_mask.sum()
        ),

    "Jumlah False Negative":
        int(
            fn_mask.sum()
        ),

    "Jumlah Seluruh Kesalahan":
        int(
            total_error_cases
        ),

    "Test SHAP Path":
        TEST_SHAP_PATH,

    "False Positive Figure":
        FP_FIGURE_PATH,

    "False Negative Figure":
        FN_FIGURE_PATH,

    "Warna Garis SHAP":
        "darkorange"
}


BLOCK13_METADATA_PATH = os.path.join(
    BLOCK13_DIR,
    "block13_metadata_logloss.json"
)


with open(
    BLOCK13_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        block13_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# VALIDASI FILE HASIL BLOK 13
# ============================================================

required_block13_files = [
    TEST_SHAP_PATH,
    SHAP_METADATA_PATH,
    ERROR_DATA_PATH,
    FP_DATA_PATH,
    FN_DATA_PATH,
    FP_SUMMARY_PATH,
    FN_SUMMARY_PATH,
    ERROR_COUNT_PATH,
    BLOCK13_RESULTS_PATH,
    BLOCK13_METADATA_PATH
]


missing_block13_files = [
    path
    for path in required_block13_files
    if not os.path.exists(
        path
    )
]


if missing_block13_files:
    raise FileNotFoundError(
        "Terdapat hasil Blok 13 yang belum tersimpan:\n"
        + "\n".join(
            missing_block13_files
        )
    )


# ============================================================
# OUTPUT BLOK 13
# ============================================================

print(
    "\n"
    + "=" * 70
)


print(
    "RINGKASAN JUMLAH KESALAHAN"
)


print(
    "=" * 70
)


display_without_index(
    error_count_summary,
    formats={
        "Persentase Data Test":
            "{:.2f}%",

        "Persentase Seluruh Kesalahan":
            "{:.2f}%"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL LENGKAP KONTRIBUSI FITUR "
    "— FALSE POSITIVE"
)


print(
    "=" * 70
)


display_without_index(
    fp_shap_summary,
    formats={
        "Persentase Kasus":
            "{:.2f}%",

        "Rata-rata Besar SHAP":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS KONTRIBUSI FITUR "
    "— FALSE POSITIVE TOP 5"
)


print(
    "=" * 70
)


display_without_index(
    fp_shap_top5,
    formats={
        "Persentase Kasus":
            "{:.2f}%",

        "Rata-rata Besar SHAP":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL LENGKAP KONTRIBUSI FITUR "
    "— FALSE NEGATIVE"
)


print(
    "=" * 70
)


display_without_index(
    fn_shap_summary,
    formats={
        "Persentase Kasus":
            "{:.2f}%",

        "Rata-rata Besar SHAP":
            "{:.6f}"
    }
)


print(
    "\n"
    + "=" * 70
)


print(
    "TABEL RINGKAS KONTRIBUSI FITUR "
    "— FALSE NEGATIVE TOP 5"
)


print(
    "=" * 70
)


display_without_index(
    fn_shap_top5,
    formats={
        "Persentase Kasus":
            "{:.2f}%",

        "Rata-rata Besar SHAP":
            "{:.6f}"
    }
)


print(
    "\nJumlah False Positive:",
    int(
        fp_mask.sum()
    )
)


print(
    "Jumlah False Negative:",
    int(
        fn_mask.sum()
    )
)


print(
    "Jumlah seluruh kesalahan:",
    total_error_cases
)


print(
    "\nGrafik False Positive:"
)

print(
    FP_FIGURE_PATH
)


print(
    "\nGrafik False Negative:"
)

print(
    FN_FIGURE_PATH
)


print(
    "\nWarna garis rata-rata SHAP:",
    "darkorange"
)


print(
    "\nSeluruh hasil Blok 13 "
    "berhasil disimpan di:"
)

print(
    BLOCK13_DIR
)

## FP DAN FN